In [ ]:
# ============================================
# Hiver SDE Intern Assignment
# Phase 1: AmazonHelp Dataset Exploration
# ============================================

import os
import gc
import pandas as pd
import numpy as np

print("Environment ready!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Environment ready!
Pandas version: 2.2.3
NumPy version: 2.1.3


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/Hiver_Support_Agent"

directories = [
    "data/raw",
    "data/processed",
    "data/golden",
    "models",
    "indexes",
    "results",
    "reports"
]

for directory in directories:
    os.makedirs(os.path.join(PROJECT_DIR, directory), exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("\nCreated:")
for directory in directories:
    print("✓", directory)

Project directory: /content/drive/MyDrive/Hiver_Support_Agent

Created:
✓ data/raw
✓ data/processed
✓ data/golden
✓ models
✓ indexes
✓ results
✓ reports


In [ ]:
# ============================================
# STEP 5 — Inspect TWCS dataset safely
# ============================================

CSV_PATH = os.path.join(
    PROJECT_DIR,
    "data/raw/twcs.csv"
)

print("Dataset path:", CSV_PATH)
print("File exists:", os.path.exists(CSV_PATH))

if os.path.exists(CSV_PATH):
    size_mb = os.path.getsize(CSV_PATH) / (1024 ** 2)
    print(f"File size: {size_mb:.2f} MB")

Dataset path: /content/drive/MyDrive/Hiver_Support_Agent/data/raw/twcs.csv
File exists: True
File size: 492.58 MB


In [ ]:
# Read only a small sample first
sample_df = pd.read_csv(
    CSV_PATH,
    nrows=10
)

print("Columns:")
print(sample_df.columns.tolist())

print("\nShape of sample:")
print(sample_df.shape)

print("\nFirst 10 rows:")
display(sample_df)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

Shape of sample:
(10, 7)

First 10 rows:


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0
5,6,sprintcare,False,Tue Oct 31 21:46:24 +0000 2017,@115712 Can you please send us a private messa...,"5,7",8.0
6,8,115712,True,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,"9,6,10",NaN
7,11,sprintcare,False,Tue Oct 31 22:10:35 +0000 2017,@115713 This is saddening to hear. Please shoo...,NaN,12.0
8,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your co...,"11,13,14",15.0
9,15,sprintcare,False,Tue Oct 31 20:03:31 +0000 2017,@115713 We understand your concerns and we'd l...,12,16.0


In [ ]:
# ============================================
# STEP 6 — Extract AmazonHelp tweets
# ============================================

TARGET_BRAND = "AmazonHelp"

amazon_parts = []
total_rows = 0
amazon_rows = 0

CHUNK_SIZE = 100_000

for chunk_number, chunk in enumerate(
    pd.read_csv(
        CSV_PATH,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):
    total_rows += len(chunk)

    amazon_chunk = chunk[
        chunk["author_id"].astype(str).eq(TARGET_BRAND)
    ].copy()

    if len(amazon_chunk) > 0:
        amazon_parts.append(amazon_chunk)
        amazon_rows += len(amazon_chunk)

    if chunk_number % 5 == 0:
        print(
            f"Processed {total_rows:,} rows | "
            f"AmazonHelp: {amazon_rows:,}"
        )

amazon_df = pd.concat(amazon_parts, ignore_index=True)

print("\n========================================")
print("EXTRACTION COMPLETE")
print("========================================")
print(f"Total dataset rows: {total_rows:,}")
print(f"AmazonHelp rows:    {len(amazon_df):,}")

Processed 500,000 rows | AmazonHelp: 42,944
Processed 1,000,000 rows | AmazonHelp: 75,499
Processed 1,500,000 rows | AmazonHelp: 108,019
Processed 2,000,000 rows | AmazonHelp: 123,825
Processed 2,500,000 rows | AmazonHelp: 150,593

EXTRACTION COMPLETE
Total dataset rows: 2,811,774
AmazonHelp rows:    169,840


In [ ]:
AMAZON_PATH = os.path.join(
    PROJECT_DIR,
    "data/processed/amazonhelp_tweets.parquet"
)

amazon_df.to_parquet(
    AMAZON_PATH,
    index=False
)

print("Saved:", AMAZON_PATH)

Saved: /content/drive/MyDrive/Hiver_Support_Agent/data/processed/amazonhelp_tweets.parquet


In [ ]:
# ============================================
# STEP 7 — AmazonHelp Dataset Overview
# ============================================

print("AmazonHelp dataset shape:")
print(amazon_df.shape)

print("\nColumns:")
print(amazon_df.columns.tolist())

print("\nInbound distribution:")
print(amazon_df["inbound"].value_counts(dropna=False))

print("\nAuthor distribution:")
print(amazon_df["author_id"].value_counts().head(10))

print("\nMissing values:")
display(
    amazon_df.isna().sum().to_frame("missing_values")
)

AmazonHelp dataset shape:
(169840, 7)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

Inbound distribution:
inbound
False    169840
Name: count, dtype: int64

Author distribution:
author_id
AmazonHelp    169840
Name: count, dtype: int64

Missing values:


,missing_values
tweet_id,0
author_id,0
inbound,0
created_at,0
text,0
response_tweet_id,84566
in_response_to_tweet_id,553


In [ ]:
# ============================================
# STEP 8 — Customer vs Amazon Support
# ============================================

customer_tweets = amazon_df[
    amazon_df["inbound"] == True
]

support_tweets = amazon_df[
    amazon_df["inbound"] == False
]

print(f"Customer tweets : {len(customer_tweets):,}")
print(f"Support tweets  : {len(support_tweets):,}")

print("\nPercentage:")
print(
    f"Customer: {len(customer_tweets) / len(amazon_df) * 100:.2f}%"
)
print(
    f"Support : {len(support_tweets) / len(amazon_df) * 100:.2f}%"
)

Customer tweets : 0
Support tweets  : 169,840

Percentage:
Customer: 0.00%
Support : 100.00%


In [ ]:
# ============================================
# STEP 9 — Inspect AmazonHelp Conversation Links
# ============================================

amazon_df["tweet_id"] = pd.to_numeric(
    amazon_df["tweet_id"],
    errors="coerce"
).astype("Int64")

amazon_df["in_response_to_tweet_id"] = pd.to_numeric(
    amazon_df["in_response_to_tweet_id"],
    errors="coerce"
).astype("Int64")

print("AmazonHelp support tweets:", len(amazon_df))

print("\nAmazonHelp tweets with a parent tweet:")
print(
    amazon_df["in_response_to_tweet_id"]
    .notna()
    .sum()
)

print("\nAmazonHelp tweets with response tweets:")
print(
    amazon_df["response_tweet_id"]
    .notna()
    .sum()
)

print("\nSample relationship data:")
display(
    amazon_df[
        [
            "tweet_id",
            "author_id",
            "inbound",
            "text",
            "response_tweet_id",
            "in_response_to_tweet_id"
        ]
    ].head(20)
)

AmazonHelp support tweets: 169840

AmazonHelp tweets with a parent tweet:
169287

AmazonHelp tweets with response tweets:
85274

Sample relationship data:


,tweet_id,author_id,inbound,text,response_tweet_id,in_response_to_tweet_id
0,269,AmazonHelp,False,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,"270,271",272
1,273,AmazonHelp,False,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,274,271
2,275,AmazonHelp,False,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,NaN,274
3,324,AmazonHelp,False,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,NaN,325
4,615,AmazonHelp,False,@115820 I'm sorry we've let you down! Without ...,616,617
5,618,AmazonHelp,False,@115820 We'd like to take a further look into ...,619,616
6,620,AmazonHelp,False,@115822 I am unable to affect your account via...,NaN,621
7,622,AmazonHelp,False,"@115824 Hi, wir erhalten die Filme/Serien so v...",623,624
8,625,AmazonHelp,False,@115824 Wir haben zu danken. Schönen Abend noc...,NaN,623
9,626,AmazonHelp,False,@115826 I'm sorry for the wait. You'll receive...,627,628


In [ ]:
# ============================================
# STEP 10 — Collect IDs Connected to AmazonHelp
# ============================================

def extract_ids(series):
    """
    Extract tweet IDs from a column that may contain:
    - NaN
    - single IDs
    - comma-separated IDs
    """
    ids = set()

    for value in series.dropna():
        for item in str(value).split(","):
            item = item.strip()

            if item:
                try:
                    ids.add(int(float(item)))
                except ValueError:
                    pass

    return ids


# IDs of AmazonHelp tweets themselves
amazon_support_ids = set(
    amazon_df["tweet_id"]
    .dropna()
    .astype(int)
)

# Tweets AmazonHelp replied to
parent_ids = set(
    amazon_df["in_response_to_tweet_id"]
    .dropna()
    .astype(int)
)

# Tweets that responded to AmazonHelp
response_ids = extract_ids(
    amazon_df["response_tweet_id"]
)

# Combine all connected tweet IDs
connected_ids = (
    amazon_support_ids
    | parent_ids
    | response_ids
)

print("AmazonHelp support tweet IDs :", f"{len(amazon_support_ids):,}")
print("Parent tweet IDs             :", f"{len(parent_ids):,}")
print("Response tweet IDs           :", f"{len(response_ids):,}")
print("-------------------------------------------")
print("Total connected tweet IDs    :", f"{len(connected_ids):,}")

AmazonHelp support tweet IDs : 169,840
Parent tweet IDs             : 155,445
Response tweet IDs           : 100,785
-------------------------------------------
Total connected tweet IDs    : 359,705


In [ ]:
# ============================================
# STEP 11 — Recover Connected Conversation Tweets
# ============================================

connected_parts = []
rows_processed = 0
rows_found = 0

CHUNK_SIZE = 100_000

for chunk_number, chunk in enumerate(
    pd.read_csv(
        CSV_PATH,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):
    rows_processed += len(chunk)

    # Convert tweet_id safely
    tweet_ids = pd.to_numeric(
        chunk["tweet_id"],
        errors="coerce"
    )

    mask = tweet_ids.isin(connected_ids)

    matched = chunk.loc[mask].copy()

    if len(matched) > 0:
        connected_parts.append(matched)
        rows_found += len(matched)

    if chunk_number % 5 == 0:
        print(
            f"Processed {rows_processed:,} rows | "
            f"Connected tweets found: {rows_found:,}"
        )

amazon_connected_df = pd.concat(
    connected_parts,
    ignore_index=True
)

print("\n============================================")
print("CONNECTED TWEET EXTRACTION COMPLETE")
print("============================================")
print(
    f"Total original rows scanned: {rows_processed:,}"
)
print(
    f"Connected tweets recovered:  {len(amazon_connected_df):,}"
)

Processed 500,000 rows | Connected tweets found: 90,651
Processed 1,000,000 rows | Connected tweets found: 158,383
Processed 1,500,000 rows | Connected tweets found: 226,389
Processed 2,000,000 rows | Connected tweets found: 259,586
Processed 2,500,000 rows | Connected tweets found: 317,080

CONNECTED TWEET EXTRACTION COMPLETE
Total original rows scanned: 2,811,774
Connected tweets recovered:  358,973


In [ ]:
# ============================================
# STEP 12 — Verify Conversation Participants
# ============================================

print("Author distribution:")
display(
    amazon_connected_df["author_id"]
    .value_counts()
    .head(20)
)

print("\nInbound distribution:")
display(
    amazon_connected_df["inbound"]
    .value_counts(dropna=False)
)

print("\nUnique authors:")
print(
    amazon_connected_df["author_id"].nunique()
)

Author distribution:


,count
author_id,
AmazonHelp,169840
326613,151
158494,115
270757,80
182606,68
360103,68
159034,68
366269,66
219482,65



Inbound distribution:


,count
inbound,
True,189132
False,169841



Unique authors:
71669


In [ ]:
# ============================================
# STEP 13 — Build Tweet Relationship Lookup
# ============================================

# Make sure tweet IDs are integers
amazon_connected_df["tweet_id"] = pd.to_numeric(
    amazon_connected_df["tweet_id"],
    errors="coerce"
).astype("Int64")

amazon_connected_df["in_response_to_tweet_id"] = pd.to_numeric(
    amazon_connected_df["in_response_to_tweet_id"],
    errors="coerce"
).astype("Int64")

# Remove any invalid tweet IDs
amazon_connected_df = amazon_connected_df[
    amazon_connected_df["tweet_id"].notna()
].copy()

# Sort chronologically
amazon_connected_df["created_at"] = pd.to_datetime(
    amazon_connected_df["created_at"],
    errors="coerce",
    utc=True
)

amazon_connected_df = amazon_connected_df.sort_values(
    "created_at"
).reset_index(drop=True)

# Lookup: tweet_id → row
tweet_lookup = amazon_connected_df.set_index("tweet_id")

print("Connected tweets:", len(amazon_connected_df))
print("Unique tweet IDs:", amazon_connected_df["tweet_id"].nunique())
print("Lookup created:", len(tweet_lookup))

/tmp/ipykernel_2236/892004728.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  amazon_connected_df["created_at"] = pd.to_datetime(


Connected tweets: 358973
Unique tweet IDs: 358973
Lookup created: 358973


In [ ]:
# ============================================
# STEP 14 — Inspect Sample Conversation Chains
# ============================================

def show_conversation(tweet_id, max_depth=10):
    """
    Follow the parent chain from a tweet backwards.
    """

    conversation = []
    current_id = tweet_id

    for _ in range(max_depth):

        if current_id not in tweet_lookup.index:
            break

        row = tweet_lookup.loc[current_id]

        conversation.append({
            "tweet_id": int(current_id),
            "author_id": row["author_id"],
            "inbound": row["inbound"],
            "created_at": row["created_at"],
            "text": row["text"],
            "parent_id": row["in_response_to_tweet_id"]
        })

        parent_id = row["in_response_to_tweet_id"]

        if pd.isna(parent_id):
            break

        current_id = int(parent_id)

    # Reverse so oldest message comes first
    conversation.reverse()

    return pd.DataFrame(conversation)

In [ ]:
# Pick a few AmazonHelp tweets that have parents
sample_ids = (
    amazon_connected_df[
        (amazon_connected_df["author_id"] == "AmazonHelp") &
        (amazon_connected_df["in_response_to_tweet_id"].notna())
    ]["tweet_id"]
    .head(5)
    .tolist()
)

for i, tweet_id in enumerate(sample_ids, 1):
    print("\n" + "=" * 80)
    print(f"CONVERSATION SAMPLE {i} | Starting tweet: {tweet_id}")
    print("=" * 80)

    display(show_conversation(tweet_id))


CONVERSATION SAMPLE 1 | Starting tweet: 648405


,tweet_id,author_id,inbound,created_at,text,parent_id
0,648407,274086,True,2015-06-13 01:34:49+00:00,@115821 what in the world is a balance withhe...,<NA>
1,648405,AmazonHelp,False,2015-06-13 02:16:59+00:00,"@274086 Sorry, I'm not quite sure what you're ...",648407



CONVERSATION SAMPLE 2 | Starting tweet: 369266


,tweet_id,author_id,inbound,created_at,text,parent_id
0,369267,203483,True,2015-06-26 19:37:06+00:00,@115821 a friend of mine bought my book and wr...,<NA>
1,369266,AmazonHelp,False,2015-06-26 19:41:21+00:00,@203483 We generally send an e-mail asking for...,369267



CONVERSATION SAMPLE 3 | Starting tweet: 369259


,tweet_id,author_id,inbound,created_at,text,parent_id
0,369267,203483,True,2015-06-26 19:37:06+00:00,@115821 a friend of mine bought my book and wr...,<NA>
1,369266,AmazonHelp,False,2015-06-26 19:41:21+00:00,@203483 We generally send an e-mail asking for...,369267
2,369261,203483,True,2015-06-26 23:30:09+00:00,@AmazonHelp Here's why it matters: I need veri...,369266
3,369259,AmazonHelp,False,2015-06-26 23:52:06+00:00,@203483 The book has to be purchased on our we...,369261



CONVERSATION SAMPLE 4 | Starting tweet: 369262


,tweet_id,author_id,inbound,created_at,text,parent_id
0,369267,203483,True,2015-06-26 19:37:06+00:00,@115821 a friend of mine bought my book and wr...,<NA>
1,369266,AmazonHelp,False,2015-06-26 19:41:21+00:00,@203483 We generally send an e-mail asking for...,369267
2,369261,203483,True,2015-06-26 23:30:09+00:00,@AmazonHelp Here's why it matters: I need veri...,369266
3,369259,AmazonHelp,False,2015-06-26 23:52:06+00:00,@203483 The book has to be purchased on our we...,369261
4,369260,203483,True,2015-06-27 05:07:00+00:00,@AmazonHelp That's exactly what she did. Bough...,369259
5,369262,AmazonHelp,False,2015-06-27 07:41:36+00:00,"@203483 Sorry, we would be unable to change th...",369260



CONVERSATION SAMPLE 5 | Starting tweet: 130745


,tweet_id,author_id,inbound,created_at,text,parent_id
0,130747,145514,True,2015-07-17 13:53:02+00:00,1/ @115850 Are you guys leaking email ids of u...,<NA>
1,130745,AmazonHelp,False,2015-07-17 14:36:17+00:00,@145514 Our customers' security is our utmost ...,130747


In [ ]:
# ============================================
# STEP 15 — Identify Conversation Roots
# ============================================

# Build a fast parent lookup
parent_lookup = (
    amazon_connected_df
    .set_index("tweet_id")["in_response_to_tweet_id"]
    .to_dict()
)

def find_root(tweet_id, max_depth=50):
    """
    Follow parent links until reaching the root tweet.
    """
    current = int(tweet_id)
    visited = set()

    for _ in range(max_depth):

        # Prevent cycles
        if current in visited:
            break

        visited.add(current)

        parent = parent_lookup.get(current)

        # No parent → root
        if parent is None or pd.isna(parent):
            return current

        parent = int(parent)

        # Parent not available in our recovered graph
        if parent not in parent_lookup:
            return parent

        current = parent

    return current


# Calculate root for every tweet
amazon_connected_df["root_tweet_id"] = (
    amazon_connected_df["tweet_id"]
    .apply(find_root)
)

print("Total tweets:", len(amazon_connected_df))
print(
    "Unique conversation roots:",
    amazon_connected_df["root_tweet_id"].nunique()
)

Total tweets: 358973
Unique conversation roots: 83572


In [ ]:
# ============================================
# STEP 16 — Conversation Size Distribution
# ============================================

conversation_sizes = (
    amazon_connected_df
    .groupby("root_tweet_id")
    .size()
)

print("Conversation statistics:")
print(
    conversation_sizes.describe()
)

print("\nConversation size distribution:")

bins = [1, 2, 3, 4, 5, 10, 20, 50, 100, np.inf]

labels = [
    "1",
    "2",
    "3",
    "4",
    "5-10",
    "11-20",
    "21-50",
    "51-100",
    "100+"
]

size_categories = pd.cut(
    conversation_sizes,
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

display(
    size_categories
    .value_counts()
    .sort_index()
    .to_frame("conversation_count")
)

Conversation statistics:
count    83572.000000
mean         4.295374
std          4.189421
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max        195.000000
dtype: float64

Conversation size distribution:


,conversation_count
1,33521
2,11794
3,14243
4,5855
5-10,14210
11-20,3264
21-50,613
51-100,60
100+,12


In [ ]:
# ============================================
# STEP 17 — Conversation Case Quality Profile
# ============================================

case_profile = (
    amazon_connected_df
    .groupby("root_tweet_id")
    .agg(
        turn_count=("tweet_id", "size"),
        customer_turns=("inbound", "sum"),
        support_turns=("inbound", lambda x: (~x).sum()),
        first_time=("created_at", "min"),
        last_time=("created_at", "max"),
        unique_authors=("author_id", "nunique")
    )
    .reset_index()
)

case_profile["duration_hours"] = (
    case_profile["last_time"] -
    case_profile["first_time"]
).dt.total_seconds() / 3600

case_profile["has_customer"] = (
    case_profile["customer_turns"] > 0
)

case_profile["has_support"] = (
    case_profile["support_turns"] > 0
)

case_profile["is_two_sided"] = (
    case_profile["has_customer"] &
    case_profile["has_support"]
)

print("Total reconstructed cases:", len(case_profile))

print("\nCases containing customer messages:",
      case_profile["has_customer"].sum())

print("Cases containing support messages:",
      case_profile["has_support"].sum())

print("Two-sided customer + support cases:",
      case_profile["is_two_sided"].sum())

print("\nCase profile:")
display(case_profile.describe())

Total reconstructed cases: 83572

Cases containing customer messages: 83494
Cases containing support messages: 83496
Two-sided customer + support cases: 83418

Case profile:


,root_tweet_id,turn_count,customer_turns,support_turns,unique_authors,duration_hours
count,8.357200e+04,83572.0,83572.000000,83572.000000,83572.000000,83572.000000
mean,1.444210e+06,4.295374,2.263102,2.032272,2.032762,13.062846
std,9.129856e+05,4.189421,2.309865,1.990470,0.533927,187.523866
min,2.720000e+02,1.0,0.000000,0.000000,1.000000,0.000000
25%,6.022742e+05,2.0,1.000000,1.000000,2.000000,0.186944
50%,1.351756e+06,3.0,2.000000,1.000000,2.000000,0.427778
75%,2.318087e+06,5.0,3.000000,2.000000,2.000000,1.313889
max,2.987939e+06,195.0,97.000000,98.000000,47.000000,20848.641111


In [ ]:
# ============================================
# STEP 18 — Inspect Single-Tweet Cases
# ============================================

single_roots = case_profile[
    case_profile["turn_count"] == 1
]["root_tweet_id"]

single_tweets = amazon_connected_df[
    amazon_connected_df["root_tweet_id"].isin(single_roots)
].copy()

print("Single-tweet cases:", len(single_tweets))

print("\nInbound distribution:")
display(single_tweets["inbound"].value_counts())

print("\nSample single-tweet cases:")
display(
    single_tweets[
        [
            "tweet_id",
            "author_id",
            "inbound",
            "text"
        ]
    ].sample(
        min(20, len(single_tweets)),
        random_state=42
    )
)

Single-tweet cases: 112

Inbound distribution:


,count
inbound,
False,59
True,53



Sample single-tweet cases:


,tweet_id,author_id,inbound,text
93575,965155,326613,True,@AmazonHelp I have got a reply stating that he...
150503,1279892,366269,True,@AmazonHelp I have been replying to your mails...
11504,197941,162537,True,@AmazonHelp @115851 got an update. Please chec...
126759,1043704,366798,True,@AmazonHelp @115821 @115850 @115851 I have fil...
106039,294299,AmazonHelp,False,"@185733 We hate to keep you waiting, however, ..."
224842,238109,AmazonHelp,False,@172659 We keep adding new content to Prime. S...
49012,279410,182606,True,@AmazonHelp No one does tht
24631,197947,162537,True,"@AmazonHelp I have checked, I didnt receive an..."
24625,197946,AmazonHelp,False,"@162537 That's strange, kindly check in the sp..."
122707,294301,AmazonHelp,False,@185733 Could you please check if you've recei...


In [ ]:
# ============================================
# STEP 19 — Initial Two-Sided Case Pool
# ============================================

candidate_cases = case_profile[
    (case_profile["is_two_sided"]) &
    (case_profile["turn_count"] >= 2)
].copy()

print(
    "Two-sided cases with >=2 turns:",
    len(candidate_cases)
)

print("\nTurn distribution:")
display(
    candidate_cases["turn_count"]
    .value_counts()
    .sort_index()
    .head(20)
)

Two-sided cases with >=2 turns: 83418

Turn distribution:


,count
turn_count,
2,33382
3,11788
4,14240
5,5851
6,5800
7,2968
8,2655
9,1520
10,1265


In [ ]:
# ============================================
# DEBUG — Verify Single-Tweet Cases
# ============================================

print("Total rows in amazon_connected_df:",
      len(amazon_connected_df))

print("Total unique roots:",
      amazon_connected_df["root_tweet_id"].nunique())

print("\nRows by turn count:")
display(
    amazon_connected_df
    .groupby("root_tweet_id")
    .size()
    .value_counts()
    .sort_index()
    .head(15)
)

# Directly identify roots occurring exactly once
direct_single_roots = (
    amazon_connected_df["root_tweet_id"]
    .value_counts()
)

direct_single_roots = direct_single_roots[
    direct_single_roots == 1
].index

print("\nDirect single-tweet roots:",
      len(direct_single_roots))

# Get those rows directly
direct_single_tweets = amazon_connected_df[
    amazon_connected_df["root_tweet_id"].isin(direct_single_roots)
].copy()

print("Direct single-tweet rows:",
      len(direct_single_tweets))

print("\nInbound distribution:")
display(
    direct_single_tweets["inbound"].value_counts()
)

print("\nSample:")
display(
    direct_single_tweets[
        ["tweet_id", "author_id", "inbound", "text"]
    ].sample(
        min(20, len(direct_single_tweets)),
        random_state=42
    )
)

Total rows in amazon_connected_df: 358973
Total unique roots: 83572

Rows by turn count:


,count
1,112
2,33409
3,11794
4,14243
5,5855
6,5800
7,2969
8,2655
9,1520
10,1266



Direct single-tweet roots: 112
Direct single-tweet rows: 112

Inbound distribution:


,count
inbound,
False,59
True,53



Sample:


,tweet_id,author_id,inbound,text
93575,965155,326613,True,@AmazonHelp I have got a reply stating that he...
150503,1279892,366269,True,@AmazonHelp I have been replying to your mails...
11504,197941,162537,True,@AmazonHelp @115851 got an update. Please chec...
126759,1043704,366798,True,@AmazonHelp @115821 @115850 @115851 I have fil...
106039,294299,AmazonHelp,False,"@185733 We hate to keep you waiting, however, ..."
224842,238109,AmazonHelp,False,@172659 We keep adding new content to Prime. S...
49012,279410,182606,True,@AmazonHelp No one does tht
24631,197947,162537,True,"@AmazonHelp I have checked, I didnt receive an..."
24625,197946,AmazonHelp,False,"@162537 That's strange, kindly check in the sp..."
122707,294301,AmazonHelp,False,@185733 Could you please check if you've recei...


In [ ]:
# ============================================
# STEP 20 — Build Conversation Cases
# ============================================

def build_case(group):
    group = group.sort_values("created_at")

    messages = []

    for _, row in group.iterrows():
        role = "customer" if row["inbound"] else "support"

        messages.append({
            "tweet_id": int(row["tweet_id"]),
            "role": role,
            "author_id": str(row["author_id"]),
            "timestamp": row["created_at"].isoformat(),
            "text": str(row["text"])
        })

    return pd.Series({
        "root_tweet_id": int(group["root_tweet_id"].iloc[0]),
        "turn_count": len(group),
        "customer_turns": int(group["inbound"].sum()),
        "support_turns": int((~group["inbound"]).sum()),
        "first_time": group["created_at"].min(),
        "last_time": group["created_at"].max(),
        "duration_hours": (
            group["created_at"].max() -
            group["created_at"].min()
        ).total_seconds() / 3600,
        "final_role": (
            "customer"
            if bool(group["inbound"].iloc[-1])
            else "support"
        ),
        "messages": messages
    })


conversation_cases = (
    amazon_connected_df
    .groupby("root_tweet_id", sort=False)
    .apply(build_case)
    .reset_index(drop=True)
)

conversation_cases["case_id"] = [
    f"AMZ-{i:06d}"
    for i in range(1, len(conversation_cases) + 1)
]

# Put case_id first
cols = ["case_id"] + [
    c for c in conversation_cases.columns
    if c != "case_id"
]

conversation_cases = conversation_cases[cols]

print("Conversation cases:", len(conversation_cases))

display(
    conversation_cases[
        [
            "case_id",
            "root_tweet_id",
            "turn_count",
            "customer_turns",
            "support_turns",
            "duration_hours",
            "final_role"
        ]
    ].head(10)
)

Conversation cases: 83572


/tmp/ipykernel_2236/2814428304.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_case)


,case_id,root_tweet_id,turn_count,customer_turns,support_turns,duration_hours,final_role
0,AMZ-000001,648407,3,2,1,0.718611,customer
1,AMZ-000002,369267,7,4,3,21.288889,customer
2,AMZ-000003,2933545,2,1,1,20848.641111,support
3,AMZ-000004,130747,3,2,1,1.134444,customer
4,AMZ-000005,940416,3,2,1,16025.251667,customer
5,AMZ-000006,1390559,4,2,2,15356.840278,support
6,AMZ-000007,1087361,9,4,5,13760.785000,support
7,AMZ-000008,903740,9,5,4,14012.028056,support
8,AMZ-000009,313821,7,4,3,1.920278,customer
9,AMZ-000010,1433907,2,1,1,12320.650000,support


In [ ]:
# ============================================
# STEP 21 — Inspect Reconstructed Cases
# ============================================

def print_case(case):
    print("=" * 90)
    print(
        f"{case['case_id']} | "
        f"{case['turn_count']} turns | "
        f"Duration: {case['duration_hours']:.2f}h"
    )
    print("=" * 90)

    for i, msg in enumerate(case["messages"], 1):
        print(f"\n[{i}] {msg['role'].upper()}")
        print(msg["text"])


for _, case in conversation_cases.sample(
    5,
    random_state=42
).iterrows():

    print_case(case)

AMZ-063507 | 8 turns | Duration: 1.75h

[1] CUSTOMER
Si je dois faire un achat pour le Black Friday, ça sera la montre Samsung Gear S3. Je compte sur vous @120533

[2] SUPPORT
@124406 N'hésitez pas à scruter les promotions, peut-être qu'il y aura une surprise pour vous 😉 ^MA

[3] CUSTOMER
@AmazonHelp J’espère bien haha

[4] SUPPORT
@124406 C'est tout ce que votre liste de Noël contient cette année ? 😜 ^MA

[5] CUSTOMER
@AmazonHelp Si j’ai plusieurs idées mais mon choix principal c’est la Gear S3 😋

[6] SUPPORT
@124406 Je vois que vous êtes un passionné des nouvelles technologies 😎 ^MA

[7] CUSTOMER
@AmazonHelp Oui assez :) En tout cas j'attends le black friday avec impatience. Faites moi plaisir ;)

[8] SUPPORT
@124406 Restez connecté alors 😉 ^MA
AMZ-079307 | 2 turns | Duration: 0.19h

[1] CUSTOMER
Hola @AmazonHelp. 
Hoy @130687 debería haberme traído el envío 201727__credit_card__ y no saben nada de él. https://t.co/v6HUUdlGpF

[2] SUPPORT
@228714 Hola, lamentamos los inconvenientes. 

In [ ]:
# ============================================
# STEP 22 — Temporal Quality Analysis
# ============================================

print("Conversation duration statistics (hours):")
display(
    conversation_cases["duration_hours"].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

print("\nDuration buckets:")

duration_bins = [
    -0.001,
    0.5,
    1,
    3,
    6,
    12,
    24,
    72,
    168,
    720,
    np.inf
]

duration_labels = [
    "<30 min",
    "30m–1h",
    "1–3h",
    "3–6h",
    "6–12h",
    "12–24h",
    "1–3 days",
    "3–7 days",
    "7–30 days",
    "30+ days"
]

duration_bucket = pd.cut(
    conversation_cases["duration_hours"],
    bins=duration_bins,
    labels=duration_labels
)

display(
    duration_bucket
    .value_counts()
    .sort_index()
    .to_frame("case_count")
)

Conversation duration statistics (hours):


,duration_hours
count,83572.000000
mean,13.062846
std,187.523866
min,0.000000
50%,0.427778
75%,1.313889
90%,11.070528
95%,30.506944
99%,206.950161
99.5%,377.167124



Duration buckets:


,case_count
duration_hours,
<30 min,46228
30m–1h,12913
1–3h,10629
3–6h,3144
6–12h,2641
12–24h,2837
1–3 days,2887
3–7 days,1259
7–30 days,857


In [ ]:
# ============================================
# STEP 23 — Inspect Long-Duration Cases
# ============================================

long_cases = conversation_cases.sort_values(
    "duration_hours",
    ascending=False
).head(20)

display(
    long_cases[
        [
            "case_id",
            "root_tweet_id",
            "turn_count",
            "customer_turns",
            "support_turns",
            "duration_hours",
            "final_role"
        ]
    ]
)

,case_id,root_tweet_id,turn_count,customer_turns,support_turns,duration_hours,final_role
2,AMZ-000003,2933545,2,1,1,20848.641111,support
4,AMZ-000005,940416,3,2,1,16025.251667,customer
5,AMZ-000006,1390559,4,2,2,15356.840278,support
7,AMZ-000008,903740,9,5,4,14012.028056,support
6,AMZ-000007,1087361,9,4,5,13760.785000,support
9,AMZ-000010,1433907,2,1,1,12320.650000,support
12,AMZ-000013,1927893,5,3,2,9207.906667,customer
15,AMZ-000016,644873,4,2,2,9162.906944,support
13,AMZ-000014,1293697,8,5,3,9119.583056,support
18,AMZ-000019,2902688,11,6,5,8948.576389,customer


In [ ]:
# ============================================
# STEP 24 — Inspect Extreme Conversations
# ============================================

for _, case in long_cases.head(5).iterrows():
    print_case(case)

AMZ-000003 | 2 turns | Duration: 20848.64h

[1] CUSTOMER
@811202 @AmazonHelp i think u should cancel your order arpit and book another on Flipkart. It will deliver ur order in 2-3 working day

[2] SUPPORT
@132991 We would like to help you, Prakhar. Please share your details with us here: https://t.co/GIJyeYqKE0. We'll get in touch shortly.
Please don't provide your order details as we consider it personal information. Our twitter page is visible to public. -Bikram
AMZ-000005 | 3 turns | Duration: 16025.25h

[1] CUSTOMER
The sound quality of The Deer Hunter on Amazon Prime on the Xbox is the worst thing I've ever heard. @AmazonHelp @XboxSupport

[2] SUPPORT
@342936 Let us know if you have any other questions or concerns after you've spoke with our Support team! ^ME

[3] CUSTOMER
@AmazonHelp @342936 Googled the sound issue &amp; discovered these tweets...the sound quality is still off, tinny..totally unwatchable. I'm On the PS4 app.
AMZ-000006 | 4 turns | Duration: 15356.84h

[1] CUSTOME

In [ ]:
# ============================================
# STEP 25 — Analyze Consecutive Message Gaps
# ============================================

def calculate_gap_stats(group):
    group = group.sort_values("created_at")

    timestamps = group["created_at"].values

    if len(timestamps) <= 1:
        return pd.Series({
            "max_gap_hours": 0.0,
            "median_gap_hours": 0.0,
            "mean_gap_hours": 0.0
        })

    gaps = (
        pd.Series(timestamps)
        .diff()
        .dt.total_seconds()
        / 3600
    ).dropna()

    return pd.Series({
        "max_gap_hours": gaps.max(),
        "median_gap_hours": gaps.median(),
        "mean_gap_hours": gaps.mean()
    })


gap_stats = (
    amazon_connected_df
    .groupby("root_tweet_id")
    .apply(calculate_gap_stats)
    .reset_index()
)

conversation_cases = conversation_cases.merge(
    gap_stats,
    on="root_tweet_id",
    how="left"
)

print("Gap statistics:")
display(
    conversation_cases[
        [
            "max_gap_hours",
            "median_gap_hours",
            "mean_gap_hours"
        ]
    ].describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
            .995,
            .999
        ]
    )
)

Gap statistics:


/tmp/ipykernel_2236/3137453929.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_gap_stats)


,max_gap_hours,median_gap_hours,mean_gap_hours
count,83572.000000,83572.000000,83572.000000
mean,11.080828,1.374285,3.013013
std,180.192960,91.798479,96.910243
min,0.000000,0.000000,0.000000
50%,0.304722,0.161667,0.205741
75%,0.801667,0.315174,0.480648
90%,9.036250,0.555556,2.344253
95%,24.780694,1.016208,6.186383
99%,165.758603,7.400814,32.257163
99.5%,284.661119,12.157955,61.662638


In [ ]:
# ============================================
# STEP 26 — Consecutive Gap Distribution
# ============================================

gap_bins = [
    -0.001,
    0.25,
    0.5,
    1,
    3,
    6,
    12,
    24,
    48,
    72,
    168,
    720,
    np.inf
]

gap_labels = [
    "<15 min",
    "15–30 min",
    "30m–1h",
    "1–3h",
    "3–6h",
    "6–12h",
    "12–24h",
    "1–2 days",
    "2–3 days",
    "3–7 days",
    "7–30 days",
    "30+ days"
]

gap_bucket = pd.cut(
    conversation_cases["max_gap_hours"],
    bins=gap_bins,
    labels=gap_labels
)

display(
    gap_bucket
    .value_counts()
    .sort_index()
    .to_frame("case_count")
)

,case_count
max_gap_hours,
<15 min,36080
15–30 min,19554
30m–1h,9178
1–3h,6572
3–6h,2310
6–12h,2592
12–24h,2892
1–2 days,1732
2–3 days,807


In [ ]:
# ============================================
# STEP 27 — Inspect Cases With Huge Message Gaps
# ============================================

worst_gap_cases = (
    conversation_cases
    .sort_values(
        "max_gap_hours",
        ascending=False
    )
    .head(10)
)

display(
    worst_gap_cases[
        [
            "case_id",
            "root_tweet_id",
            "turn_count",
            "customer_turns",
            "support_turns",
            "duration_hours",
            "max_gap_hours",
            "final_role"
        ]
    ]
)

,case_id,root_tweet_id,turn_count,customer_turns,support_turns,duration_hours,max_gap_hours,final_role
2,AMZ-000003,2933545,2,1,1,20848.641111,20848.641111,support
4,AMZ-000005,940416,3,2,1,16025.251667,16024.694722,customer
5,AMZ-000006,1390559,4,2,2,15356.840278,15356.550278,support
7,AMZ-000008,903740,9,5,4,14012.028056,14011.140556,support
6,AMZ-000007,1087361,9,4,5,13760.785000,13700.781389,support
9,AMZ-000010,1433907,2,1,1,12320.650000,12320.650000,support
12,AMZ-000013,1927893,5,3,2,9207.906667,9207.658333,customer
15,AMZ-000016,644873,4,2,2,9162.906944,9156.791111,support
18,AMZ-000019,2902688,11,6,5,8948.576389,8946.063611,customer
16,AMZ-000017,231516,6,2,4,7925.590556,7923.351111,support


In [ ]:
for _, case in worst_gap_cases.head(5).iterrows():
    print_case(case)

AMZ-000003 | 2 turns | Duration: 20848.64h

[1] CUSTOMER
@811202 @AmazonHelp i think u should cancel your order arpit and book another on Flipkart. It will deliver ur order in 2-3 working day

[2] SUPPORT
@132991 We would like to help you, Prakhar. Please share your details with us here: https://t.co/GIJyeYqKE0. We'll get in touch shortly.
Please don't provide your order details as we consider it personal information. Our twitter page is visible to public. -Bikram
AMZ-000005 | 3 turns | Duration: 16025.25h

[1] CUSTOMER
The sound quality of The Deer Hunter on Amazon Prime on the Xbox is the worst thing I've ever heard. @AmazonHelp @XboxSupport

[2] SUPPORT
@342936 Let us know if you have any other questions or concerns after you've spoke with our Support team! ^ME

[3] CUSTOMER
@AmazonHelp @342936 Googled the sound issue &amp; discovered these tweets...the sound quality is still off, tinny..totally unwatchable. I'm On the PS4 app.
AMZ-000006 | 4 turns | Duration: 15356.84h

[1] CUSTOME

In [ ]:
# ============================================
# STEP 28 — Resolution Candidate Analysis
# ============================================

print("Total cases:", len(conversation_cases))

print("\nFinal role distribution:")
display(
    conversation_cases["final_role"]
    .value_counts()
    .to_frame("case_count")
)

print("\nCases with both customer + support:")
two_sided = conversation_cases[
    (conversation_cases["customer_turns"] > 0) &
    (conversation_cases["support_turns"] > 0)
].copy()

print("Two-sided cases:", len(two_sided))

print("\nFinal role among two-sided cases:")
display(
    two_sided["final_role"]
    .value_counts()
    .to_frame("case_count")
)

print("\nTurn statistics:")
display(
    two_sided[
        [
            "turn_count",
            "customer_turns",
            "support_turns"
        ]
    ].describe()
)

Total cases: 83572

Final role distribution:


,case_count
final_role,
support,66088
customer,17484



Cases with both customer + support:
Two-sided cases: 83418

Final role among two-sided cases:


,case_count
final_role,
support,66010
customer,17408



Turn statistics:


,turn_count,customer_turns,support_turns
count,83418.000000,83418.000000,83418.000000
mean,4.300511,2.265722,2.034789
std,4.191267,2.310443,1.991130
min,2.000000,1.000000,1.000000
25%,2.000000,1.000000,1.000000
50%,3.000000,2.000000,1.000000
75%,5.000000,3.000000,2.000000
max,195.000000,97.000000,98.000000


In [ ]:
# ============================================
# STEP 29 — Extract Support Resolution Pairs
# ============================================

def extract_resolution(case):

    messages = case["messages"]

    customer_messages = [
        m for m in messages
        if m["role"] == "customer"
    ]

    support_messages = [
        m for m in messages
        if m["role"] == "support"
    ]

    if not customer_messages or not support_messages:
        return None

    # Last customer message before final support response
    last_support_index = max(
        i for i, m in enumerate(messages)
        if m["role"] == "support"
    )

    previous_customer = None

    for i in range(last_support_index - 1, -1, -1):
        if messages[i]["role"] == "customer":
            previous_customer = messages[i]
            break

    final_support = messages[last_support_index]

    return {
        "case_id": case["case_id"],
        "root_tweet_id": case["root_tweet_id"],
        "turn_count": case["turn_count"],
        "customer_turns": case["customer_turns"],
        "support_turns": case["support_turns"],
        "duration_hours": case["duration_hours"],
        "max_gap_hours": case["max_gap_hours"],
        "customer_query": (
            previous_customer["text"]
            if previous_customer
            else ""
        ),
        "support_response": final_support["text"],
        "final_role": case["final_role"]
    }


resolution_records = []

for _, case in two_sided.iterrows():

    record = extract_resolution(case)

    if record is not None:
        resolution_records.append(record)


resolution_df = pd.DataFrame(resolution_records)

print("Resolution candidates:", len(resolution_df))

display(
    resolution_df[
        [
            "case_id",
            "turn_count",
            "customer_query",
            "support_response"
        ]
    ].head(10)
)

Resolution candidates: 83418


,case_id,turn_count,customer_query,support_response
0,AMZ-000001,3,@115821 what in the world is a balance withhe...,"@274086 Sorry, I'm not quite sure what you're ..."
1,AMZ-000002,7,@AmazonHelp That's exactly what she did. Bough...,"@203483 Sorry, we would be unable to change th..."
2,AMZ-000003,2,@811202 @AmazonHelp i think u should cancel yo...,"@132991 We would like to help you, Prakhar. Pl..."
3,AMZ-000004,3,1/ @115850 Are you guys leaking email ids of u...,@145514 Our customers' security is our utmost ...
4,AMZ-000005,3,The sound quality of The Deer Hunter on Amazon...,@342936 Let us know if you have any other ques...
5,AMZ-000006,4,@AmazonHelp I got a call today 10/19/17 from ...,"@443453 As this is not one of our numbers, ple..."
6,AMZ-000007,9,@AmazonHelp @376508 You really want to keep my...,@376509 I'm sorry for any trouble. Are you a s...
7,AMZ-000008,9,@AmazonHelp And btw I encounter the same issue...,@334536 We'd like to get the details so we can...
8,AMZ-000009,7,@amazonhelp I already A2Z claims 2 times on se...,@15547 Hi there! kindly refer to the email sen...
9,AMZ-000010,2,"Compré un #ajedrez por @115821.Moderno,vino c...",@452932 Gracias por notificarnos. Nos gustaría...


In [ ]:
# ============================================
# STEP 30 — Support Response Pattern Analysis
# ============================================

resolution_df["support_response_length"] = (
    resolution_df["support_response"]
    .astype(str)
    .str.len()
)

resolution_df["customer_query_length"] = (
    resolution_df["customer_query"]
    .astype(str)
    .str.len()
)

print("Support response length:")
display(
    resolution_df["support_response_length"].describe(
        percentiles=[.25, .50, .75, .90, .95, .99]
    )
)

Support response length:


,support_response_length
count,83418.000000
mean,123.906231
std,48.931077
min,7.000000
25%,96.000000
50%,124.000000
75%,136.000000
90%,190.000000
95%,228.000000
99%,270.000000


In [ ]:
# ============================================
# STEP 31 — Sample Historical Responses
# ============================================

sample_resolution = resolution_df.sample(
    30,
    random_state=42
)

for _, row in sample_resolution.iterrows():

    print("=" * 90)
    print("CASE:", row["case_id"])
    print("\nCUSTOMER:")
    print(row["customer_query"])

    print("\nSUPPORT:")
    print(row["support_response"])

CASE: AMZ-038301

CUSTOMER:
@AmazonHelp Poor seller support team Contacted several time make new case but no solution on that..kindly ulter few of the policy

SUPPORT:
@448715 Sorry to know that. I'd like to take a closer look into this, please share your details here: https://t.co/GIJyeYqKE0. ^PS
CASE: AMZ-008792

CUSTOMER:
@AmazonHelp Can you please tell me when i am going to receive my order ??
Pls be straightforward .. dont loop me in ur words...

SUPPORT:
@215016 We're right on it! Be assured we will get the package delivered at the earliest.^MN
CASE: AMZ-080712

CUSTOMER:
@AmazonHelp @AmazonHelp Hello?

SUPPORT:
@239938 We would love to help get this order completed Dakota, but we are unable to access your https://t.co/nUUp5MtGzL account through Twitter. Please give us a call or chat here so that we can properly assist you:  https://t.co/hApLpM3Ejd ^JD
CASE: AMZ-033183

CUSTOMER:
@AmazonHelp Hey there could the same balance be used for videocond2h recharge?? could you please prov

In [ ]:
# ============================================
# STEP 32 — Inspect Short / Generic Responses
# ============================================

short_responses = (
    resolution_df
    .sort_values("support_response_length")
    .head(30)
)

for _, row in short_responses.iterrows():

    print("=" * 80)
    print("CASE:", row["case_id"])

    print("\nCUSTOMER:")
    print(row["customer_query"])

    print("\nSUPPORT:")
    print(row["support_response"])

CASE: AMZ-071175

CUSTOMER:
@AmazonHelp Nö 😂😂 Aver vllt eurem Lieferanten :D

SUPPORT:
@131789
CASE: AMZ-080119

CUSTOMER:
Slt @183104  @136301 🤣 Encore un petit colis de chez @120533 en rentrant du boulot 😛😛 Merci pour vos alertes c est top 👌 https://t.co/niGNAzykF4

SUPPORT:
@234634 👍
CASE: AMZ-015608

CUSTOMER:
@120533 qu'est ce qui est jaune et qui attend? https://t.co/I6FbBHvVfO

SUPPORT:
@365408 🤔
CASE: AMZ-070750

CUSTOMER:
Et d'ailleurs j'avais pas vu celui ci aussi toujours chez @120533  pour le prix de 20€ avec 5 boosters et 6 cartes promo exclusives https://t.co/5aAyIcg3YW

SUPPORT:
@695828 😉
CASE: AMZ-082794

CUSTOMER:
Sur @120533 ils prévoient des expéditions un dimanche. Badass !

SUPPORT:
@255991 👏
CASE: AMZ-046719

CUSTOMER:
@620877 @129009 @120533 Ho yes, @120533 en avait 😍

SUPPORT:
@227701 😘
CASE: AMZ-001902

CUSTOMER:
Hier j'ai reçu mon siège et ma burn ! Au top prêt à tryhard 💪👊Merci @170311 &amp; @120533 ❤️

SUPPORT:
@170310 👍
CASE: AMZ-012983

CUSTOMER:
@AmazonHe

In [ ]:
# ============================================
# STEP 33A — Resolution Quality Signals
# ============================================

import re
import numpy as np
import pandas as pd

df = resolution_df.copy()

# ------------------------------------------------
# Basic text features
# ------------------------------------------------

df["customer_len"] = (
    df["customer_query"]
    .fillna("")
    .astype(str)
    .str.len()
)

df["support_len"] = (
    df["support_response"]
    .fillna("")
    .astype(str)
    .str.len()
)

df["customer_words"] = (
    df["customer_query"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

df["support_words"] = (
    df["support_response"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

# ------------------------------------------------
# Support response patterns
# ------------------------------------------------

support_text = (
    df["support_response"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["has_url"] = support_text.str.contains(
    r"http[s]?://|t\.co/",
    regex=True
)

df["asks_for_details"] = support_text.str.contains(
    r"share.*details|provide.*details|send.*details|"
    r"dm us|direct message|private message|"
    r"contact us|email us|call us",
    regex=True
)

df["has_action_language"] = support_text.str.contains(
    r"please|check|try|follow|contact|call|email|"
    r"update|reset|change|return|refund|replace|"
    r"send|share|provide|visit",
    regex=True
)

df["has_apology"] = support_text.str.contains(
    r"sorry|apolog|apologies|regret",
    regex=True
)

df["has_policy_language"] = support_text.str.contains(
    r"unable|cannot|can't|don't have|"
    r"not possible|policy|eligible|available",
    regex=True
)

df["has_security_language"] = support_text.str.contains(
    r"security|scam|fraud|password|personal information|"
    r"account.*security|do not provide",
    regex=True
)

# ------------------------------------------------
# Conversation signals
# ------------------------------------------------

df["ends_with_support"] = (
    df["final_role"] == "support"
)

df["has_customer_followup"] = (
    df["final_role"] == "customer"
)

print("Rows:", len(df))

display(
    df[
        [
            "support_len",
            "support_words",
            "has_url",
            "asks_for_details",
            "has_action_language",
            "has_apology",
            "has_policy_language",
            "has_security_language",
            "ends_with_support",
            "has_customer_followup"
        ]
    ].describe(include="all")
)

Rows: 83418


,support_len,support_words,has_url,asks_for_details,has_action_language,has_apology,has_policy_language,has_security_language,ends_with_support,has_customer_followup
count,83418.000000,83418.000000,83418,83418,83418,83418,83418,83418,83418,83418
unique,NaN,NaN,2,2,2,2,2,2,2,2
top,NaN,NaN,False,False,False,False,False,False,True,False
freq,NaN,NaN,44138,74248,41917,62565,78022,81106,66010,66010
mean,123.906231,19.543012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,48.931077,9.183168,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,7.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,96.000000,15.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,124.000000,19.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,136.000000,23.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ============================================
# STEP 33B — Pattern Counts
# ============================================

patterns = {
    "Ends with support": df["ends_with_support"],
    "Ends with customer": df["has_customer_followup"],
    "Asks for details": df["asks_for_details"],
    "Has action language": df["has_action_language"],
    "Has apology": df["has_apology"],
    "Has policy language": df["has_policy_language"],
    "Has security language": df["has_security_language"],
    "Contains URL": df["has_url"]
}

pattern_counts = pd.DataFrame({
    "count": {k: int(v.sum()) for k, v in patterns.items()},
    "percentage": {
        k: round(v.mean() * 100, 2)
        for k, v in patterns.items()
    }
})

display(pattern_counts)

,count,percentage
Ends with support,66010,79.13
Ends with customer,17408,20.87
Asks for details,9170,10.99
Has action language,41501,49.75
Has apology,20853,25.00
Has policy language,5396,6.47
Has security language,2312,2.77
Contains URL,39280,47.09


In [ ]:
# ============================================
# STEP 34 — Provisional Resolution Buckets
# ============================================

df = resolution_df.copy()

# Recreate/use signals from Step 33A
support_text = (
    df["support_response"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["support_len"] = support_text.str.len()

df["asks_for_details"] = support_text.str.contains(
    r"share.*details|provide.*details|send.*details|"
    r"dm us|direct message|private message|"
    r"contact us|email us|call us",
    regex=True
)

df["has_action_language"] = support_text.str.contains(
    r"please|check|try|follow|contact|call|email|"
    r"update|reset|change|return|refund|replace|"
    r"send|share|provide|visit",
    regex=True
)

df["has_policy_language"] = support_text.str.contains(
    r"unable|cannot|can't|don't have|"
    r"not possible|policy|eligible|available",
    regex=True
)

df["has_security_language"] = support_text.str.contains(
    r"security|scam|fraud|password|personal information|"
    r"account.*security|do not provide",
    regex=True
)

df["ends_with_support"] = df["final_role"] == "support"
df["ends_with_customer"] = df["final_role"] == "customer"

# --------------------------------------------
# Provisional buckets
# --------------------------------------------

conditions = [
    # Potential unresolved
    df["ends_with_customer"],

    # Investigation / handoff
    (
        df["ends_with_support"] &
        df["asks_for_details"]
    ),

    # Stronger evidence candidate
    (
        df["ends_with_support"] &
        (
            df["has_action_language"] |
            df["has_policy_language"]
        ) &
        (~df["asks_for_details"])
    )
]

choices = [
    "POTENTIAL_UNRESOLVED",
    "FOLLOWUP_OR_INVESTIGATION",
    "RESOLUTION_CANDIDATE"
]

df["provisional_bucket"] = np.select(
    conditions,
    choices,
    default="WEAK_OR_OTHER"
)

print("Provisional bucket distribution:")

display(
    df["provisional_bucket"]
    .value_counts()
    .to_frame("case_count")
)

Provisional bucket distribution:


,case_count
provisional_bucket,
WEAK_OR_OTHER,32241
RESOLUTION_CANDIDATE,26053
POTENTIAL_UNRESOLVED,17408
FOLLOWUP_OR_INVESTIGATION,7716


In [ ]:
bucket_summary = (
    df["provisional_bucket"]
    .value_counts()
    .to_frame("case_count")
)

bucket_summary["percentage"] = (
    bucket_summary["case_count"]
    / len(df)
    * 100
).round(2)

display(bucket_summary)

,case_count,percentage
provisional_bucket,,
WEAK_OR_OTHER,32241,38.65
RESOLUTION_CANDIDATE,26053,31.23
POTENTIAL_UNRESOLVED,17408,20.87
FOLLOWUP_OR_INVESTIGATION,7716,9.25


In [ ]:
# ============================================
# STEP 35 — Stratified Human Validation Sample
# ============================================

samples_per_bucket = 50

validation_sample = (
    df.groupby(
        "provisional_bucket",
        group_keys=False
    )
    .apply(
        lambda x: x.sample(
            min(samples_per_bucket, len(x)),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print(
    "Validation sample size:",
    len(validation_sample)
)

display(
    validation_sample[
        [
            "case_id",
            "provisional_bucket",
            "turn_count",
            "customer_query",
            "support_response"
        ]
    ].head(20)
)

Validation sample size: 200


/tmp/ipykernel_2236/917968081.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,case_id,provisional_bucket,turn_count,customer_query,support_response
0,AMZ-020425,FOLLOWUP_OR_INVESTIGATION,3,@115830 why does it need to be so hard to fin...,@562936 2/2 Note: these are not Freephone num...
1,AMZ-005250,FOLLOWUP_OR_INVESTIGATION,3,@AmazonHelp @115850 supplier MK Laboratories i...,@192217 Please don’t provide your order detail...
2,AMZ-040694,FOLLOWUP_OR_INVESTIGATION,2,@AmazonHelp It says my parcel was delivered to...,@472833 That's interesting! We would like to l...
3,AMZ-006979,FOLLOWUP_OR_INVESTIGATION,2,@17629 @115850\nNeed help !\nPaid With card ! ...,@202269 We're sorry for the payment issue you'...
4,AMZ-069869,FOLLOWUP_OR_INVESTIGATION,2,@115821 well done great job !! Box looks like ...,@710779 I'm sorry your order arrived in this c...
5,AMZ-025411,FOLLOWUP_OR_INVESTIGATION,3,@115850 MyPickupForPhone HasNotDoneInPast3Days...,"@354046 Also, please don't provide your order/..."
6,AMZ-030175,FOLLOWUP_OR_INVESTIGATION,2,Not happy with whoever hacked my @115821 accou...,@414996 Sorry to hear about the acc issue. Ple...
7,AMZ-058917,FOLLOWUP_OR_INVESTIGATION,8,"@AmazonHelp Hi there, it won’t let me put in a...",@197677 Very sorry for the misdirecting link. ...
8,AMZ-027103,FOLLOWUP_OR_INVESTIGATION,4,@AmazonHelp I need the item delivered today. R...,"@382154 I understand your concern, Deen. Pleas..."
9,AMZ-031473,FOLLOWUP_OR_INVESTIGATION,2,@AmazonHelp hello this is my order number\n 1...,@426336 Please don't provide your order detail...


In [ ]:
# ============================================
# STEP 36 — Human Annotation Template
# ============================================

annotation_df = validation_sample[
    [
        "case_id",
        "root_tweet_id",
        "turn_count",
        "customer_turns",
        "support_turns",
        "duration_hours",
        "max_gap_hours",
        "customer_query",
        "support_response",
        "provisional_bucket"
    ]
].copy()

# Human labels — initially empty
annotation_df["resolution_label"] = ""
annotation_df["resolution_type"] = ""
annotation_df["risk_label"] = ""
annotation_df["notes"] = ""

display(annotation_df.head())

,case_id,root_tweet_id,turn_count,customer_turns,support_turns,duration_hours,max_gap_hours,customer_query,support_response,provisional_bucket,resolution_label,resolution_type,risk_label,notes
0,AMZ-020425,1887476,3,1,2,0.176389,0.167778,@115830 why does it need to be so hard to fin...,@562936 2/2 Note: these are not Freephone num...,FOLLOWUP_OR_INVESTIGATION,,,,
1,AMZ-005250,319698,3,1,2,1.658611,1.648611,@AmazonHelp @115850 supplier MK Laboratories i...,@192217 Please don’t provide your order detail...,FOLLOWUP_OR_INVESTIGATION,,,,
2,AMZ-040694,1521180,2,1,1,0.187778,0.187778,@AmazonHelp It says my parcel was delivered to...,@472833 That's interesting! We would like to l...,FOLLOWUP_OR_INVESTIGATION,,,,
3,AMZ-006979,363997,2,1,1,1.760833,1.760833,@17629 @115850\nNeed help !\nPaid With card ! ...,@202269 We're sorry for the payment issue you'...,FOLLOWUP_OR_INVESTIGATION,,,,
4,AMZ-069869,2488998,2,1,1,0.067222,0.067222,@115821 well done great job !! Box looks like ...,@710779 I'm sorry your order arrived in this c...,FOLLOWUP_OR_INVESTIGATION,,,,


In [ ]:
# ============================================================
# STEP 35 — BUILD GOLDEN EVALUATION SET
# ============================================================

import os
import pandas as pd
import numpy as np

# Work from the dataframe created in previous steps
source_df = df.copy()

# ------------------------------------------------------------
# 1. Stratified sampling
# ------------------------------------------------------------

BUCKETS = [
    "RESOLUTION_CANDIDATE",
    "FOLLOWUP_OR_INVESTIGATION",
    "POTENTIAL_UNRESOLVED",
    "WEAK_OR_OTHER"
]

SAMPLES_PER_BUCKET = 50
RANDOM_STATE = 42

golden_parts = []

for bucket in BUCKETS:

    bucket_df = source_df[
        source_df["provisional_bucket"] == bucket
    ].copy()

    n = min(SAMPLES_PER_BUCKET, len(bucket_df))

    sample = bucket_df.sample(
        n=n,
        random_state=RANDOM_STATE
    )

    golden_parts.append(sample)

golden_df = pd.concat(
    golden_parts,
    ignore_index=True
)

# Shuffle final dataset so buckets aren't grouped together
golden_df = golden_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Add annotation columns
# ------------------------------------------------------------

golden_df["gold_intent"] = ""
golden_df["resolution_quality"] = ""
golden_df["risk_level"] = ""
golden_df["should_auto_handle"] = ""
golden_df["annotation_notes"] = ""

# ------------------------------------------------------------
# 3. Add stable annotation ID
# ------------------------------------------------------------

golden_df.insert(
    0,
    "golden_id",
    [
        f"GOLD-{i:04d}"
        for i in range(1, len(golden_df) + 1)
    ]
)

# ------------------------------------------------------------
# 4. Select useful columns
# ------------------------------------------------------------

golden_df = golden_df[
    [
        "golden_id",
        "case_id",
        "root_tweet_id",
        "turn_count",
        "customer_turns",
        "support_turns",
        "duration_hours",
        "max_gap_hours",
        "customer_query",
        "support_response",
        "provisional_bucket",
        "gold_intent",
        "resolution_quality",
        "risk_level",
        "should_auto_handle",
        "annotation_notes"
    ]
]

# ------------------------------------------------------------
# 5. Save
# ------------------------------------------------------------

EVAL_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "evaluation"
)

os.makedirs(EVAL_DIR, exist_ok=True)

GOLDEN_PATH = os.path.join(
    EVAL_DIR,
    "golden_evaluation_set.csv"
)

golden_df.to_csv(
    GOLDEN_PATH,
    index=False
)

# ------------------------------------------------------------
# 6. Verification
# ------------------------------------------------------------

print("=" * 70)
print("GOLDEN EVALUATION SET CREATED")
print("=" * 70)

print(f"Total examples : {len(golden_df)}")
print()

print("Distribution:")
print(
    golden_df["provisional_bucket"]
    .value_counts()
)

print()
print("Saved to:")
print(GOLDEN_PATH)

print()
print("First 10 examples:")
display(
    golden_df[
        [
            "golden_id",
            "case_id",
            "provisional_bucket",
            "customer_query",
            "support_response"
        ]
    ].head(10)
)

GOLDEN EVALUATION SET CREATED
Total examples : 200

Distribution:
provisional_bucket
FOLLOWUP_OR_INVESTIGATION    50
RESOLUTION_CANDIDATE         50
WEAK_OR_OTHER                50
POTENTIAL_UNRESOLVED         50
Name: count, dtype: int64

Saved to:
/content/drive/MyDrive/Hiver_Support_Agent/data/evaluation/golden_evaluation_set.csv

First 10 examples:


,golden_id,case_id,provisional_bucket,customer_query,support_response
0,GOLD-0001,AMZ-012695,FOLLOWUP_OR_INVESTIGATION,Contact @115821 regarding payment on an order....,@311616 That is definitely not the experience ...
1,GOLD-0002,AMZ-072455,RESOLUTION_CANDIDATE,@AmazonHelp LOL...you've missed the delivery d...,@797722 I'm sorry for the recent delays. Pleas...
2,GOLD-0003,AMZ-034151,RESOLUTION_CANDIDATE,So bummed that my package was delayed even wit...,@682197 Very sorry to hear. Were you provided ...
3,GOLD-0004,AMZ-076394,WEAK_OR_OTHER,I love @115821 for my Christmas shopping gift ...,@819368 Let us know if we can help you find an...
4,GOLD-0005,AMZ-064736,POTENTIAL_UNRESOLVED,@AmazonHelp Bleg on the phone 😕 what version s...,@284280 The best place to answer your question...
5,GOLD-0006,AMZ-037296,POTENTIAL_UNRESOLVED,@115850 amazon stick from https://t.co/cvJQ06p...,@120462 You can find the list of authorized se...
6,GOLD-0007,AMZ-006548,FOLLOWUP_OR_INVESTIGATION,@117795 Unethical delivery setup. Worst cust c...,@199458 Please don't provide your order detail...
7,GOLD-0008,AMZ-036798,WEAK_OR_OTHER,@AmazonHelp expected delivery yesterday and it...,"@757634 When you go to Your Orders, what's the..."
8,GOLD-0009,AMZ-050849,WEAK_OR_OTHER,@667472 @115833 My 5 year old son ordered a Ni...,@667471 Oh my! You may want to restrict voice ...
9,GOLD-0010,AMZ-004813,RESOLUTION_CANDIDATE,#KaroMilkeLateDelivery is the mantra of @11585...,@183457 shared here: https://t.co/NTkrxpsbHJ a...


In [ ]:
# ============================================================
# STEP 36 — INSPECT GOLDEN SET (FIRST 20 CASES)
# ============================================================

pd.set_option("display.max_colwidth", 500)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

inspection_cols = [
    "golden_id",
    "case_id",
    "provisional_bucket",
    "turn_count",
    "customer_query",
    "support_response"
]

batch_1 = golden_df[inspection_cols].head(20).copy()

display(batch_1)

,golden_id,case_id,provisional_bucket,turn_count,customer_query,support_response
0,GOLD-0001,AMZ-012695,FOLLOWUP_OR_INVESTIGATION,2,"Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!",@311616 That is definitely not the experience we want to hear about. Can you provide us more details? Please no account info. ^CH
1,GOLD-0002,AMZ-072455,RESOLUTION_CANDIDATE,4,"@AmazonHelp LOL...you've missed the delivery date of every order of mine shipped via usps, for the last 5 months including todays order.",@797722 I'm sorry for the recent delays. Please get in touch with us here so we can leave carrier feedback on your behalf and go over any available options: https://t.co/NlKbkFnOK5 ^CL
2,GOLD-0003,AMZ-034151,RESOLUTION_CANDIDATE,2,So bummed that my package was delayed even with Amazon Prime shipping. Now I don't have a costume for tomorrow. ☹️,@682197 Very sorry to hear. Were you provided a revised delivery date? ^TP
3,GOLD-0004,AMZ-076394,WEAK_OR_OTHER,2,I love @115821 for my Christmas shopping gift needs.,"@819368 Let us know if we can help you find anything, Christine! ^BL"
4,GOLD-0005,AMZ-064736,POTENTIAL_UNRESOLVED,7,@AmazonHelp Bleg on the phone 😕 what version should it be showing?,@284280 The best place to answer your questions would be to contact us via the link above. Our Echo specialist can review this with you in real time.^SM
5,GOLD-0006,AMZ-037296,POTENTIAL_UNRESOLVED,3,@115850 amazon stick from https://t.co/cvJQ06p7ve is authorized seller?,@120462 You can find the list of authorized seller's here: https://t.co/JotO6OLmGA ^SH
6,GOLD-0007,AMZ-006548,FOLLOWUP_OR_INVESTIGATION,3,"@117795 Unethical delivery setup. Worst cust care. Order no 404-1061350-8629132 against canceled order at high cost, late delivery.",@199458 Please don't provide your order details as we consider it personal info. Our twitter page is visible to public.(2/2) ^CB
7,GOLD-0008,AMZ-036798,WEAK_OR_OTHER,4,@AmazonHelp expected delivery yesterday and it's not even there. This was a gift. So much for that.,"@757634 When you go to Your Orders, what's the shipping status? Was this being shipped or fullfilled by Amazon? ^EM"
8,GOLD-0009,AMZ-050849,WEAK_OR_OTHER,2,@667472 @115833 My 5 year old son ordered a Nintendo switch on Alexa.,"@667471 Oh my! You may want to restrict voice purchasing, here's how: https://t.co/7TspTEfzim"
9,GOLD-0010,AMZ-004813,RESOLUTION_CANDIDATE,3,#KaroMilkeLateDelivery is the mantra of @115850 can say it with experience. They don't delivery at commmitted time &amp; give excuses https://t.co/6I4kO0jtRL,"@183457 shared here: https://t.co/NTkrxpsbHJ and reply to the email correspondence, we'll reach out. (2/2) ^KA"


In [ ]:
# ============================================================
# STEP 36B — READABLE CONVERSATION VIEW
# ============================================================

for _, row in batch_1.iterrows():

    print("\n" + "=" * 100)
    print(
        f"{row['golden_id']} | "
        f"{row['case_id']} | "
        f"Bucket: {row['provisional_bucket']} | "
        f"Turns: {row['turn_count']}"
    )
    print("=" * 100)

    print("\nCUSTOMER:")
    print(row["customer_query"])

    print("\nHISTORICAL SUPPORT RESPONSE:")
    print(row["support_response"])

    print()


GOLD-0001 | AMZ-012695 | Bucket: FOLLOWUP_OR_INVESTIGATION | Turns: 2

CUSTOMER:
Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!

HISTORICAL SUPPORT RESPONSE:
@311616 That is definitely not the experience we want to hear about.  Can you provide us more details? Please no account info. ^CH


GOLD-0002 | AMZ-072455 | Bucket: RESOLUTION_CANDIDATE | Turns: 4

CUSTOMER:
@AmazonHelp LOL...you've missed the delivery date of every order of mine  shipped via usps, for the last 5 months including todays order.

HISTORICAL SUPPORT RESPONSE:
@797722 I'm sorry for the recent delays. Please get in touch with us here so we can leave carrier feedback on your behalf and go over any available options: https://t.co/NlKbkFnOK5 ^CL


GOLD-0003 | AMZ-034151 | Bucket: RESOLUTION_CANDIDATE | Turns: 2

CUSTOMER:
So bummed that my package was delayed even with Amazon Prime shipping. Now I don't have a costume for tomor

In [ ]:
# ============================================================
# STEP 36C — PRINT ALL 20 GOLDEN CASES CLEANLY
# ============================================================

for i, (_, row) in enumerate(batch_1.iterrows(), start=1):

    print("\n" + "#" * 100)
    print(
        f"CASE {i}/20 | {row['golden_id']} | {row['case_id']}"
    )
    print(
        f"PROVISIONAL: {row['provisional_bucket']} | "
        f"TURNS: {row['turn_count']}"
    )
    print("#" * 100)

    print("\nCUSTOMER:")
    print(row["customer_query"])

    print("\nSUPPORT:")
    print(row["support_response"])

    print()


####################################################################################################
CASE 1/20 | GOLD-0001 | AMZ-012695
PROVISIONAL: FOLLOWUP_OR_INVESTIGATION | TURNS: 2
####################################################################################################

CUSTOMER:
Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!

SUPPORT:
@311616 That is definitely not the experience we want to hear about.  Can you provide us more details? Please no account info. ^CH


####################################################################################################
CASE 2/20 | GOLD-0002 | AMZ-072455
PROVISIONAL: RESOLUTION_CANDIDATE | TURNS: 4
####################################################################################################

CUSTOMER:
@AmazonHelp LOL...you've missed the delivery date of every order of mine  shipped via usps, for the last 5 months including 

In [ ]:
# ============================================================
# STEP 37 — BATCH 1 GOLD ANNOTATIONS
# ============================================================

batch1_annotations = {
    "GOLD-0001": {
        "intent": "PAYMENT_BILLING",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },
    "GOLD-0002": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },
    "GOLD-0003": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },
    "GOLD-0004": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
    },
    "GOLD-0005": {
        "intent": "DEVICE_TECHNICAL",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },
    "GOLD-0006": {
        "intent": "SELLER_AUTHENTICITY",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
    },
    "GOLD-0007": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "UNRESOLVED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
    },
    "GOLD-0008": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },
    "GOLD-0009": {
        "intent": "PURCHASE_CONTROL",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },
    "GOLD-0010": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },
    "GOLD-0011": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },
    "GOLD-0012": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },
    "GOLD-0013": {
        "intent": "ORDER_PURCHASE",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },
    "GOLD-0014": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },
    "GOLD-0015": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
    },
    "GOLD-0016": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
    },
    "GOLD-0017": {
        "intent": "ACCOUNT_SECURITY",
        "resolution_status": "ESCALATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
    },
    "GOLD-0018": {
        "intent": "PREORDER_DELIVERY",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
    },
    "GOLD-0019": {
        "intent": "ORDER_TRACKING",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },
    "GOLD-0020": {
        "intent": "MARKETPLACE_SELLING",
        "resolution_status": "UNRESOLVED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },
}

print("Batch 1 annotations created:", len(batch1_annotations))

Batch 1 annotations created: 20


In [ ]:
# ============================================================
# STEP 38 — APPLY BATCH 1 ANNOTATIONS
# ============================================================

for gold_id, annotation in batch1_annotations.items():
    mask = golden_df["golden_id"] == gold_id

    for column, value in annotation.items():
        golden_df.loc[mask, column] = value


print("=" * 70)
print("BATCH 1 ANNOTATIONS APPLIED")
print("=" * 70)

print("Rows updated:", golden_df["golden_id"].isin(batch1_annotations.keys()).sum())

print("\nBatch 1 preview:")
display(
    golden_df[
        golden_df["golden_id"].isin(batch1_annotations.keys())
    ][[
        "golden_id",
        "case_id",
        "provisional_bucket",
        "intent",
        "resolution_status",
        "risk_level",
        "automation_decision"
    ]].head(20)
)

BATCH 1 ANNOTATIONS APPLIED
Rows updated: 20

Batch 1 preview:


,golden_id,case_id,provisional_bucket,intent,resolution_status,risk_level,automation_decision
0,GOLD-0001,AMZ-012695,FOLLOWUP_OR_INVESTIGATION,PAYMENT_BILLING,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
1,GOLD-0002,AMZ-072455,RESOLUTION_CANDIDATE,ORDER_DELIVERY,PARTIALLY_RESOLVED,LOW,HUMAN
2,GOLD-0003,AMZ-034151,RESOLUTION_CANDIDATE,ORDER_DELIVERY,INVESTIGATION_REQUIRED,LOW,HUMAN
3,GOLD-0004,AMZ-076394,WEAK_OR_OTHER,GENERAL_SOCIAL,NOT_APPLICABLE,LOW,AUTO
4,GOLD-0005,AMZ-064736,POTENTIAL_UNRESOLVED,DEVICE_TECHNICAL,INVESTIGATION_REQUIRED,LOW,HUMAN
5,GOLD-0006,AMZ-037296,POTENTIAL_UNRESOLVED,SELLER_AUTHENTICITY,RESOLVED,LOW,AUTO
6,GOLD-0007,AMZ-006548,FOLLOWUP_OR_INVESTIGATION,ORDER_DELIVERY,UNRESOLVED,HIGH,HUMAN
7,GOLD-0008,AMZ-036798,WEAK_OR_OTHER,ORDER_DELIVERY,INVESTIGATION_REQUIRED,LOW,HUMAN
8,GOLD-0009,AMZ-050849,WEAK_OR_OTHER,PURCHASE_CONTROL,PARTIALLY_RESOLVED,MEDIUM,HUMAN
9,GOLD-0010,AMZ-004813,RESOLUTION_CANDIDATE,ORDER_DELIVERY,PARTIALLY_RESOLVED,LOW,HUMAN


In [ ]:
# ============================================================
# STEP 39 — INSPECT GOLD BATCH 2
# GOLD-0021 → GOLD-0040
# ============================================================

batch2_df = golden_df[
    (golden_df["golden_id"] >= "GOLD-0021") &
    (golden_df["golden_id"] <= "GOLD-0040")
].copy()

print("=" * 80)
print("GOLD BATCH 2 | GOLD-0021 → GOLD-0040")
print("=" * 80)
print("Total examples:", len(batch2_df))

display(
    batch2_df[[
        "golden_id",
        "case_id",
        "provisional_bucket",
        "turn_count",
        "customer_query",
        "support_response"
    ]]
)

GOLD BATCH 2 | GOLD-0021 → GOLD-0040
Total examples: 20


,golden_id,case_id,provisional_bucket,turn_count,customer_query,support_response
20,GOLD-0021,AMZ-026795,POTENTIAL_UNRESOLVED,5,"@AmazonHelp I still can not access my account. This has been going on over a month. Told I would be contacted, but I have not.",@216862 I'm sorry this hasn't been resolved. Have you received an e-mail from our Account Specialists since contacting us? ^MO
21,GOLD-0022,AMZ-058405,RESOLUTION_CANDIDATE,6,@AmazonHelp I've. Still awaiting a proper response!,@751296 We'll check this and get back to you. ^SG
22,GOLD-0023,AMZ-066755,POTENTIAL_UNRESOLVED,3,"@115851 @AmazonHelp @115850 #jeffbezos be honest, earn respect!!!\nTake action against bad behavior done by your customer care person!!!\n\nDont escape from responsibility...dont be coward be brave!!!!",@147113 Sorry about the unpleasant experience. We'd like to take a closer look at this. Kindly fill in your details here: https://t.co/GIJyeYqKE0. ^NR
23,GOLD-0024,AMZ-012950,FOLLOWUP_OR_INVESTIGATION,2,@117795 your delivery person wouldn't answer his phone and cancelled my order. Why can't I cancel his tip?,"@313375 Oh no! Sorry about the poor experience, Maiya! Please contact us via the Prime Now app for assistance! We'd like to help! ^AD"
24,GOLD-0025,AMZ-067459,FOLLOWUP_OR_INVESTIGATION,3,@115850 #ReceivedDefectiveProduct #VerifiedByaAmazonTecnician #HandedProducttoamazoncourierboy #ProductSlipgivenbydeliveryboy #Correctproductdeliveredtowearhouse #emptyboxatfullfilmentcenter #withoutanydefault #I'm #facingissues... \n#IwantMyProductRefund!! https://t.co/Xl1PHGR49Q,@199900 Apologies for the hassle regarding the returned product. We'd like to investigate this issue. Kindly share your details here: - https://t.co/beaaDm0muc and we'll get in touch with you at the earliest. 1/2^SC
25,GOLD-0026,AMZ-060927,FOLLOWUP_OR_INVESTIGATION,2,"@AmazonHelp here is my order details 402-0242407-4245951 pls tell me the status , because yesterday I was not available so plz.thanks","@772749 We wouldn't be able to access your Amazon account details on Twitter. Kindly contact our support team here: https://t.co/vlvfJr4nN9 &amp; we'll be glad to help you. Please don't provide your order details, we consider it to be personal information. ^SY"
26,GOLD-0027,AMZ-071328,FOLLOWUP_OR_INVESTIGATION,2,@AmazonHelp I'm confused... https://t.co/ikSK76AvrW,@790379 We'd like to look into the delivery for you. Please contact us at: https://t.co/JzP7hlA23B ^LI
27,GOLD-0028,AMZ-021614,FOLLOWUP_OR_INVESTIGATION,4,"@115850 Tracking #: 5180712005991\nPlease update the status of this tracking number, one of the worst experience i am facing with amazon","@629953 Please don't provide your tracking details, we consider it personal information. Our Twitter page is visible to public. ^MJ"
28,GOLD-0029,AMZ-026679,POTENTIAL_UNRESOLVED,4,"@115821 / @AmazonHelp - Two day delivery is the promise, not “two days + processing + transit time.” Feeling like a 2nd class Prime member. https://t.co/tflnDw0Uin",@375664 (2/2) details here: https://t.co/hQdlywCsO7 ^AR
29,GOLD-0030,AMZ-040351,POTENTIAL_UNRESOLVED,6,@AmazonHelp You’ve dropped the ball on this one. I have cancelled my Amazon order and picked the game up from @sainsburys for similar price 👍,@467933 Please again accept our apologies ^TD


In [ ]:
# ============================================================
# STEP 40 — BATCH 2 GOLD ANNOTATIONS
# GOLD-0021 → GOLD-0040
# ============================================================

batch2_annotations = {

    "GOLD-0021": {
        "intent": "ACCOUNT_ACCESS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },

    "GOLD-0022": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },

    "GOLD-0023": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },

    "GOLD-0024": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },

    "GOLD-0025": {
        "intent": "RETURNS_REFUNDS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },

    "GOLD-0026": {
        "intent": "ORDER_TRACKING",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },

    "GOLD-0027": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },

    "GOLD-0028": {
        "intent": "ORDER_TRACKING",
        "resolution_status": "UNRESOLVED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },

    "GOLD-0029": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },

    "GOLD-0030": {
        "intent": "ORDER_CANCELLATION",
        "resolution_status": "RESOLVED_BY_CUSTOMER",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
    },

    "GOLD-0031": {
        "intent": "RETURNS_REFUNDS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },

    "GOLD-0032": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
    },

    "GOLD-0033": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "UNRESOLVED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },

    "GOLD-0034": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "UNRESOLVED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },

    "GOLD-0035": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },

    "GOLD-0036": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
    },

    "GOLD-0037": {
        "intent": "RETURNS_REFUNDS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },

    "GOLD-0038": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },

    "GOLD-0039": {
        "intent": "DIGITAL_CONTENT",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
    },

    "GOLD-0040": {
        "intent": "RETURNS_REFUNDS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
    },
}

print("=" * 70)
print("BATCH 2 ANNOTATIONS CREATED")
print("=" * 70)
print("Total annotations:", len(batch2_annotations))

BATCH 2 ANNOTATIONS CREATED
Total annotations: 20


In [ ]:
# ============================================================
# STEP 41 — APPLY BATCH 2 ANNOTATIONS
# ============================================================

for gold_id, annotation in batch2_annotations.items():

    mask = golden_df["golden_id"] == gold_id

    for column, value in annotation.items():
        golden_df.loc[mask, column] = value


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

batch2_ids = set(batch2_annotations.keys())

updated_count = golden_df["golden_id"].isin(batch2_ids).sum()

print("=" * 70)
print("BATCH 2 ANNOTATIONS APPLIED")
print("=" * 70)

print("Batch 2 rows updated:", updated_count)

# Count completed annotations across the whole golden set
annotation_columns = [
    "intent",
    "resolution_status",
    "risk_level",
    "automation_decision"
]

completed_mask = (
    golden_df[annotation_columns]
    .notna()
    .all(axis=1)
    &
    (golden_df[annotation_columns] != "")
    .all(axis=1)
)

print("Total completed annotations:", completed_mask.sum())
print("Remaining:", len(golden_df) - completed_mask.sum())

print("\nBatch 2 preview:")

display(
    golden_df[
        golden_df["golden_id"].isin(batch2_ids)
    ][[
        "golden_id",
        "case_id",
        "provisional_bucket",
        "intent",
        "resolution_status",
        "risk_level",
        "automation_decision"
    ]]
)

BATCH 2 ANNOTATIONS APPLIED
Batch 2 rows updated: 20
Total completed annotations: 40
Remaining: 160

Batch 2 preview:


,golden_id,case_id,provisional_bucket,intent,resolution_status,risk_level,automation_decision
20,GOLD-0021,AMZ-026795,POTENTIAL_UNRESOLVED,ACCOUNT_ACCESS,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
21,GOLD-0022,AMZ-058405,RESOLUTION_CANDIDATE,GENERAL_SUPPORT,INVESTIGATION_REQUIRED,LOW,HUMAN
22,GOLD-0023,AMZ-066755,POTENTIAL_UNRESOLVED,GENERAL_SUPPORT,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
23,GOLD-0024,AMZ-012950,FOLLOWUP_OR_INVESTIGATION,ORDER_DELIVERY,PARTIALLY_RESOLVED,LOW,HUMAN
24,GOLD-0025,AMZ-067459,FOLLOWUP_OR_INVESTIGATION,RETURNS_REFUNDS,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
25,GOLD-0026,AMZ-060927,FOLLOWUP_OR_INVESTIGATION,ORDER_TRACKING,INVESTIGATION_REQUIRED,LOW,HUMAN
26,GOLD-0027,AMZ-071328,FOLLOWUP_OR_INVESTIGATION,ORDER_DELIVERY,INVESTIGATION_REQUIRED,LOW,HUMAN
27,GOLD-0028,AMZ-021614,FOLLOWUP_OR_INVESTIGATION,ORDER_TRACKING,UNRESOLVED,LOW,HUMAN
28,GOLD-0029,AMZ-026679,POTENTIAL_UNRESOLVED,ORDER_DELIVERY,INVESTIGATION_REQUIRED,LOW,HUMAN
29,GOLD-0030,AMZ-040351,POTENTIAL_UNRESOLVED,ORDER_CANCELLATION,RESOLVED_BY_CUSTOMER,LOW,AUTO


In [ ]:
# ============================================================
# BATCH 3 INSPECTION — GOLD-0041 to GOLD-0060
# ============================================================

batch3_ids = [f"GOLD-{i:04d}" for i in range(41, 61)]

batch3 = golden_df[
    golden_df["golden_id"].isin(batch3_ids)
].copy()

print("=" * 100)
print("BATCH 3 — GOLD-0041 → GOLD-0060")
print("=" * 100)
print(f"Cases found: {len(batch3)}")

for _, row in batch3.sort_values("golden_id").iterrows():
    print("\n" + "=" * 100)
    print(f"{row['golden_id']} | Case: {row['case_id']} | "
          f"Turns: {row['turn_count']} | "
          f"Provisional: {row['provisional_bucket']}")
    print("-" * 100)

    print("CUSTOMER QUERY:")
    print(row["customer_query"])

    print("\nSUPPORT RESPONSE:")
    print(row["support_response"])

    print("\nMETADATA:")
    print(f"Customer turns : {row['customer_turns']}")
    print(f"Support turns  : {row['support_turns']}")
    print(f"Duration       : {row['duration_hours']:.2f} hours")

BATCH 3 — GOLD-0041 → GOLD-0060
Cases found: 20

GOLD-0041 | Case: AMZ-003240 | Turns: 2 | Provisional: FOLLOWUP_OR_INVESTIGATION
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
Watched the Amazon delivery lady toss my package (soft envelope) onto my doormat like she was playing beanbag toss at a festival. #Amazon

SUPPORT RESPONSE:
@180717 I'm so sorry! We'd like to look into this for you. Please provide your order/contact details here: https://t.co/xO0B5FRWbV ^AG

METADATA:
Customer turns : 1
Support turns  : 1
Duration       : 0.76 hours

GOLD-0042 | Case: AMZ-060655 | Turns: 4 | Provisional: WEAK_OR_OTHER
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
@AmazonHelp わかりますが、同じ表品を2セットで別はやっぱり嫌です…
開けると「入るやん」って思ってしまいますし…
でも、わざわざありがとうございます😊

SUPPORT RESPONSE:
@781505 とんでもないことでございます。ご意見をお寄せいただき、ありがとうございます。ご指摘いただいた点は担当部署に伝えさせていただき、今後もサービスの改善に努めて参ります。 YM

M

In [ ]:
# ============================================================
# BATCH 3 ANNOTATIONS — GOLD-0041 to GOLD-0060
# ============================================================

batch3_annotations = {

    "GOLD-0041": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports careless package handling. Support requests private order/contact details for investigation."
    },

    "GOLD-0042": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer is giving product/service feedback rather than requesting a concrete support action. Support acknowledges and forwards feedback."
    },

    "GOLD-0043": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Package is late and delivery attempt appears incorrect. Support needs order-level investigation."
    },

    "GOLD-0044": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer query is too vague to determine the underlying issue. Support appropriately requests investigation through a secure channel."
    },

    "GOLD-0045": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer provides packaging feedback. Support acknowledges the issue and directs the customer to the feedback channel."
    },

    "GOLD-0046": {
        "intent": "PAYMENT_BILLING",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports delayed delivery together with an extra charge/payment-related status. Details are unclear and require account/order investigation."
    },

    "GOLD-0047": {
        "intent": "ORDER_TRACKING",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Orders show as delivered but were not received. Support provides troubleshooting steps and asks customer to follow up, so case remains dependent on future verification."
    },

    "GOLD-0048": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports repeated missed guaranteed delivery dates. Support needs to investigate the specific order."
    },

    "GOLD-0049": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer complains about poor delivery handling. Support requests direct contact to investigate."
    },

    "GOLD-0050": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer appears to have an urgent order issue but provides insufficient context. Support requests private details and explicitly warns against posting personal information publicly."
    },

    "GOLD-0051": {
        "intent": "DEVICE_TECHNICAL",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports application/device functionality problems across multiple machines. Support proposes troubleshooting but issue is not confirmed resolved."
    },

    "GOLD-0052": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Support directly explains the distinction between dispatch time and Prime delivery time and provides additional information."
    },

    "GOLD-0053": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports missed promised delivery time and current out-for-delivery delay. Support asks for carrier/order context before determining the issue."
    },

    "GOLD-0054": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "UNRESOLVED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer complains about delivery arriving outside the promised time window. Support does not provide a concrete resolution."
    },

    "GOLD-0055": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "RESOLVED_BY_CUSTOMER",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "The package was received; customer is mainly expressing dissatisfaction about delivery timing. Support acknowledges/apologizes."
    },

    "GOLD-0056": {
        "intent": "DEVICE_TECHNICAL",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports Kindle/iPad synchronization failure and missing highlights. Support redirects to real-time troubleshooting; issue remains unresolved."
    },

    "GOLD-0057": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "UNRESOLVED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports delivery failure and incorrect agent contact. Order information is present in the conversation, creating a privacy/security concern."
    },

    "GOLD-0058": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer says prior contact produced no result and asks Amazon to check records. Support requests private details for investigation."
    },

    "GOLD-0059": {
        "intent": "PREORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Release-day/preorder shipment has been delayed with no confirmed delivery date. Support says a date will be provided when confirmed."
    },

    "GOLD-0060": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Package is still out for delivery beyond the expected date. Support asks customer to contact them for further investigation."
    }
}

print("Batch 3 annotations prepared:", len(batch3_annotations))

Batch 3 annotations prepared: 20


In [ ]:
# ============================================================
# APPLY BATCH 3
# ============================================================

for gold_id, annotation in batch3_annotations.items():
    mask = golden_df["golden_id"] == gold_id

    for column, value in annotation.items():
        golden_df.loc[mask, column] = value

annotation_columns = [
    "intent",
    "resolution_status",
    "risk_level",
    "automation_decision"
]

completed_mask = (
    golden_df[annotation_columns].notna().all(axis=1)
    & (golden_df[annotation_columns] != "").all(axis=1)
)

print("=" * 70)
print("BATCH 3 ANNOTATIONS APPLIED")
print("=" * 70)
print(f"Batch 3 rows updated: {len(batch3_annotations)}")
print(f"Total completed annotations: {completed_mask.sum()}")
print(f"Remaining: {len(golden_df) - completed_mask.sum()}")

display(
    golden_df[
        golden_df["golden_id"].isin(batch3_annotations.keys())
    ][[
        "golden_id",
        "intent",
        "resolution_status",
        "risk_level",
        "automation_decision"
    ]].sort_values("golden_id")
)

BATCH 3 ANNOTATIONS APPLIED
Batch 3 rows updated: 20
Total completed annotations: 60
Remaining: 140


,golden_id,intent,resolution_status,risk_level,automation_decision
40,GOLD-0041,ORDER_DELIVERY,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
41,GOLD-0042,GENERAL_SUPPORT,NOT_APPLICABLE,LOW,AUTO
42,GOLD-0043,ORDER_DELIVERY,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
43,GOLD-0044,GENERAL_SUPPORT,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
44,GOLD-0045,GENERAL_SUPPORT,PARTIALLY_RESOLVED,LOW,AUTO
45,GOLD-0046,PAYMENT_BILLING,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
46,GOLD-0047,ORDER_TRACKING,PARTIALLY_RESOLVED,MEDIUM,HUMAN
47,GOLD-0048,ORDER_DELIVERY,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
48,GOLD-0049,ORDER_DELIVERY,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
49,GOLD-0050,ORDER_DELIVERY,INVESTIGATION_REQUIRED,MEDIUM,HUMAN


In [ ]:
# ============================================================
# BATCH 4 INSPECTION — GOLD-0061 to GOLD-0080
# ============================================================

batch4_ids = [f"GOLD-{i:04d}" for i in range(61, 81)]

batch4 = golden_df[
    golden_df["golden_id"].isin(batch4_ids)
].copy()

print("=" * 100)
print("BATCH 4 — GOLD-0061 → GOLD-0080")
print("=" * 100)
print(f"Cases found: {len(batch4)}")

for _, row in batch4.sort_values("golden_id").iterrows():

    print("\n" + "=" * 100)
    print(
        f"{row['golden_id']} | "
        f"Case: {row['case_id']} | "
        f"Turns: {row['turn_count']} | "
        f"Provisional: {row['provisional_bucket']}"
    )

    print("-" * 100)

    print("CUSTOMER QUERY:")
    print(row["customer_query"])

    print("\nSUPPORT RESPONSE:")
    print(row["support_response"])

    print("\nMETADATA:")
    print(f"Customer turns : {row['customer_turns']}")
    print(f"Support turns  : {row['support_turns']}")
    print(f"Duration       : {row['duration_hours']:.2f} hours")

BATCH 4 — GOLD-0061 → GOLD-0080
Cases found: 20

GOLD-0061 | Case: AMZ-000754 | Turns: 4 | Provisional: WEAK_OR_OTHER
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
@AmazonHelp Thanks for your early response. I contacted the Customer Service and they've assured to hlp me.

SUPPORT RESPONSE:
@158383 Do keep us posted for any further issues. ^MP

METADATA:
Customer turns : 2
Support turns  : 2
Duration       : 4.73 hours

GOLD-0062 | Case: AMZ-038191 | Turns: 2 | Provisional: FOLLOWUP_OR_INVESTIGATION
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
Here's what I ordered: two "ten piece" packs of bulbs vs. what @115821 sent me: two individual bulbs. 😡 https://t.co/PSIA3q7Mel

SUPPORT RESPONSE:
@449843 I'm so sorry, Tim! Please contact us via phone or chat so we may see what options are available: https://t.co/hApLpMlfHN ^KL

METADATA:
Customer turns : 

In [ ]:
# ============================================================
# BATCH 4 ANNOTATIONS — GOLD-0061 to GOLD-0080
# ============================================================

batch4_annotations = {

    "GOLD-0061": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer confirms Customer Service has already assured them of help. Support asks them to keep posted for further issues."
    },

    "GOLD-0062": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer received fewer items than ordered. Support needs order-level investigation and offers phone/chat assistance."
    },

    "GOLD-0063": {
        "intent": "PAYMENT_BILLING",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer asks about GST/business purchasing. Support explains the Amazon Business route and provides the relevant registration information."
    },

    "GOLD-0064": {
        "intent": "RETURNS_REFUNDS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer has requested a refund but selected the wrong return drop-off location. Support needs to inspect the return situation."
    },

    "GOLD-0065": {
        "intent": "ORDER_CANCELLATION",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer asks when money will be returned after cancelling an unshipped purchase. Support provides refund-processing information."
    },

    "GOLD-0066": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports poor customer service without specifying the underlying issue. Support asks for more information while warning against posting account/order details."
    },

    "GOLD-0067": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer is concerned about a product review not being posted. Support explains that reviews must follow the applicable guidelines."
    },

    "GOLD-0068": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer confirms completion of the previous interaction. Support simply invites them to return if further concerns arise."
    },

    "GOLD-0069": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports that another person's package arrived while theirs has not. Support directs them to private support for investigation."
    },

    "GOLD-0070": {
        "intent": "ACCOUNT_ACCESS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer cannot receive password reset codes and has already waited for support. Account recovery/security issue requires human handling."
    },

    "GOLD-0071": {
        "intent": "MARKETPLACE_SELLING",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer complains about a third-party supplier and includes order information. Support correctly moves away from public disclosure; marketplace complaint requires investigation."
    },

    "GOLD-0072": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer is expressing affection/appreciation rather than requesting support. Support responds socially."
    },

    "GOLD-0073": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer expresses appreciation and excitement about an upcoming delivery. Support responds conversationally."
    },

    "GOLD-0074": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports a delivery-related issue and confusion around Prime. Support needs to inspect the order and requests private details."
    },

    "GOLD-0075": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer explicitly states that the shipping problem was resolved and expresses satisfaction with the service."
    },

    "GOLD-0076": {
        "intent": "PURCHASE_CONTROL",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer says they will not renew Prime because of poor service. Support asks whether there is something they can help with, so the underlying issue remains unspecified."
    },

    "GOLD-0077": {
        "intent": "RETURNS_REFUNDS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer asks about the sequence of refund issuance and bank processing. Support requests further details to investigate."
    },

    "GOLD-0078": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer is praising a purchased book. Support responds conversationally and asks why they like it."
    },

    "GOLD-0079": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer is concerned that an order may miss its promised delivery time. Support asks them to follow up if it does not arrive."
    },

    "GOLD-0080": {
        "intent": "PURCHASE_CONTROL",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer asks whether an exchange is available for a recently launched phone. Support directly explains that no exchange availability has been announced."
    }
}

print("Batch 4 annotations prepared:", len(batch4_annotations))

Batch 4 annotations prepared: 20


In [ ]:
# ============================================================
# APPLY BATCH 4
# ============================================================

for gold_id, annotation in batch4_annotations.items():
    mask = golden_df["golden_id"] == gold_id

    for column, value in annotation.items():
        golden_df.loc[mask, column] = value

annotation_columns = [
    "intent",
    "resolution_status",
    "risk_level",
    "automation_decision"
]

completed_mask = (
    golden_df[annotation_columns].notna().all(axis=1)
    & (golden_df[annotation_columns] != "").all(axis=1)
)

print("=" * 70)
print("BATCH 4 ANNOTATIONS APPLIED")
print("=" * 70)
print(f"Batch 4 rows updated: {len(batch4_annotations)}")
print(f"Total completed annotations: {completed_mask.sum()}")
print(f"Remaining: {len(golden_df) - completed_mask.sum()}")

display(
    golden_df[
        golden_df["golden_id"].isin(batch4_annotations.keys())
    ][[
        "golden_id",
        "intent",
        "resolution_status",
        "risk_level",
        "automation_decision"
    ]].sort_values("golden_id")
)

BATCH 4 ANNOTATIONS APPLIED
Batch 4 rows updated: 20
Total completed annotations: 80
Remaining: 120


,golden_id,intent,resolution_status,risk_level,automation_decision
60,GOLD-0061,GENERAL_SUPPORT,PARTIALLY_RESOLVED,LOW,AUTO
61,GOLD-0062,ORDER_DELIVERY,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
62,GOLD-0063,PAYMENT_BILLING,RESOLVED,LOW,AUTO
63,GOLD-0064,RETURNS_REFUNDS,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
64,GOLD-0065,ORDER_CANCELLATION,RESOLVED,LOW,AUTO
65,GOLD-0066,GENERAL_SUPPORT,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
66,GOLD-0067,GENERAL_SUPPORT,PARTIALLY_RESOLVED,LOW,AUTO
67,GOLD-0068,GENERAL_SUPPORT,RESOLVED,LOW,AUTO
68,GOLD-0069,ORDER_DELIVERY,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
69,GOLD-0070,ACCOUNT_ACCESS,INVESTIGATION_REQUIRED,HIGH,HUMAN


In [ ]:
# ============================================================
# BATCH 5 INSPECTION — GOLD-0081 to GOLD-0100
# ============================================================

batch5_ids = [f"GOLD-{i:04d}" for i in range(81, 101)]

batch5 = golden_df[
    golden_df["golden_id"].isin(batch5_ids)
].copy()

print("=" * 100)
print("BATCH 5 — GOLD-0081 → GOLD-0100")
print("=" * 100)
print(f"Cases found: {len(batch5)}")

for _, row in batch5.sort_values("golden_id").iterrows():

    print("\n" + "=" * 100)
    print(
        f"{row['golden_id']} | "
        f"Case: {row['case_id']} | "
        f"Turns: {row['turn_count']} | "
        f"Provisional: {row['provisional_bucket']}"
    )

    print("-" * 100)

    print("CUSTOMER QUERY:")
    print(row["customer_query"])

    print("\nSUPPORT RESPONSE:")
    print(row["support_response"])

    print("\nMETADATA:")
    print(f"Customer turns : {row['customer_turns']}")
    print(f"Support turns  : {row['support_turns']}")
    print(f"Duration       : {row['duration_hours']:.2f} hours")

BATCH 5 — GOLD-0081 → GOLD-0100
Cases found: 20

GOLD-0081 | Case: AMZ-026370 | Turns: 2 | Provisional: RESOLUTION_CANDIDATE
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
@AmazonHelp third class help from you guys been chasing you guys since last 21 days without any help.पैसा ले कर निकल लिये?कहां जाओगे?

SUPPORT RESPONSE:
@370614 Sorry for the stretch. Please reply to our email for further help.  You may do so from here: https://t.co/8DAc10S7ww. ^HA

METADATA:
Customer turns : 1
Support turns  : 1
Duration       : 0.23 hours

GOLD-0082 | Case: AMZ-048538 | Turns: 2 | Provisional: FOLLOWUP_OR_INVESTIGATION
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
@115850 I ordered an I phone for my fiance for her birthday, I am a NRI and paid from my overseas credit card. Money has been deducted from my card, order is on hold, please help as I don't want my s

In [ ]:
# ============================================================
# BATCH 5 ANNOTATIONS — GOLD-0081 to GOLD-0100
# ============================================================

batch5_annotations = {

    "GOLD-0081": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports an unresolved issue after 21 days and indicates financial frustration. Support redirects to email for further assistance."
    },

    "GOLD-0082": {
        "intent": "PAYMENT_BILLING",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Payment was deducted but order is on hold. Cross-border payment/order issue requires account and payment investigation; customer is also warned not to expose order details publicly."
    },

    "GOLD-0083": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer query is too incomplete to determine a specific intent. Support says the relevant department will check and update the item page."
    },

    "GOLD-0084": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer complains about an unusual delivery method. Support acknowledges the concern and asks whether the package arrived safely."
    },

    "GOLD-0085": {
        "intent": "PAYMENT_BILLING",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports multiple unexpected charges. Financial/account billing issue requires secure investigation."
    },

    "GOLD-0086": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer asks whether email support is available. Support directly provides the contact route."
    },

    "GOLD-0087": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer is discussing favorite books rather than requesting customer support. Support responds conversationally."
    },

    "GOLD-0088": {
        "intent": "RETURNS_REFUNDS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Returned shipment has missing tracking information and refund status is uncertain. Support asks whether refund confirmation was received."
    },

    "GOLD-0089": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "RESOLVED_BY_CUSTOMER",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer explicitly says thanks after receiving assistance. Support closes the interaction."
    },

    "GOLD-0090": {
        "intent": "RETURNS_REFUNDS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer received the wrong item and needs support guidance. Support directs them to customer service but the issue is not yet confirmed resolved."
    },

    "GOLD-0091": {
        "intent": "PAYMENT_BILLING",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Displayed product price differs from the cart price. Pricing discrepancy requires investigation of the listing/cart state."
    },

    "GOLD-0092": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer is highly dissatisfied and requests management contact. Existing email support is still pending, so escalation/follow-up is required."
    },

    "GOLD-0093": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Conversation context is incomplete and support is asking whether there was a response from another party. Specific issue cannot be confidently classified."
    },

    "GOLD-0094": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports a delivery marked as handed to residents but apparently not received. Support provides troubleshooting steps, but delivery status still needs confirmation."
    },

    "GOLD-0095": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer explicitly confirms that customer service resolved the matter. Support acknowledges the successful resolution."
    },

    "GOLD-0096": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer expresses strong frustration with using Amazon but does not specify the underlying problem. Support asks whether there is a specific issue."
    },

    "GOLD-0097": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer has already tried delivery troubleshooting but still lacks a delivery notification. Support redirects to real-time assistance."
    },

    "GOLD-0098": {
        "intent": "ORDER_TRACKING",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Order is marked delivered despite no delivery occurring and customer has already paid. Potential missing-package/payment concern requires immediate investigation."
    },

    "GOLD-0099": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Expected delivery date has passed. Support requests further contact to inspect the order."
    },

    "GOLD-0100": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Order has not shipped by the time expected. Support asks for the estimated delivery date before proceeding."
    }
}

print("Batch 5 annotations prepared:", len(batch5_annotations))

Batch 5 annotations prepared: 20


In [ ]:
# ============================================================
# APPLY BATCH 5
# ============================================================

for gold_id, annotation in batch5_annotations.items():
    mask = golden_df["golden_id"] == gold_id

    for column, value in annotation.items():
        golden_df.loc[mask, column] = value

annotation_columns = [
    "intent",
    "resolution_status",
    "risk_level",
    "automation_decision"
]

completed_mask = (
    golden_df[annotation_columns].notna().all(axis=1)
    & (golden_df[annotation_columns] != "").all(axis=1)
)

print("=" * 70)
print("BATCH 5 ANNOTATIONS APPLIED")
print("=" * 70)
print(f"Batch 5 rows updated: {len(batch5_annotations)}")
print(f"Total completed annotations: {completed_mask.sum()}")
print(f"Remaining: {len(golden_df) - completed_mask.sum()}")

display(
    golden_df[
        golden_df["golden_id"].isin(batch5_annotations.keys())
    ][[
        "golden_id",
        "intent",
        "resolution_status",
        "risk_level",
        "automation_decision"
    ]].sort_values("golden_id")
)

BATCH 5 ANNOTATIONS APPLIED
Batch 5 rows updated: 20
Total completed annotations: 100
Remaining: 100


,golden_id,intent,resolution_status,risk_level,automation_decision
80,GOLD-0081,GENERAL_SUPPORT,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
81,GOLD-0082,PAYMENT_BILLING,INVESTIGATION_REQUIRED,HIGH,HUMAN
82,GOLD-0083,GENERAL_SUPPORT,INVESTIGATION_REQUIRED,LOW,HUMAN
83,GOLD-0084,ORDER_DELIVERY,PARTIALLY_RESOLVED,LOW,AUTO
84,GOLD-0085,PAYMENT_BILLING,INVESTIGATION_REQUIRED,HIGH,HUMAN
85,GOLD-0086,GENERAL_SUPPORT,RESOLVED,LOW,AUTO
86,GOLD-0087,GENERAL_SOCIAL,NOT_APPLICABLE,LOW,AUTO
87,GOLD-0088,RETURNS_REFUNDS,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
88,GOLD-0089,GENERAL_SUPPORT,RESOLVED_BY_CUSTOMER,LOW,AUTO
89,GOLD-0090,RETURNS_REFUNDS,INVESTIGATION_REQUIRED,MEDIUM,HUMAN


In [ ]:
# ============================================================
# BATCH 6 INSPECTION — GOLD-0101 to GOLD-0120
# ============================================================

batch6_ids = [f"GOLD-{i:04d}" for i in range(101, 121)]

batch6 = golden_df[
    golden_df["golden_id"].isin(batch6_ids)
].copy()

print("=" * 100)
print("BATCH 6 — GOLD-0101 → GOLD-0120")
print("=" * 100)
print(f"Cases found: {len(batch6)}")

for _, row in batch6.sort_values("golden_id").iterrows():

    print("\n" + "=" * 100)
    print(
        f"{row['golden_id']} | "
        f"Case: {row['case_id']} | "
        f"Turns: {row['turn_count']} | "
        f"Provisional: {row['provisional_bucket']}"
    )

    print("-" * 100)

    print("CUSTOMER QUERY:")
    print(row["customer_query"])

    print("\nSUPPORT RESPONSE:")
    print(row["support_response"])

    print("\nMETADATA:")
    print(f"Customer turns : {row['customer_turns']}")
    print(f"Support turns  : {row['support_turns']}")
    print(f"Duration       : {row['duration_hours']:.2f} hours")

BATCH 6 — GOLD-0101 → GOLD-0120
Cases found: 20

GOLD-0101 | Case: AMZ-008983 | Turns: 2 | Provisional: RESOLUTION_CANDIDATE
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
@121284 will it accept commands to work with @128308 ?

SUPPORT RESPONSE:
@215403 Request you to keep checking the Echo page for new devices that will support Alexa. ^GS

METADATA:
Customer turns : 1
Support turns  : 1
Duration       : 0.32 hours

GOLD-0102 | Case: AMZ-056378 | Turns: 2 | Provisional: RESOLUTION_CANDIDATE
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
Crap day. Amazon delivery never turned up, work extremely busy and then everything else. 😭☹️

SUPPORT RESPONSE:
@154241 Oh no! I'm so sorry to hear this. What is the current status of the tracking for your order? You can check here: https://t.co/Y5jpI9gRhE ^JY

METADATA:
Customer turns : 1
Support turns  : 1
Duratio

In [ ]:
# ============================================================
# BATCH 6 ANNOTATIONS — GOLD-0101 to GOLD-0120
# ============================================================

batch6_annotations = {

    "GOLD-0101": {
        "intent": "DEVICE_TECHNICAL",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer asks about device compatibility with Alexa. Support gives a direct informational answer by directing them to the Echo compatibility page."
    },

    "GOLD-0102": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports a missed delivery. Support asks for the current tracking status before determining the next action."
    },

    "GOLD-0103": {
        "intent": "ACCOUNT_ACCESS",
        "resolution_status": "ESCALATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports account lockout/deactivation while redeeming gift cards after repeated contact with support. Account access issue requires secure escalation."
    },

    "GOLD-0104": {
        "intent": "ACCOUNT_SECURITY",
        "resolution_status": "ESCALATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports a hacked login email and needs it restored. Account compromise/security issue is explicitly high risk and routed to Account Specialists."
    },

    "GOLD-0105": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Package has reached the customer's city but has not gone out for delivery. Support indicates courier coordination but delivery is not yet confirmed."
    },

    "GOLD-0106": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "UNRESOLVED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports repeated delivery delays and another promised date. Support asks the customer to contact them if the package still does not arrive."
    },

    "GOLD-0107": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer wants a refund for paid next-day delivery. Support cannot access the account through Twitter and directs the customer to secure support."
    },

    "GOLD-0108": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "ESCALATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer threatens consequences and attempts to publish a phone number for contact. Support explicitly prevents disclosure of personal information. High-risk interaction."
    },

    "GOLD-0109": {
        "intent": "ORDER_CANCELLATION",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer has repeatedly cancelled orders because of a recurring problem. Support asks about previous customer-service options; underlying issue remains unresolved."
    },

    "GOLD-0110": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer only says they need help with an order. There is insufficient information to determine the specific issue, so clarification is required."
    },

    "GOLD-0111": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer is sharing a positive holiday/product presentation experience. Support responds socially."
    },

    "GOLD-0112": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports implausible delivery locations. Support asks about the carrier and provides package-location troubleshooting."
    },

    "GOLD-0113": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer is expressing appreciation for purchased books. Support responds conversationally."
    },

    "GOLD-0114": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer is thanking Amazon support. Support gives a polite conversational response."
    },

    "GOLD-0115": {
        "intent": "RETURNS_REFUNDS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports a poor-quality product and requests a refund, while also exposing an order number publicly. Refund issue plus exposed personal/order information requires secure human handling."
    },

    "GOLD-0116": {
        "intent": "ORDER_TRACKING",
        "resolution_status": "PARTIALLY_RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Support explains how to identify the carrier from the package label. This directly addresses the customer's question, although further confirmation is requested."
    },

    "GOLD-0117": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer received only 3 of 4 ordered items and publicly attached order details. Missing-item issue requires investigation and privacy-safe handling."
    },

    "GOLD-0118": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer threatens to cancel their account because of poor service but does not specify the underlying issue. Support asks for clarification."
    },

    "GOLD-0119": {
        "intent": "DIGITAL_CONTENT",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer asks how to purchase a movie. Support directly explains that it has not yet been released and provides the pre-order option."
    },

    "GOLD-0120": {
        "intent": "MARKETPLACE_SELLING",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer complains about seller performance. Support says it has responded through another channel, but the underlying seller issue is not independently confirmed resolved."
    }
}

print("Batch 6 annotations prepared:", len(batch6_annotations))

Batch 6 annotations prepared: 20


In [ ]:
# ============================================================
# APPLY BATCH 6
# ============================================================

for gold_id, annotation in batch6_annotations.items():
    mask = golden_df["golden_id"] == gold_id

    for column, value in annotation.items():
        golden_df.loc[mask, column] = value

annotation_columns = [
    "intent",
    "resolution_status",
    "risk_level",
    "automation_decision"
]

completed_mask = (
    golden_df[annotation_columns].notna().all(axis=1)
    & (golden_df[annotation_columns] != "").all(axis=1)
)

print("=" * 70)
print("BATCH 6 ANNOTATIONS APPLIED")
print("=" * 70)
print(f"Batch 6 rows updated: {len(batch6_annotations)}")
print(f"Total completed annotations: {completed_mask.sum()}")
print(f"Remaining: {len(golden_df) - completed_mask.sum()}")

display(
    golden_df[
        golden_df["golden_id"].isin(batch6_annotations.keys())
    ][[
        "golden_id",
        "intent",
        "resolution_status",
        "risk_level",
        "automation_decision"
    ]].sort_values("golden_id")
)

BATCH 6 ANNOTATIONS APPLIED
Batch 6 rows updated: 20
Total completed annotations: 120
Remaining: 80


,golden_id,intent,resolution_status,risk_level,automation_decision
100,GOLD-0101,DEVICE_TECHNICAL,RESOLVED,LOW,AUTO
101,GOLD-0102,ORDER_DELIVERY,INVESTIGATION_REQUIRED,LOW,HUMAN
102,GOLD-0103,ACCOUNT_ACCESS,ESCALATION_REQUIRED,HIGH,HUMAN
103,GOLD-0104,ACCOUNT_SECURITY,ESCALATION_REQUIRED,HIGH,HUMAN
104,GOLD-0105,ORDER_DELIVERY,INVESTIGATION_REQUIRED,LOW,HUMAN
105,GOLD-0106,ORDER_DELIVERY,UNRESOLVED,LOW,HUMAN
106,GOLD-0107,ORDER_DELIVERY,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
107,GOLD-0108,GENERAL_SUPPORT,ESCALATION_REQUIRED,HIGH,HUMAN
108,GOLD-0109,ORDER_CANCELLATION,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
109,GOLD-0110,GENERAL_SUPPORT,INVESTIGATION_REQUIRED,LOW,HUMAN


In [ ]:
# ============================================================
# BATCH 7 INSPECTION — GOLD-0121 to GOLD-0140
# ============================================================

batch7_ids = [f"GOLD-{i:04d}" for i in range(121, 141)]

batch7 = golden_df[
    golden_df["golden_id"].isin(batch7_ids)
].copy()

print("=" * 100)
print("BATCH 7 — GOLD-0121 → GOLD-0140")
print("=" * 100)
print(f"Cases found: {len(batch7)}")

for _, row in batch7.sort_values("golden_id").iterrows():

    print("\n" + "=" * 100)

    print(
        f"{row['golden_id']} | "
        f"Case: {row['case_id']} | "
        f"Turns: {row['turn_count']} | "
        f"Provisional: {row['provisional_bucket']}"
    )

    print("-" * 100)

    print("CUSTOMER QUERY:")
    print(row["customer_query"])

    print("\nSUPPORT RESPONSE:")
    print(row["support_response"])

    print("\nMETADATA:")
    print(f"Customer turns : {row['customer_turns']}")
    print(f"Support turns  : {row['support_turns']}")
    print(f"Duration       : {row['duration_hours']:.2f} hours")

BATCH 7 — GOLD-0121 → GOLD-0140
Cases found: 20

GOLD-0121 | Case: AMZ-020675 | Turns: 6 | Provisional: RESOLUTION_CANDIDATE
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
@AmazonHelp It doesn't appear to be. It appears to be an email impersonating you wanting me to click on a link...

SUPPORT RESPONSE:
@571566 You can report this e-mail here: https://t.co/s1gLcKDiQy If you need further guidance, please don't hesitate to reach out! ^KN

METADATA:
Customer turns : 3
Support turns  : 3
Duration       : 1.39 hours

GOLD-0122 | Case: AMZ-011021 | Turns: 3 | Provisional: POTENTIAL_UNRESOLVED
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
@117804 Lindão!

SUPPORT RESPONSE:
@299949 Demais, Eduardo 😉 Vai comprar esse novo Kindle? ^LG

METADATA:
Customer turns : 2
Support turns  : 1
Duration       : 2.99 hours

GOLD-0123 | Case: AMZ-068346 | Turns: 4 | Prov

In [ ]:
# ============================================================
# BATCH 7 ANNOTATIONS — GOLD-0121 to GOLD-0140
# ============================================================

batch7_annotations = {

    "GOLD-0121": {
        "intent": "ACCOUNT_SECURITY",
        "resolution_status": "RESOLVED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer identifies a possible phishing email impersonating Amazon. Support provides the official reporting route, but security-related cases should remain human-reviewed."
    },

    "GOLD-0122": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer is making a positive/social comment about a Kindle rather than requesting support."
    },

    "GOLD-0123": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "RESOLVED_BY_CUSTOMER",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer confirms the item arrived and thanks Amazon. Support acknowledges the successful delivery."
    },

    "GOLD-0124": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer is expressing appreciation for a purchase/deal. No support issue is present."
    },

    "GOLD-0125": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer raises concern about recipient identification during delivery. The proposed use of identity information makes this privacy/security-sensitive and requires human handling."
    },

    "GOLD-0126": {
        "intent": "RETURNS_REFUNDS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer asks how to follow a warranty procedure. Support requests details through a secure channel, so the specific warranty path still needs assistance."
    },

    "GOLD-0127": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer expresses appreciation about finding a product at a good price. No support request is present."
    },

    "GOLD-0128": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Long-running support interaction with repeated email contact but no clear resolution. Requires continued investigation/escalation."
    },

    "GOLD-0129": {
        "intent": "DIGITAL_CONTENT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer asks about accessing a digital service through a subscription. Support redirects to direct assistance rather than providing a definitive answer."
    },

    "GOLD-0130": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer indicates they will contact another support route. Support provides contact information; the underlying issue is not sufficiently visible to classify more specifically."
    },

    "GOLD-0131": {
        "intent": "GENERAL_SOCIAL",
        "resolution_status": "NOT_APPLICABLE",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer gives positive feedback about Amazon's service for parents. Support responds socially."
    },

    "GOLD-0132": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer message is too incomplete to infer the underlying issue. Support routes the matter to the Social Media team for investigation."
    },

    "GOLD-0133": {
        "intent": "ORDER_CANCELLATION",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports an unexpected event after cancelling an order. Long multi-turn interaction remains unresolved and requires order-level investigation."
    },

    "GOLD-0134": {
        "intent": "ACCOUNT_SECURITY",
        "resolution_status": "RESOLVED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports a phishing/scam message impersonating Amazon. Support gives concrete anti-phishing guidance. Security-sensitive interaction remains human-handled."
    },

    "GOLD-0135": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Release-day delivery failed and carrier tracking has lost the package. Support needs to investigate the shipment."
    },

    "GOLD-0136": {
        "intent": "RETURNS_REFUNDS",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer believes a book was improperly packaged and may need support options. Support offers self-service and real-time assistance rather than confirming a resolution."
    },

    "GOLD-0137": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "MEDIUM",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports an inappropriate delivery location despite someone being home. Support needs to investigate the delivery."
    },

    "GOLD-0138": {
        "intent": "GENERAL_SUPPORT",
        "resolution_status": "RESOLVED",
        "risk_level": "LOW",
        "automation_decision": "AUTO",
        "annotation_notes": "Customer says they found an easy way to purchase the desired product and expresses satisfaction. Support responds positively."
    },

    "GOLD-0139": {
        "intent": "PAYMENT_BILLING",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "HIGH",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports card charged but payment not reflected as completed. Financial/payment discrepancy requires secure investigation."
    },

    "GOLD-0140": {
        "intent": "ORDER_DELIVERY",
        "resolution_status": "INVESTIGATION_REQUIRED",
        "risk_level": "LOW",
        "automation_decision": "HUMAN",
        "annotation_notes": "Customer reports a delivery delay with no reason given. Support asks them to follow up if the package misses the specified deadline."
    }
}

print("Batch 7 annotations prepared:", len(batch7_annotations))

Batch 7 annotations prepared: 20


In [ ]:
# ============================================================
# APPLY BATCH 7
# ============================================================

for gold_id, annotation in batch7_annotations.items():
    mask = golden_df["golden_id"] == gold_id

    for column, value in annotation.items():
        golden_df.loc[mask, column] = value

annotation_columns = [
    "intent",
    "resolution_status",
    "risk_level",
    "automation_decision"
]

completed_mask = (
    golden_df[annotation_columns].notna().all(axis=1)
    & (golden_df[annotation_columns] != "").all(axis=1)
)

print("=" * 70)
print("BATCH 7 ANNOTATIONS APPLIED")
print("=" * 70)
print(f"Batch 7 rows updated: {len(batch7_annotations)}")
print(f"Total completed annotations: {completed_mask.sum()}")
print(f"Remaining: {len(golden_df) - completed_mask.sum()}")

display(
    golden_df[
        golden_df["golden_id"].isin(batch7_annotations.keys())
    ][[
        "golden_id",
        "intent",
        "resolution_status",
        "risk_level",
        "automation_decision"
    ]].sort_values("golden_id")
)

BATCH 7 ANNOTATIONS APPLIED
Batch 7 rows updated: 20
Total completed annotations: 140
Remaining: 60


,golden_id,intent,resolution_status,risk_level,automation_decision
120,GOLD-0121,ACCOUNT_SECURITY,RESOLVED,HIGH,HUMAN
121,GOLD-0122,GENERAL_SOCIAL,NOT_APPLICABLE,LOW,AUTO
122,GOLD-0123,GENERAL_SOCIAL,RESOLVED_BY_CUSTOMER,LOW,AUTO
123,GOLD-0124,GENERAL_SOCIAL,NOT_APPLICABLE,LOW,AUTO
124,GOLD-0125,ORDER_DELIVERY,INVESTIGATION_REQUIRED,HIGH,HUMAN
125,GOLD-0126,RETURNS_REFUNDS,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
126,GOLD-0127,GENERAL_SOCIAL,NOT_APPLICABLE,LOW,AUTO
127,GOLD-0128,GENERAL_SUPPORT,INVESTIGATION_REQUIRED,MEDIUM,HUMAN
128,GOLD-0129,DIGITAL_CONTENT,INVESTIGATION_REQUIRED,LOW,HUMAN
129,GOLD-0130,GENERAL_SUPPORT,INVESTIGATION_REQUIRED,LOW,HUMAN


In [ ]:
# ============================================================
# BATCH 8 INSPECTION — GOLD-0141 to GOLD-0160
# ============================================================

batch8_ids = [f"GOLD-{i:04d}" for i in range(141, 161)]

batch8 = golden_df[
    golden_df["golden_id"].isin(batch8_ids)
].copy()

print("=" * 100)
print("BATCH 8 — GOLD-0141 → GOLD-0160")
print("=" * 100)
print(f"Cases found: {len(batch8)}")

for _, row in batch8.sort_values("golden_id").iterrows():

    print("\n" + "=" * 100)

    print(
        f"{row['golden_id']} | "
        f"Case: {row['case_id']} | "
        f"Turns: {row['turn_count']} | "
        f"Provisional: {row['provisional_bucket']}"
    )

    print("-" * 100)

    print("CUSTOMER QUERY:")
    print(row["customer_query"])

    print("\nSUPPORT RESPONSE:")
    print(row["support_response"])

    print("\nMETADATA:")
    print(f"Customer turns : {row['customer_turns']}")
    print(f"Support turns  : {row['support_turns']}")
    print(f"Duration       : {row['duration_hours']:.2f} hours")

BATCH 8 — GOLD-0141 → GOLD-0160
Cases found: 20

GOLD-0141 | Case: AMZ-067665 | Turns: 2 | Provisional: WEAK_OR_OTHER
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
So I order a tv from @115830 and is to be delivered by @24123 on Saturday the 25th now they say they will be delivering it on Monday the 29th I get one day off work which is Saturday what a joke 👎👎

SUPPORT RESPONSE:
@236297 I'm sorry to hear of this. Have you reached out to us directly so that we can reschedule delivery for you? ^KI

METADATA:
Customer turns : 1
Support turns  : 1
Duration       : 0.04 hours

GOLD-0142 | Case: AMZ-027728 | Turns: 2 | Provisional: WEAK_OR_OTHER
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
@115821  US "ships" but @115850 India " dispatches" ... Hmm.. Localization ?? :-) https://t.co/5RI2v2jBlL

SUPPORT RESPONSE:
@390143 Doesn't dispatch sound nice!? 🙂 

In [ ]:
batch8_labels = {
    "GOLD-0141": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0142": ("GENERAL_SOCIAL", "NOT_APPLICABLE", "LOW", "AUTO"),
    "GOLD-0143": ("PREORDER_DELIVERY", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0144": ("GENERAL_SUPPORT", "RESOLVED", "LOW", "AUTO"),
    "GOLD-0145": ("RETURNS_REFUNDS", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0146": ("DIGITAL_CONTENT", "RESOLVED", "LOW", "AUTO"),
    "GOLD-0147": ("DIGITAL_CONTENT", "PARTIALLY_RESOLVED", "LOW", "AUTO"),
    "GOLD-0148": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0149": ("RETURNS_REFUNDS", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0150": ("ACCOUNT_ACCESS", "ESCALATION_REQUIRED", "HIGH", "HUMAN"),
    "GOLD-0151": ("GENERAL_SUPPORT", "RESOLVED_BY_CUSTOMER", "LOW", "AUTO"),
    "GOLD-0152": ("ACCOUNT_ACCESS", "ESCALATION_REQUIRED", "HIGH", "HUMAN"),
    "GOLD-0153": ("DIGITAL_CONTENT", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0154": ("DEVICE_TECHNICAL", "RESOLVED", "LOW", "AUTO"),
    "GOLD-0155": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0156": ("ORDER_TRACKING", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0157": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0158": ("ORDER_DELIVERY", "RESOLVED", "LOW", "AUTO"),
    "GOLD-0159": ("GENERAL_SUPPORT", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0160": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
}

for gold_id, (intent, resolution, risk, decision) in batch8_labels.items():
    mask = golden_df["golden_id"] == gold_id

    golden_df.loc[mask, "intent"] = intent
    golden_df.loc[mask, "resolution_status"] = resolution
    golden_df.loc[mask, "risk_level"] = risk
    golden_df.loc[mask, "automation_decision"] = decision

print("Batch 8 rows updated:", len(batch8_labels))
print("Total completed annotations:",
      golden_df["intent"].notna().sum())
print("Remaining:",
      len(golden_df) - golden_df["intent"].notna().sum())

Batch 8 rows updated: 20
Total completed annotations: 160
Remaining: 40


In [ ]:
# ============================================================
# BATCH 9 — GOLD-0161 → GOLD-0180
# ============================================================

batch9_ids = [f"GOLD-{i:04d}" for i in range(161, 181)]

batch9 = golden_df[
    golden_df["golden_id"].isin(batch9_ids)
].copy()

print("=" * 100)
print("BATCH 9 — GOLD-0161 → GOLD-0180")
print("=" * 100)
print("Cases found:", len(batch9))

for _, row in batch9.iterrows():
    print("\n" + "=" * 100)
    print(f"{row['golden_id']} | Case: {row['case_id']} | "
          f"Turns: {row['turn_count']} | "
          f"Provisional: {row['provisional_bucket']}")
    print("-" * 100)
    print("CUSTOMER QUERY:")
    print(row["customer_query"])
    print("\nSUPPORT RESPONSE:")
    print(row["support_response"])
    print("\nMETADATA:")
    print(f"Customer turns : {row['customer_turns']}")
    print(f"Support turns  : {row['support_turns']}")
    print(f"Duration       : {row['duration_hours']:.2f} hours")

BATCH 9 — GOLD-0161 → GOLD-0180
Cases found: 20

GOLD-0161 | Case: AMZ-053000 | Turns: 2 | Provisional: WEAK_OR_OTHER
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
Bonjour @120533 @AmazonHelp 2eme fois que je signale, on me dit oui oui et rien : regardez ces 4 références : B006SYMICU B006SYMJ3S B00WFXJ2QS B006SYMIRA toutes ces refs sont inversées du coup on reçoit pas ce qu'on commande, voir photo pour les refs et caractéristiques RÉELLES https://t.co/K2DD2R8mlb

SUPPORT RESPONSE:
@689670 Bonjour, je suis vraiment désolée pour ce désagrément, est ce que vous avez passé commande sur Amazon.fr ou https://t.co/nUUp5MLhYl ? ^ASZ

METADATA:
Customer turns : 1
Support turns  : 1
Duration       : 0.12 hours

GOLD-0162 | Case: AMZ-072496 | Turns: 3 | Provisional: POTENTIAL_UNRESOLVED
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
Stayed home and did my #B

In [ ]:
# ============================================================
# BATCH 9 — GOLD-0161 → GOLD-0180
# ============================================================

batch9_labels = {
    "GOLD-0161": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0162": ("ORDER_TRACKING", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0163": ("ORDER_DELIVERY", "PARTIALLY_RESOLVED", "LOW", "HUMAN"),
    "GOLD-0164": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0165": ("ORDER_TRACKING", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0166": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0167": ("GENERAL_SUPPORT", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0168": ("GENERAL_SUPPORT", "PARTIALLY_RESOLVED", "LOW", "AUTO"),
    "GOLD-0169": ("GENERAL_SOCIAL", "NOT_APPLICABLE", "LOW", "AUTO"),
    "GOLD-0170": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0171": ("PAYMENT_BILLING", "INVESTIGATION_REQUIRED", "HIGH", "HUMAN"),
    "GOLD-0172": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "HIGH", "HUMAN"),
    "GOLD-0173": ("GENERAL_SUPPORT", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0174": ("ACCOUNT_ACCESS", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0175": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0176": ("RETURNS_REFUNDS", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0177": ("PAYMENT_BILLING", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0178": ("DIGITAL_CONTENT", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0179": ("RETURNS_REFUNDS", "PARTIALLY_RESOLVED", "LOW", "AUTO"),
    "GOLD-0180": ("ACCOUNT_ACCESS", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
}

for gold_id, (intent, resolution, risk, decision) in batch9_labels.items():
    mask = golden_df["golden_id"] == gold_id

    golden_df.loc[mask, "intent"] = intent
    golden_df.loc[mask, "resolution_status"] = resolution
    golden_df.loc[mask, "risk_level"] = risk
    golden_df.loc[mask, "automation_decision"] = decision

print("Batch 9 rows updated:", len(batch9_labels))
print(
    "Total completed annotations:",
    golden_df["intent"].notna().sum()
)
print(
    "Remaining:",
    len(golden_df) - golden_df["intent"].notna().sum()
)

Batch 9 rows updated: 20
Total completed annotations: 180
Remaining: 20


In [ ]:
# ============================================================
# BATCH 10 — GOLD-0181 → GOLD-0200
# ============================================================

batch10_ids = [f"GOLD-{i:04d}" for i in range(181, 201)]

batch10 = golden_df[
    golden_df["golden_id"].isin(batch10_ids)
].copy()

print("=" * 100)
print("BATCH 10 — GOLD-0181 → GOLD-0200")
print("=" * 100)
print("Cases found:", len(batch10))

for _, row in batch10.iterrows():
    print("\n" + "=" * 100)
    print(
        f"{row['golden_id']} | Case: {row['case_id']} | "
        f"Turns: {row['turn_count']} | "
        f"Provisional: {row['provisional_bucket']}"
    )
    print("-" * 100)

    print("CUSTOMER QUERY:")
    print(row["customer_query"])

    print("\nSUPPORT RESPONSE:")
    print(row["support_response"])

    print("\nMETADATA:")
    print(f"Customer turns : {row['customer_turns']}")
    print(f"Support turns  : {row['support_turns']}")
    print(f"Duration       : {row['duration_hours']:.2f} hours")

BATCH 10 — GOLD-0181 → GOLD-0200
Cases found: 20

GOLD-0181 | Case: AMZ-006520 | Turns: 6 | Provisional: RESOLUTION_CANDIDATE
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
@AmazonHelp Doesn't seem to.

SUPPORT RESPONSE:
@199295 That's unusual, please fill this form: https://t.co/GIJyeYqKE0 and we'll connect with you on this. ^MP

METADATA:
Customer turns : 3
Support turns  : 3
Duration       : 49.86 hours

GOLD-0182 | Case: AMZ-040694 | Turns: 2 | Provisional: FOLLOWUP_OR_INVESTIGATION
----------------------------------------------------------------------------------------------------
CUSTOMER QUERY:
@AmazonHelp It says my parcel was delivered today, and I got a notification on my phone, but it's not here, and according to the tracking(1)

SUPPORT RESPONSE:
@472833 That's interesting! We would like to look into this issue with you in real-time. Contact us here: https://t.co/JzP7hlA23B ^LS

METADATA:
Customer turns :

In [ ]:
# ============================================================
# BATCH 10 — GOLD-0181 → GOLD-0200
# ============================================================

batch10_labels = {
    "GOLD-0181": ("GENERAL_SUPPORT", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0182": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "HIGH", "HUMAN"),
    "GOLD-0183": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0184": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0185": ("GENERAL_SOCIAL", "NOT_APPLICABLE", "LOW", "AUTO"),
    "GOLD-0186": ("GENERAL_SUPPORT", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0187": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0188": ("PREORDER_DELIVERY", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0189": ("RETURNS_REFUNDS", "INVESTIGATION_REQUIRED", "MEDIUM", "HUMAN"),
    "GOLD-0190": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "HIGH", "HUMAN"),
    "GOLD-0191": ("PAYMENT_BILLING", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
    "GOLD-0192": ("PAYMENT_BILLING", "PARTIALLY_RESOLVED", "MEDIUM", "HUMAN"),
    "GOLD-0193": ("DIGITAL_CONTENT", "UNRESOLVED", "LOW", "HUMAN"),
    "GOLD-0194": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "HIGH", "HUMAN"),
    "GOLD-0195": ("ACCOUNT_ACCESS", "INVESTIGATION_REQUIRED", "HIGH", "HUMAN"),
    "GOLD-0196": ("ORDER_DELIVERY", "UNRESOLVED", "MEDIUM", "HUMAN"),
    "GOLD-0197": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "HIGH", "HUMAN"),
    "GOLD-0198": ("ACCOUNT_ACCESS", "PARTIALLY_RESOLVED", "MEDIUM", "AUTO"),
    "GOLD-0199": ("DIGITAL_CONTENT", "PARTIALLY_RESOLVED", "LOW", "AUTO"),
    "GOLD-0200": ("ORDER_DELIVERY", "INVESTIGATION_REQUIRED", "LOW", "HUMAN"),
}

for gold_id, (intent, resolution, risk, decision) in batch10_labels.items():
    mask = golden_df["golden_id"] == gold_id

    golden_df.loc[mask, "intent"] = intent
    golden_df.loc[mask, "resolution_status"] = resolution
    golden_df.loc[mask, "risk_level"] = risk
    golden_df.loc[mask, "automation_decision"] = decision

print("Batch 10 rows updated:", len(batch10_labels))
print(
    "Total completed annotations:",
    golden_df["intent"].notna().sum()
)
print(
    "Remaining:",
    len(golden_df) - golden_df["intent"].notna().sum()
)

Batch 10 rows updated: 20
Total completed annotations: 200
Remaining: 0


In [ ]:
import os

# Create the required directory
os.makedirs("data/evaluation", exist_ok=True)

# Save the completed golden set
golden_path = "data/evaluation/golden_evaluation_set.csv"

golden_df.to_csv(golden_path, index=False)

print("Golden evaluation set saved:", golden_path)
print("Total cases:", len(golden_df))
print("Completed annotations:", golden_df["intent"].notna().sum())

print(
    "Missing labels:",
    golden_df[
        [
            "intent",
            "resolution_status",
            "risk_level",
            "automation_decision"
        ]
    ].isna().sum().sum()
)

Golden evaluation set saved: data/evaluation/golden_evaluation_set.csv
Total cases: 200
Completed annotations: 200
Missing labels: 0


In [ ]:
import shutil

backup_path = "data/evaluation/golden_evaluation_set_frozen.csv"

shutil.copy2(
    "data/evaluation/golden_evaluation_set.csv",
    backup_path
)

print("Frozen backup created:", backup_path)

Frozen backup created: data/evaluation/golden_evaluation_set_frozen.csv


In [ ]:
# ============================================================
# GOLDEN SET — BASIC ANALYSIS
# ============================================================

import pandas as pd

# Load the saved evaluation set
golden_eval = pd.read_csv(
    "data/evaluation/golden_evaluation_set.csv"
)

print("=" * 70)
print("GOLDEN SET ANALYSIS")
print("=" * 70)

print("\nDataset shape:")
print(golden_eval.shape)

print("\nColumns:")
print(golden_eval.columns.tolist())

print("\nMissing values:")
print(
    golden_eval[
        [
            "intent",
            "resolution_status",
            "risk_level",
            "automation_decision"
        ]
    ].isna().sum()
)

GOLDEN SET ANALYSIS

Dataset shape:
(200, 19)

Columns:
['golden_id', 'case_id', 'root_tweet_id', 'turn_count', 'customer_turns', 'support_turns', 'duration_hours', 'max_gap_hours', 'customer_query', 'support_response', 'provisional_bucket', 'gold_intent', 'resolution_quality', 'risk_level', 'should_auto_handle', 'annotation_notes', 'intent', 'resolution_status', 'automation_decision']

Missing values:
intent                 0
resolution_status      0
risk_level             0
automation_decision    0
dtype: int64


In [ ]:
print("\n" + "=" * 70)
print("INTENT DISTRIBUTION")
print("=" * 70)

intent_counts = (
    golden_eval["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="count")
)

intent_counts["percentage"] = (
    intent_counts["count"] / len(golden_eval) * 100
).round(2)

print(intent_counts.to_string(index=False))


INTENT DISTRIBUTION
             intent  count  percentage
     ORDER_DELIVERY     69        34.5
    GENERAL_SUPPORT     35        17.5
     GENERAL_SOCIAL     17         8.5
    RETURNS_REFUNDS     16         8.0
    PAYMENT_BILLING     11         5.5
    DIGITAL_CONTENT      9         4.5
     ORDER_TRACKING      9         4.5
     ACCOUNT_ACCESS      9         4.5
   DEVICE_TECHNICAL      5         2.5
   ACCOUNT_SECURITY      4         2.0
 ORDER_CANCELLATION      4         2.0
  PREORDER_DELIVERY      4         2.0
   PURCHASE_CONTROL      3         1.5
MARKETPLACE_SELLING      3         1.5
     ORDER_PURCHASE      1         0.5
SELLER_AUTHENTICITY      1         0.5


In [ ]:
print("\n" + "=" * 70)
print("RESOLUTION DISTRIBUTION")
print("=" * 70)

resolution_counts = (
    golden_eval["resolution_status"]
    .value_counts()
    .rename_axis("resolution_status")
    .reset_index(name="count")
)

resolution_counts["percentage"] = (
    resolution_counts["count"] / len(golden_eval) * 100
).round(2)

print(resolution_counts.to_string(index=False))


RESOLUTION DISTRIBUTION
     resolution_status  count  percentage
INVESTIGATION_REQUIRED    121        60.5
              RESOLVED     21        10.5
    PARTIALLY_RESOLVED     20        10.0
        NOT_APPLICABLE     17         8.5
            UNRESOLVED     10         5.0
   ESCALATION_REQUIRED      6         3.0
  RESOLVED_BY_CUSTOMER      5         2.5


In [ ]:
print("\n" + "=" * 70)
print("RISK DISTRIBUTION")
print("=" * 70)

risk_counts = (
    golden_eval["risk_level"]
    .value_counts()
    .rename_axis("risk_level")
    .reset_index(name="count")
)

risk_counts["percentage"] = (
    risk_counts["count"] / len(golden_eval) * 100
).round(2)

print(risk_counts.to_string(index=False))


print("\n" + "=" * 70)
print("AUTOMATION DISTRIBUTION")
print("=" * 70)

automation_counts = (
    golden_eval["automation_decision"]
    .value_counts()
    .rename_axis("automation_decision")
    .reset_index(name="count")
)

automation_counts["percentage"] = (
    automation_counts["count"] / len(golden_eval) * 100
).round(2)

print(automation_counts.to_string(index=False))


RISK DISTRIBUTION
risk_level  count  percentage
       LOW    106        53.0
    MEDIUM     68        34.0
      HIGH     26        13.0

AUTOMATION DISTRIBUTION
automation_decision  count  percentage
              HUMAN    148        74.0
               AUTO     52        26.0


In [ ]:
print("\n" + "=" * 70)
print("RISK × AUTOMATION DECISION")
print("=" * 70)

risk_automation = pd.crosstab(
    golden_eval["risk_level"],
    golden_eval["automation_decision"],
    margins=True
)

print(risk_automation)


RISK × AUTOMATION DECISION
automation_decision  AUTO  HUMAN  All
risk_level                           
HIGH                    0     26   26
LOW                    51     55  106
MEDIUM                  1     67   68
All                    52    148  200


In [ ]:
# ============================================================
# CHECK DUPLICATE / LEGACY ANNOTATION COLUMNS
# ============================================================

pairs = [
    ("gold_intent", "intent"),
    ("resolution_quality", "resolution_status"),
    ("should_auto_handle", "automation_decision"),
]

for col1, col2 in pairs:
    print("\n" + "=" * 70)
    print(f"{col1}  vs  {col2}")
    print("=" * 70)

    print("Unique values —", col1)
    print(golden_eval[col1].value_counts(dropna=False))

    print("\nUnique values —", col2)
    print(golden_eval[col2].value_counts(dropna=False))

    print("\nNon-null counts:")
    print(col1, golden_eval[col1].notna().sum())
    print(col2, golden_eval[col2].notna().sum())


gold_intent  vs  intent
Unique values — gold_intent
gold_intent
NaN    200
Name: count, dtype: int64

Unique values — intent
intent
ORDER_DELIVERY         69
GENERAL_SUPPORT        35
GENERAL_SOCIAL         17
RETURNS_REFUNDS        16
PAYMENT_BILLING        11
DIGITAL_CONTENT         9
ORDER_TRACKING          9
ACCOUNT_ACCESS          9
DEVICE_TECHNICAL        5
ACCOUNT_SECURITY        4
ORDER_CANCELLATION      4
PREORDER_DELIVERY       4
PURCHASE_CONTROL        3
MARKETPLACE_SELLING     3
ORDER_PURCHASE          1
SELLER_AUTHENTICITY     1
Name: count, dtype: int64

Non-null counts:
gold_intent 0
intent 200

resolution_quality  vs  resolution_status
Unique values — resolution_quality
resolution_quality
NaN    200
Name: count, dtype: int64

Unique values — resolution_status
resolution_status
INVESTIGATION_REQUIRED    121
RESOLVED                   21
PARTIALLY_RESOLVED         20
NOT_APPLICABLE             17
UNRESOLVED                 10
ESCALATION_REQUIRED         6
RESOLVED_BY_CUS

In [ ]:
print("\n" + "=" * 70)
print("COLUMN CONSISTENCY")
print("=" * 70)

print(
    "gold_intent == intent:",
    (golden_eval["gold_intent"] == golden_eval["intent"]).mean()
)

print(
    "resolution_quality == resolution_status:",
    (
        golden_eval["resolution_quality"]
        == golden_eval["resolution_status"]
    ).mean()
)

print(
    "should_auto_handle == automation_decision:",
    (
        golden_eval["should_auto_handle"].astype(str).str.upper()
        == golden_eval["automation_decision"].astype(str).str.upper()
    ).mean()
)


COLUMN CONSISTENCY
gold_intent == intent: 0.0
resolution_quality == resolution_status: 0.0
should_auto_handle == automation_decision: 0.0


In [ ]:
# ============================================================
# CLEAN GOLDEN EVALUATION SET SCHEMA
# ============================================================

legacy_columns = [
    "gold_intent",
    "resolution_quality",
    "should_auto_handle"
]

golden_clean = golden_eval.drop(
    columns=legacy_columns
).copy()

print("Removed legacy columns:", legacy_columns)
print("\nFinal shape:", golden_clean.shape)

print("\nFinal columns:")
print(golden_clean.columns.tolist())

Removed legacy columns: ['gold_intent', 'resolution_quality', 'should_auto_handle']

Final shape: (200, 16)

Final columns:
['golden_id', 'case_id', 'root_tweet_id', 'turn_count', 'customer_turns', 'support_turns', 'duration_hours', 'max_gap_hours', 'customer_query', 'support_response', 'provisional_bucket', 'risk_level', 'annotation_notes', 'intent', 'resolution_status', 'automation_decision']


In [ ]:
# ============================================================
# FINAL GOLDEN SET VALIDATION
# ============================================================

required_columns = [
    "golden_id",
    "case_id",
    "root_tweet_id",
    "turn_count",
    "customer_turns",
    "support_turns",
    "duration_hours",
    "max_gap_hours",
    "customer_query",
    "support_response",
    "provisional_bucket",
    "intent",
    "resolution_status",
    "risk_level",
    "automation_decision"
]

missing_columns = [
    col for col in required_columns
    if col not in golden_clean.columns
]

print("Required columns missing:", missing_columns)

print(
    "Total rows:",
    len(golden_clean)
)

print(
    "Rows with complete labels:",
    golden_clean[
        [
            "intent",
            "resolution_status",
            "risk_level",
            "automation_decision"
        ]
    ].notna().all(axis=1).sum()
)

Required columns missing: []
Total rows: 200
Rows with complete labels: 200


In [ ]:
# ============================================================
# SAVE CLEAN FROZEN GOLDEN SET
# ============================================================

clean_path = "data/evaluation/golden_evaluation_set.csv"

golden_clean.to_csv(
    clean_path,
    index=False
)

# Separate immutable-style backup
frozen_path = "data/evaluation/golden_evaluation_set_frozen.csv"

golden_clean.to_csv(
    frozen_path,
    index=False
)

print("Clean golden set:", clean_path)
print("Frozen copy:", frozen_path)

Clean golden set: data/evaluation/golden_evaluation_set.csv
Frozen copy: data/evaluation/golden_evaluation_set_frozen.csv


In [ ]:
print("Available DataFrames:")

for name, obj in list(globals().items()):
    if hasattr(obj, "shape") and hasattr(obj, "columns"):
        print(f"{name:30s} {obj.shape}")

Available DataFrames:
sample_df                      (10, 7)
chunk                          (11774, 7)
amazon_chunk                   (656, 7)
amazon_df                      (169840, 7)
customer_tweets                (0, 7)
support_tweets                 (169840, 7)
matched                        (1376, 7)
amazon_connected_df            (358973, 8)
tweet_lookup                   (358973, 6)
case_profile                   (83572, 11)
single_tweets                  (112, 8)
candidate_cases                (83418, 11)
direct_single_tweets           (112, 8)
conversation_cases             (83572, 13)
long_cases                     (20, 10)
gap_stats                      (83572, 4)
worst_gap_cases                (10, 13)
two_sided                      (83418, 13)
resolution_df                  (83418, 12)
sample_resolution              (30, 12)
short_responses                (30, 12)
df                             (83418, 20)
pattern_counts                 (8, 2)
bucket_summary              

In [ ]:
# ============================================================
# FULL RESOLUTION CORPUS — SCHEMA INSPECTION
# ============================================================

print("=" * 80)
print("RESOLUTION CORPUS")
print("=" * 80)

print("Shape:", resolution_df.shape)

print("\nColumns:")
for i, col in enumerate(resolution_df.columns, 1):
    print(f"{i:2d}. {col}")

print("\nData types:")
print(resolution_df.dtypes)

print("\nMissing values:")
print(
    resolution_df.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nFirst 3 records:")
display(resolution_df.head(3))

RESOLUTION CORPUS
Shape: (83418, 12)

Columns:
 1. case_id
 2. root_tweet_id
 3. turn_count
 4. customer_turns
 5. support_turns
 6. duration_hours
 7. max_gap_hours
 8. customer_query
 9. support_response
10. final_role
11. support_response_length
12. customer_query_length

Data types:
case_id                     object
root_tweet_id                int64
turn_count                   int64
customer_turns               int64
support_turns                int64
duration_hours             float64
max_gap_hours              float64
customer_query              object
support_response            object
final_role                  object
support_response_length      int64
customer_query_length        int64
dtype: object

Missing values:
case_id                    0
root_tweet_id              0
turn_count                 0
customer_turns             0
support_turns              0
duration_hours             0
max_gap_hours              0
customer_query             0
support_response           0


,case_id,root_tweet_id,turn_count,customer_turns,support_turns,duration_hours,max_gap_hours,customer_query,support_response,final_role,support_response_length,customer_query_length
0,AMZ-000001,648407,3,2,1,0.718611,0.702778,@115821 what in the world is a balance withheld? #customerservice doesnt seem to know!,"@274086 Sorry, I'm not quite sure what you're having trouble with. Was there an authorization you were wondering about? ^JY",customer,123,88
1,AMZ-000002,369267,7,4,3,21.288889,9.213889,@AmazonHelp That's exactly what she did. Bought it from Amazon / posted the review. But not thru your email to her. She posted via website.,"@203483 Sorry, we would be unable to change this as it is an automatic feature done through email.^CD",customer,101,139
2,AMZ-000003,2933545,2,1,1,20848.641111,20848.641111,@811202 @AmazonHelp i think u should cancel your order arpit and book another on Flipkart. It will deliver ur order in 2-3 working day,"@132991 We would like to help you, Prakhar. Please share your details with us here: https://t.co/GIJyeYqKE0. We'll get in touch shortly.\nPlease don't provide your order details as we consider it personal information. Our twitter page is visible to public. -Bikram",support,263,134


In [ ]:
# ============================================================
# TEXT COLUMN INSPECTION
# ============================================================

text_columns = []

for col in resolution_df.columns:
    if resolution_df[col].dtype == "object":
        text_columns.append(col)

print("Text/object columns:")
print(text_columns)

for col in text_columns:
    print(
        f"\n{col}:",
        resolution_df[col].dropna().astype(str).str.len().describe()
    )

Text/object columns:
['case_id', 'customer_query', 'support_response', 'final_role']

case_id: count    83418.0
mean        10.0
std          0.0
min         10.0
25%         10.0
50%         10.0
75%         10.0
max         10.0
Name: case_id, dtype: float64

customer_query: count    83418.000000
mean       112.177576
std         59.832098
min          0.000000
25%         66.000000
50%        111.000000
75%        142.000000
max        365.000000
Name: customer_query, dtype: float64

support_response: count    83418.000000
mean       123.906231
std         48.931077
min          7.000000
25%         96.000000
50%        124.000000
75%        136.000000
max        305.000000
Name: support_response, dtype: float64

final_role: count    83418.000000
mean         7.208684
std          0.406370
min          7.000000
25%          7.000000
50%          7.000000
75%          7.000000
max          8.000000
Name: final_role, dtype: float64


In [ ]:
# ============================================================
# INTENT DISCOVERY — PREPARE CUSTOMER QUERIES
# ============================================================

import re
import pandas as pd
import numpy as np

intent_corpus = resolution_df[
    [
        "case_id",
        "customer_query",
        "support_response"
    ]
].copy()

def clean_for_intent(text):
    text = str(text)

    # URLs
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)

    # Email addresses
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " EMAIL ",
        text
    )

    # Twitter-style mentions
    text = re.sub(r"@\w+", " USER ", text)

    # Order IDs / long numeric identifiers
    text = re.sub(r"\b\d{8,}\b", " ID ", text)

    # Phone-like numbers
    text = re.sub(r"\+?\d[\d\s\-]{8,}\d", " PHONE ", text)

    # HTML entities
    text = re.sub(r"&\w+;", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

intent_corpus["clean_query"] = (
    intent_corpus["customer_query"]
    .map(clean_for_intent)
)

print("Cases:", len(intent_corpus))
print("Missing cleaned queries:", intent_corpus["clean_query"].isna().sum())
print("\nExample lengths:")
print(intent_corpus["clean_query"].str.len().describe())

Cases: 83418
Missing cleaned queries: 0

Example lengths:
count    83418.000000
mean       102.097269
std         58.868375
min          0.000000
25%         57.000000
50%        101.000000
75%        133.000000
max        333.000000
Name: clean_query, dtype: float64


In [ ]:
# ============================================================
# TF-IDF REPRESENTATION
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words="english",
    ngram_range=(1, 2),
    min_df=20,
    max_df=0.95,
    max_features=25000,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(
    intent_corpus["clean_query"]
)

print("TF-IDF shape:", X_tfidf.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

TF-IDF shape: (83418, 6682)
Vocabulary size: 6682


In [ ]:
# ============================================================
# NMF TOPIC DISCOVERY
# ============================================================

from sklearn.decomposition import NMF

N_TOPICS = 20

nmf = NMF(
    n_components=N_TOPICS,
    init="nndsvda",
    random_state=42,
    max_iter=300
)

W = nmf.fit_transform(X_tfidf)
H = nmf.components_

print("Topic matrix shape:", W.shape)
print("Component matrix shape:", H.shape)

Topic matrix shape: (83418, 20)
Component matrix shape: (20, 6682)


In [ ]:
# ============================================================
# DISPLAY DISCOVERED TOPICS
# ============================================================

feature_names = np.array(tfidf.get_feature_names_out())

print("=" * 90)
print("DISCOVERED INTENT TOPICS")
print("=" * 90)

for topic_idx, topic in enumerate(H):
    top_indices = topic.argsort()[-15:][::-1]
    top_words = feature_names[top_indices]

    print(
        f"\nTOPIC {topic_idx + 1:02d}:"
    )
    print(" | ".join(top_words))

DISCOVERED INTENT TOPICS

TOPIC 01:
user | こ丁寧にありかとうこさいます | amzl | user amazon | ありかとうこさいました | user amzl | gracias | わさわさありかとうこさいます | user こ丁寧にありかとうこさいます | ok | dm | app | user ok | ich | danke

TOPIC 02:
url | packaging | box | url url | amazon url | hey user | hey | really | help url | like | think | seriously | waste | user packaging | order url

TOPIC 03:
amazon | user amazon | amazon prime | amazon logistics | logistics | india | amazon url | amazon pay | amazon india | pay | amazon echo | amazon delivery | amazon shipping | echo | amazon music

TOPIC 04:
user user | user | product | india | customers | guys | days | id | fake | money | reply | worst | update | people | fraud

TOPIC 05:
user url | user | url | thanks user | amazon user | gracias user | merci user | wtf user | packaging user | thank user | thanks | amo | really user | wtf | gracias

TOPIC 06:
delivery | user delivery | date | day delivery | delivery date | time | day | driver | deliver | amazon delivery | today | d

In [ ]:
# ============================================================
# ASSIGN DOMINANT DISCOVERY TOPIC
# ============================================================

intent_corpus["discovered_topic"] = (
    W.argmax(axis=1) + 1
)

intent_corpus["topic_strength"] = (
    W.max(axis=1)
)

topic_counts = (
    intent_corpus["discovered_topic"]
    .value_counts()
    .sort_index()
)

print("=" * 70)
print("CASES PER DISCOVERED TOPIC")
print("=" * 70)

print(topic_counts)

CASES PER DISCOVERED TOPIC
discovered_topic
1     12622
2      5280
3      2437
4      3862
5      1903
6      2563
7      2260
8      1481
9      1412
10      458
11     5775
12     3852
13     5316
14    10033
15     4449
16     2753
17     6545
18     3434
19     3229
20     3754
Name: count, dtype: int64


In [ ]:
# ============================================================
# GOLDEN SET — INTENT SUMMARY
# ============================================================

golden_eval = pd.read_csv(
    "data/evaluation/golden_evaluation_set.csv"
)

intent_summary = (
    golden_eval["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="count")
)

intent_summary["percentage"] = (
    intent_summary["count"] / len(golden_eval) * 100
).round(2)

print("=" * 80)
print("FINAL GOLDEN INTENT DISTRIBUTION")
print("=" * 80)

print(intent_summary.to_string(index=False))

FINAL GOLDEN INTENT DISTRIBUTION
             intent  count  percentage
     ORDER_DELIVERY     69        34.5
    GENERAL_SUPPORT     35        17.5
     GENERAL_SOCIAL     17         8.5
    RETURNS_REFUNDS     16         8.0
    PAYMENT_BILLING     11         5.5
    DIGITAL_CONTENT      9         4.5
     ORDER_TRACKING      9         4.5
     ACCOUNT_ACCESS      9         4.5
   DEVICE_TECHNICAL      5         2.5
   ACCOUNT_SECURITY      4         2.0
 ORDER_CANCELLATION      4         2.0
  PREORDER_DELIVERY      4         2.0
   PURCHASE_CONTROL      3         1.5
MARKETPLACE_SELLING      3         1.5
     ORDER_PURCHASE      1         0.5
SELLER_AUTHENTICITY      1         0.5


In [ ]:
# ============================================================
# INTENT × RESOLUTION
# ============================================================

intent_resolution = pd.crosstab(
    golden_eval["intent"],
    golden_eval["resolution_status"]
)

print("=" * 80)
print("INTENT × RESOLUTION STATUS")
print("=" * 80)

display(intent_resolution)

INTENT × RESOLUTION STATUS


resolution_status,ESCALATION_REQUIRED,INVESTIGATION_REQUIRED,NOT_APPLICABLE,PARTIALLY_RESOLVED,RESOLVED,RESOLVED_BY_CUSTOMER,UNRESOLVED
intent,,,,,,,
ACCOUNT_ACCESS,3,5,0,1,0,0,0
ACCOUNT_SECURITY,2,0,0,0,2,0,0
DEVICE_TECHNICAL,0,3,0,0,2,0,0
DIGITAL_CONTENT,0,3,0,2,3,0,1
GENERAL_SOCIAL,0,0,16,0,0,1,0
GENERAL_SUPPORT,1,22,1,4,5,2,0
MARKETPLACE_SELLING,0,2,0,0,0,0,1
ORDER_CANCELLATION,0,2,0,0,1,1,0
ORDER_DELIVERY,0,49,0,8,4,1,7


In [ ]:
# ============================================================
# INTENT × AUTOMATION
# ============================================================

intent_automation = pd.crosstab(
    golden_eval["intent"],
    golden_eval["automation_decision"]
)

print("=" * 80)
print("INTENT × AUTOMATION DECISION")
print("=" * 80)

display(intent_automation)

INTENT × AUTOMATION DECISION


automation_decision,AUTO,HUMAN
intent,,
ACCOUNT_ACCESS,1,8
ACCOUNT_SECURITY,0,4
DEVICE_TECHNICAL,2,3
DIGITAL_CONTENT,5,4
GENERAL_SOCIAL,17,0
GENERAL_SUPPORT,12,23
MARKETPLACE_SELLING,0,3
ORDER_CANCELLATION,2,2
ORDER_DELIVERY,7,62


In [ ]:
print("=" * 80)
print("INTENT × RISK LEVEL")
print("=" * 80)

intent_risk = pd.crosstab(
    golden_clean["intent"],
    golden_clean["risk_level"]
)

display(intent_risk)

print("\n" + "=" * 80)
print("RISK × RESOLUTION STATUS")
print("=" * 80)

risk_resolution = pd.crosstab(
    golden_clean["risk_level"],
    golden_clean["resolution_status"]
)

display(risk_resolution)

INTENT × RISK LEVEL


risk_level,HIGH,LOW,MEDIUM
intent,,,
ACCOUNT_ACCESS,5,0,4
ACCOUNT_SECURITY,4,0,0
DEVICE_TECHNICAL,0,5,0
DIGITAL_CONTENT,0,8,1
GENERAL_SOCIAL,0,17,0
GENERAL_SUPPORT,1,17,17
MARKETPLACE_SELLING,1,0,2
ORDER_CANCELLATION,0,2,2
ORDER_DELIVERY,9,38,22



RISK × RESOLUTION STATUS


resolution_status,ESCALATION_REQUIRED,INVESTIGATION_REQUIRED,NOT_APPLICABLE,PARTIALLY_RESOLVED,RESOLVED,RESOLVED_BY_CUSTOMER,UNRESOLVED
risk_level,,,,,,,
HIGH,6,16,0,0,2,0,2
LOW,0,44,17,15,19,5,6
MEDIUM,0,61,0,5,0,0,2


In [ ]:
print("=" * 80)
print("RISK × AUTOMATION DECISION")
print("=" * 80)

risk_auto = pd.crosstab(
    golden_clean["risk_level"],
    golden_clean["automation_decision"]
)

display(risk_auto)

print("\n" + "=" * 80)
print("AUTOMATION RATE BY RISK")
print("=" * 80)

automation_rate = (
    golden_clean
    .groupby("risk_level")["automation_decision"]
    .apply(lambda x: (x == "AUTO").mean() * 100)
    .reset_index(name="auto_rate_percent")
)

display(automation_rate)

RISK × AUTOMATION DECISION


automation_decision,AUTO,HUMAN
risk_level,,
HIGH,0,26
LOW,51,55
MEDIUM,1,67



AUTOMATION RATE BY RISK


,risk_level,auto_rate_percent
0,HIGH,0.000000
1,LOW,48.113208
2,MEDIUM,1.470588


In [ ]:
golden_clean.to_csv(
    "data/evaluation/golden_evaluation_set_frozen.csv",
    index=False
)

print("Frozen golden set saved.")
print(golden_clean.shape)

Frozen golden set saved.
(200, 16)


In [ ]:
# ============================================================
# BASELINE 1 — MAJORITY-CLASS INTENT CLASSIFIER
# ============================================================

from sklearn.metrics import accuracy_score, classification_report

# Ground-truth labels from frozen golden set
y_true = golden_clean["intent"]

# Most frequent intent
majority_intent = y_true.value_counts().idxmax()

# Predict the same intent for every case
y_pred = [majority_intent] * len(y_true)

accuracy = accuracy_score(y_true, y_pred)

print("=" * 80)
print("BASELINE 1 — MAJORITY-CLASS INTENT CLASSIFIER")
print("=" * 80)

print(f"\nMajority intent: {majority_intent}")
print(f"Evaluation cases: {len(y_true)}")
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")

print("\nPer-class performance:")
print(
    classification_report(
        y_true,
        y_pred,
        zero_division=0
    )
)

BASELINE 1 — MAJORITY-CLASS INTENT CLASSIFIER

Majority intent: ORDER_DELIVERY
Evaluation cases: 200
Accuracy: 0.3450 (34.50%)

Per-class performance:
                     precision    recall  f1-score   support

     ACCOUNT_ACCESS       0.00      0.00      0.00         9
   ACCOUNT_SECURITY       0.00      0.00      0.00         4
   DEVICE_TECHNICAL       0.00      0.00      0.00         5
    DIGITAL_CONTENT       0.00      0.00      0.00         9
     GENERAL_SOCIAL       0.00      0.00      0.00        17
    GENERAL_SUPPORT       0.00      0.00      0.00        35
MARKETPLACE_SELLING       0.00      0.00      0.00         3
 ORDER_CANCELLATION       0.00      0.00      0.00         4
     ORDER_DELIVERY       0.34      1.00      0.51        69
     ORDER_PURCHASE       0.00      0.00      0.00         1
     ORDER_TRACKING       0.00      0.00      0.00         9
    PAYMENT_BILLING       0.00      0.00      0.00        11
  PREORDER_DELIVERY       0.00      0.00      0.00     

In [ ]:
# ============================================================
# PREPARE INTENT TRAINING DATA
# ============================================================

import pandas as pd
import re

# Frozen evaluation case IDs
golden_case_ids = set(golden_clean["case_id"])

# Start from the reconstructed resolution dataset
intent_train_df = resolution_df[
    ~resolution_df["case_id"].isin(golden_case_ids)
].copy()

print("=" * 80)
print("INTENT TRAINING DATA")
print("=" * 80)

print(f"Training cases: {len(intent_train_df):,}")
print(f"Golden evaluation cases excluded: {len(golden_case_ids)}")

print("\nTraining columns:")
print(intent_train_df.columns.tolist())

INTENT TRAINING DATA
Training cases: 83,218
Golden evaluation cases excluded: 200

Training columns:
['case_id', 'root_tweet_id', 'turn_count', 'customer_turns', 'support_turns', 'duration_hours', 'max_gap_hours', 'customer_query', 'support_response', 'final_role', 'support_response_length', 'customer_query_length']


In [ ]:
# ============================================================
# INTENT SIGNAL INSPECTION
# ============================================================

import re
from collections import Counter

queries = intent_train_df["customer_query"].fillna("").astype(str)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"\b\d{5,}\b", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

intent_train_df["clean_query"] = queries.apply(clean_text)

# Candidate keyword groups
intent_keywords = {
    "ORDER_DELIVERY": [
        "delivery", "deliver", "delivered", "package",
        "parcel", "arrive", "shipping", "driver"
    ],
    "ORDER_TRACKING": [
        "tracking", "track my", "where is my order",
        "shipment", "tracking number"
    ],
    "RETURNS_REFUNDS": [
        "refund", "return", "returned", "money back"
    ],
    "PAYMENT_BILLING": [
        "charged", "charge", "payment", "billing",
        "credit card", "debit card", "paid"
    ],
    "ORDER_CANCELLATION": [
        "cancel order", "cancelled order",
        "cancel my order"
    ],
    "ACCOUNT_ACCESS": [
        "can't login", "cannot login", "login",
        "sign in", "password", "account access"
    ],
    "ACCOUNT_SECURITY": [
        "hacked", "hack", "security", "unauthorized",
        "stolen account", "someone accessed"
    ],
    "DIGITAL_CONTENT": [
        "prime video", "video", "music", "kindle",
        "ebook", "digital"
    ],
    "PURCHASE_CONTROL": [
        "one click", "purchase control",
        "accidental purchase", "child purchase"
    ],
}

print("=" * 80)
print("HIGH-PRECISION INTENT SIGNAL COVERAGE")
print("=" * 80)

for intent, keywords in intent_keywords.items():
    pattern = "|".join(re.escape(k) for k in keywords)

    matches = intent_train_df["clean_query"].str.contains(
        pattern,
        regex=True,
        na=False
    )

    count = matches.sum()
    pct = count / len(intent_train_df) * 100

    print(f"{intent:25s} {count:7,} ({pct:6.2f}%)")

HIGH-PRECISION INTENT SIGNAL COVERAGE
ORDER_DELIVERY             18,555 ( 22.30%)
ORDER_TRACKING              1,351 (  1.62%)
RETURNS_REFUNDS             4,413 (  5.30%)
PAYMENT_BILLING             2,779 (  3.34%)
ORDER_CANCELLATION            154 (  0.19%)
ACCOUNT_ACCESS                468 (  0.56%)
ACCOUNT_SECURITY              272 (  0.33%)
DIGITAL_CONTENT             2,365 (  2.84%)
PURCHASE_CONTROL               16 (  0.02%)


In [ ]:
# ============================================================
# PRECISION-FIRST WEAK LABELING
# ============================================================

import re
import pandas as pd

def contains_any(text, phrases):
    return any(phrase in text for phrase in phrases)


# More specific intents first.
# This reduces obvious overlap between broad categories.
RULES = {
    "ACCOUNT_SECURITY": [
        "hacked",
        "hack my account",
        "account hacked",
        "unauthorized access",
        "unauthorized login",
        "someone accessed my account",
        "someone has access to my account",
        "stolen account",
        "account compromised",
    ],

    "PURCHASE_CONTROL": [
        "one click",
        "accidental purchase",
        "accidental order",
        "child purchased",
        "child purchase",
        "purchase control",
    ],

    "ORDER_CANCELLATION": [
        "cancel my order",
        "cancel the order",
        "cancel order",
        "cancelled my order",
        "canceled my order",
    ],

    "ORDER_TRACKING": [
        "tracking number",
        "tracking says",
        "tracking shows",
        "track my order",
        "track order",
        "where is my order",
        "where's my order",
        "where is the package",
        "where's the package",
        "track my package",
        "track package",
    ],

    "RETURNS_REFUNDS": [
        "refund",
        "refund me",
        "money back",
        "return my",
        "return the",
        "return item",
        "return product",
        "returned item",
        "returning item",
    ],

    "PAYMENT_BILLING": [
        "charged",
        "charge",
        "payment",
        "billing",
        "credit card",
        "debit card",
        "payment failed",
        "payment declined",
        "charged twice",
        "double charged",
    ],

    "ACCOUNT_ACCESS": [
        "can't login",
        "cannot login",
        "cant login",
        "can't log in",
        "cannot log in",
        "cant log in",
        "sign in",
        "can't sign in",
        "cannot sign in",
        "forgot password",
        "reset password",
        "password reset",
    ],

    "DIGITAL_CONTENT": [
        "prime video",
        "prime music",
        "amazon music",
        "kindle",
        "ebook",
        "e-book",
        "digital content",
        "digital purchase",
    ],

    "ORDER_DELIVERY": [
        "delivery",
        "deliver",
        "delivered",
        "package",
        "parcel",
        "shipping",
        "shipment",
        "driver",
        "arrive",
        "arrived",
    ],
}


def assign_weak_label(text):
    """
    Returns:
        label        : assigned intent or None
        match_count  : number of intent rules matched
        matched      : intents matched
    """

    matches = []

    for intent, phrases in RULES.items():
        if contains_any(text, phrases):
            matches.append(intent)

    # Only assign automatically when exactly one intent matches.
    # Ambiguous cases are deliberately left unlabeled.
    if len(matches) == 1:
        return matches[0], 1, matches

    return None, len(matches), matches


results = intent_train_df["clean_query"].apply(assign_weak_label)

intent_train_df["weak_intent"] = results.apply(lambda x: x[0])
intent_train_df["rule_match_count"] = results.apply(lambda x: x[1])
intent_train_df["matched_intents"] = results.apply(lambda x: x[2])


# ============================================================
# COVERAGE
# ============================================================

total = len(intent_train_df)

labeled = intent_train_df["weak_intent"].notna().sum()
unlabeled = total - labeled

print("=" * 80)
print("WEAK LABELING SUMMARY")
print("=" * 80)

print(f"Total training cases : {total:,}")
print(f"Labeled cases        : {labeled:,}")
print(f"Unlabeled cases      : {unlabeled:,}")
print(f"Coverage             : {labeled / total * 100:.2f}%")

print("\n" + "=" * 80)
print("WEAK LABEL DISTRIBUTION")
print("=" * 80)

display(
    intent_train_df["weak_intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="count")
)

print("\n" + "=" * 80)
print("AMBIGUOUS CASES")
print("=" * 80)

ambiguous = intent_train_df[
    intent_train_df["rule_match_count"] > 1
]

print(f"Ambiguous cases: {len(ambiguous):,}")
print(f"Ambiguity rate : {len(ambiguous) / total * 100:.2f}%")

display(
    ambiguous["rule_match_count"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_matching_intents")
    .reset_index(name="count")
)

WEAK LABELING SUMMARY
Total training cases : 83,218
Labeled cases        : 23,066
Unlabeled cases      : 60,152
Coverage             : 27.72%

WEAK LABEL DISTRIBUTION


,intent,count
0,ORDER_DELIVERY,17489
1,RETURNS_REFUNDS,2082
2,DIGITAL_CONTENT,1488
3,PAYMENT_BILLING,1468
4,ORDER_CANCELLATION,179
5,ACCOUNT_ACCESS,143
6,ORDER_TRACKING,109
7,ACCOUNT_SECURITY,98
8,PURCHASE_CONTROL,10



AMBIGUOUS CASES
Ambiguous cases: 1,576
Ambiguity rate : 1.89%


,number_of_matching_intents,count
0,2,1508
1,3,67
2,4,1


In [ ]:
# ============================================================
# INSPECT UNLABELED TRAINING CASES
# ============================================================

unlabeled_cases = intent_train_df[
    intent_train_df["weak_intent"].isna()
].copy()

print("=" * 80)
print("UNLABELED CASE SAMPLE")
print("=" * 80)

sample_unlabeled = (
    unlabeled_cases[
        ["case_id", "customer_query", "support_response"]
    ]
    .sample(
        n=min(100, len(unlabeled_cases)),
        random_state=42
    )
    .reset_index(drop=True)
)

display(sample_unlabeled)

UNLABELED CASE SAMPLE


,case_id,customer_query,support_response
0,AMZ-069267,@AmazonHelp I have had it for one day. It's not a troubleshooting issue I just wanted to know if this is a common thing across all devices or if it is a fault.,@601146 Sorry to hear you're having issues with your Kindle Oasis. Please reach out to us here: https://t.co/JzP7hlA23B so we can look into this for you.^ES
1,AMZ-020080,アマゾンで頼んだ商品が到着予定日を2日過ぎても届かないので、もう本当にそれどころじゃないくらい焦ってる。今週中に来てくれないと大変。,@428350 なお、Amazonマーケットプレイスの出品者が発送する商品の場合は、出品者に直接お問い合わせください。出品者へのお問い合わせ方法はこちらをご参照ください。https://t.co/ZXCXcbN3Tv TN
2,AMZ-024288,"@AmazonHelp And when we will be able to buy products which cannot be send to jammu like furniture, sofa , recliners ....",@207168 No concrete update on this yet. Please stay tuned. ^MP
3,AMZ-046068,"@115821 Hola gracias por vuestra respuesta, fue directamente desde vuestra web. Ya está en devolución, pero una pena para la peque. La próxima, a embalar!","@529829 Estamos para ayudar. En caso de necesitar de asistencia en el futuro, no dudes en contactarnos de nuevo. Saludos. ^CR"
4,AMZ-004726,Gracias @116875 otra compra con ustedes 👍 https://t.co/FM4JQ4yDSJ,@189674 ¡Genial! ¿Cuando nos invitas a una partidita? ^JD
...,...,...,...
95,AMZ-017306,"¡Han llegado cositas de @115821!\nParece que esto de autoregalarse, funcionaaa 😂👌 https://t.co/X06xBaa4hh",@425619 ¡Maravilloso! Un sentimiento increíble. Que disfrutes. ❤️😊^JD https://t.co/iD5HfbZmCA
96,AMZ-036015,"When you pay extra for same day delivery but @115821 decides “nahhhhh, you’ll get it in 2 days instead” AND they don’t refund me https://t.co/MenWm9WZV6","@745559 I'm sorry! When you reached out to us initially, what options or insight were we able to provide? ^WT"
97,AMZ-027440,@AmazonHelp キャンセルしました。返信ありがとうございました。,@387609 とんでもないことでございます。今後もhttps://t.co/st4oU5QbhPをどうぞよろしくお願いいたします。TN
98,AMZ-019818,@AmazonHelp What's your phone number?,@161464 We're available 24/7 here: https://t.co/EKXRLsnxJu ^GG


In [ ]:
# ============================================================
# INSPECT AMBIGUOUS CASES
# ============================================================

print("=" * 80)
print("AMBIGUOUS CASE SAMPLE")
print("=" * 80)

ambiguous_sample = (
    ambiguous[
        [
            "case_id",
            "customer_query",
            "support_response",
            "matched_intents"
        ]
    ]
    .sample(
        n=min(50, len(ambiguous)),
        random_state=42
    )
    .reset_index(drop=True)
)

display(ambiguous_sample)

AMBIGUOUS CASE SAMPLE


,case_id,customer_query,support_response,matched_intents
0,AMZ-070394,"@AmazonHelp Yeah, I know I can return the whole thing but I don't think that we need to bring three shipping charges, and two vehicles into what can be solved by shipping one replacement cable.","@547100 If the cable is shipped from and sold by Amazon, a replacement may be an option. If you'd prefer, you're welcome to contact us by phone, chat or e-mail to explore options here: https://t.co/hApLpMlfHN ^ST","[RETURNS_REFUNDS, PAYMENT_BILLING, ORDER_DELIVERY]"
1,AMZ-049440,@AmazonHelp If I wanted my package delivered eventually I wouldn't have paid 100 for prime. At this point I really just want a refund on prime because getting 5 dollars doesn't make up for multiple failures to deliver on promised times by Amazon proper no third parties involved,"@245497 Hey, we'd like to investigate it further, please submit this secure form: https://t.co/mk7tow11c4 ^AT","[RETURNS_REFUNDS, ORDER_DELIVERY]"
2,AMZ-067036,"@AmazonHelp Mhm I haven’t received an email yet, but if I’m not mistaken if the payment was taken out of my account does that mean it’s in the shipping process or has shipped already ?",@155469 Usually you will see a pending authorization when the order is placed which and then see a posted charge once the order is shipped. ^KH,"[PAYMENT_BILLING, ORDER_DELIVERY]"
3,AMZ-032060,$15 for shipping on a 16lb bag of cat food is robbery! Can I transfer my gift card balance back to my debit card? @115821 @AmazonHelp,@431910 I can understand your concern! We're happy to look into avail options w/ you via phone/chat here: https://t.co/hApLpMlfHN ^CS,"[PAYMENT_BILLING, ORDER_DELIVERY]"
4,AMZ-030457,How’s my Thursday going? Well my amazon account got hacked and was charged $900... &amp; I’m honestly impressed bc I didn’t know I had $900,@425297 I'm sorry to hear about the unexpected charge! Please call us to ensure this is reported: https://t.co/jzvkhdlrK5 ^LR,"[ACCOUNT_SECURITY, PAYMENT_BILLING]"
5,AMZ-081344,"@115850 I had cancelled my order 171-3881888-6984365, but I am yet to receive my refund. Can you pls update?","@243540 Also, please don't provide your order details, as we consider it to be personal information. Our Twitter page is visible to the public. ^AP","[ORDER_CANCELLATION, RETURNS_REFUNDS]"
6,AMZ-031382,"Watching my daughter read on her #kindle paperwhite, while my kindle won't charge or reset/reboot. On hold with @115821 now for 30 min+","@425662 I'm sorry for the wait, please keep us updated on the outcome of your phone call. We want to ensure you're helped. ^LJ","[PAYMENT_BILLING, DIGITAL_CONTENT]"
7,AMZ-016523,@115830 i just found out amazon charged me £79 for prime for a year. Why ? Can i get my money back ? Im not a reg customer...,"@388012 I'm sorry! If no benefits have been used, canceling issues a full refund. You can do so here: https://t.co/nNXpGJVjuR. ^JZ","[RETURNS_REFUNDS, PAYMENT_BILLING]"
8,AMZ-029375,"Hey, @115821?! Any chance you can tell me what's up w/ this? All say free shipping, but EVERY ONE tacks on a shipping charge at checkout! https://t.co/7AJ6QBKdmF",@405396 Sorry for the trouble! We'd like to hear more from you on this issue. Please reach us here: https://t.co/hApLpMlfHN. ^VB,"[PAYMENT_BILLING, ORDER_DELIVERY]"
9,AMZ-004425,@117093 You are constantly taking monthly charges from my disabled brothers account when you can't deliver to his new residential area! 4 times even after he tried cancelling 4x!!😡what's going on? 1/2,"@186596 I'm sorry to hear this! Have you had a chance to reach out to us? If so, what info did we provide you? ^AC","[PAYMENT_BILLING, ORDER_DELIVERY]"


In [ ]:
# ============================================================
# SAVE UNLABELED SAMPLE FOR SYSTEMATIC INSPECTION
# ============================================================

unlabeled_sample_200 = (
    unlabeled_cases[
        ["case_id", "customer_query", "support_response"]
    ]
    .sample(
        n=min(200, len(unlabeled_cases)),
        random_state=123
    )
    .reset_index(drop=True)
)

display(unlabeled_sample_200)

# Save locally for inspection
unlabeled_sample_200.to_csv(
    "data/evaluation/unlabeled_intent_sample_200.csv",
    index=False
)

print(f"\nSaved {len(unlabeled_sample_200)} unlabeled examples.")

,case_id,customer_query,support_response
0,AMZ-031330,@AmazonHelp el problema es q mi epsoso viaja mañana y por eso necesitamos que aparezca el paquete,"@425299 Hola Lucia, ¿en cuál de nuestros sitios web realizaste tu pedido? ^JD"
1,AMZ-042373,@AmazonHelp Nope nothing there,"@488265 Thanks for checking. We'd like to take another go at this with you. Using ^GM's link, please reach us when you can. ^DW"
2,AMZ-072613,"@AmazonHelp Also, how do I resolve this error message? Thanks! https://t.co/Lhd6NXO60B","@155232 That's odd! If you haven't already, try restarting your device. If the issue persists, please let us know. ^MV"
3,AMZ-071648,@115821 it's impossible to open claim on defective item. I hit return andbrheybdenied tried every instruction u posted. No help. ADVISE,"@675306 Sorry for any inconvenience caused, we're here to help! Click here: https://t.co/UADgbVzyJF to learn how to file an A-to-z Guarantee claim. ^JE"
4,AMZ-042210,@AmazonHelp What an inconvenience. Pls fix your calls to customer care issue. I feel Amazon is getting worse day by day,@486414 here: https://t.co/GIJyeYqKE0 if your concern regarding pay balance is not resolved. 2/2 ^MJ
...,...,...,...
195,AMZ-028886,"@AmazonHelp Klassiker, hat funktioniert^^",@145706 Na also ;-) ^MF
196,AMZ-014767,https://t.co/NyAuiFpvBr,@218503 Please reply to the email correspondence here: https://t.co/DTSNmGldJf and we'll be glad to help you. (2/2) ^SI
197,AMZ-041893,"@121284 Didn't get any update, it's been more than a day 😕 https://t.co/huhKYWefGV",@484859 I’d like to help you; please fill this form: https://t.co/beaaDlIL5C and I’ll contact you soon. ^PJ
198,AMZ-016537,"@AmazonHelp Yep, I got the email but nothing happened after that. Will just sending a mail resolve the issue?","@319186 My apologies for the trouble, Ganesh. Kindly revert to the email for more information on this. ^KS"



Saved 200 unlabeled examples.


In [ ]:
# ============================================================
# WEAK LABELER V2
# Add missing high-confidence intents
# ============================================================

RULES_V2 = {

    # ---------- HIGH PRIORITY / SPECIFIC ----------

    "ACCOUNT_SECURITY": [
        "hacked",
        "hack my account",
        "account hacked",
        "unauthorized access",
        "unauthorized login",
        "someone accessed my account",
        "someone has access to my account",
        "stolen account",
        "account compromised",
        "account was hacked",
    ],

    "SELLER_AUTHENTICITY": [
        "fake product",
        "fake item",
        "counterfeit",
        "counterfeit product",
        "counterfeit item",
        "is this fake",
        "is this genuine",
        "is this authentic",
        "not genuine",
        "not authentic",
        "seller sent fake",
        "seller sold fake",
    ],

    "PURCHASE_CONTROL": [
        "one click",
        "accidental purchase",
        "accidental order",
        "child purchased",
        "child purchase",
        "purchase control",
    ],

    "MARKETPLACE_SELLING": [
        "sell on amazon",
        "selling on amazon",
        "amazon seller",
        "seller account",
        "become a seller",
        "sell products",
        "selling products",
        "marketplace seller",
    ],

    "PREORDER_DELIVERY": [
        "preorder",
        "pre-order",
        "pre ordered",
        "pre-ordered",
        "preorder delivery",
        "preorder date",
    ],

    "ORDER_CANCELLATION": [
        "cancel my order",
        "cancel the order",
        "cancel order",
        "cancelled my order",
        "canceled my order",
    ],

    "ORDER_PURCHASE": [
        "how do i order",
        "how can i order",
        "how to order",
        "place an order",
        "placing an order",
        "want to buy",
        "where can i buy",
        "how can i buy",
        "available to buy",
    ],

    # ---------- ACCOUNT ----------

    "ACCOUNT_ACCESS": [
        "can't login",
        "cannot login",
        "cant login",
        "can't log in",
        "cannot log in",
        "cant log in",
        "sign in",
        "can't sign in",
        "cannot sign in",
        "forgot password",
        "reset password",
        "password reset",
        "unable to login",
        "unable to sign in",
    ],

    # ---------- TECHNICAL ----------

    "DEVICE_TECHNICAL": [
        "error message",
        "error code",
        "device not working",
        "device isn't working",
        "device is not working",
        "device won't work",
        "device wont work",
        "won't turn on",
        "wont turn on",
        "won't charge",
        "wont charge",
        "not charging",
        "can't charge",
        "cannot charge",
        "crashing",
        "app crashes",
        "app crash",
        "app not working",
        "app isn't working",
        "app is not working",
        "software issue",
        "technical issue",
        "technical problem",
        "troubleshooting",
        "troubleshoot",
        "reboot",
        "reset device",
        "restart device",
        "kindle won't",
        "kindle wont",
    ],

    # ---------- ORDERS ----------

    "ORDER_TRACKING": [
        "tracking number",
        "tracking says",
        "tracking shows",
        "track my order",
        "track order",
        "where is my order",
        "where's my order",
        "where is the package",
        "where's the package",
        "track my package",
        "track package",
        "shipment tracking",
    ],

    "RETURNS_REFUNDS": [
        "refund",
        "refund me",
        "money back",
        "return my",
        "return the",
        "return item",
        "return product",
        "returned item",
        "returning item",
        "get a refund",
        "want a refund",
        "need a refund",
    ],

    "PAYMENT_BILLING": [
        "charged",
        "charge",
        "payment",
        "billing",
        "credit card",
        "debit card",
        "payment failed",
        "payment declined",
        "charged twice",
        "double charged",
        "unexpected charge",
        "monthly charge",
    ],

    "DIGITAL_CONTENT": [
        "prime video",
        "prime music",
        "amazon music",
        "kindle",
        "ebook",
        "e-book",
        "digital content",
        "digital purchase",
        "kindle unlimited",
    ],

    "ORDER_DELIVERY": [
        "delivery",
        "deliver",
        "delivered",
        "package",
        "parcel",
        "shipping",
        "shipment",
        "driver",
        "arrive",
        "arrived",
        "delivery date",
        "late delivery",
        "delayed delivery",
    ],
}


def assign_weak_label_v2(text):
    matches = []

    for intent, phrases in RULES_V2.items():
        if contains_any(text, phrases):
            matches.append(intent)

    # Precision-first:
    # exactly one intent = label
    # multiple intents = ambiguous
    # zero intents = unknown
    if len(matches) == 1:
        return matches[0], 1, matches

    return None, len(matches), matches


results_v2 = intent_train_df["clean_query"].apply(
    assign_weak_label_v2
)

intent_train_df["weak_intent_v2"] = results_v2.apply(
    lambda x: x[0]
)

intent_train_df["rule_match_count_v2"] = results_v2.apply(
    lambda x: x[1]
)

intent_train_df["matched_intents_v2"] = results_v2.apply(
    lambda x: x[2]
)


# ============================================================
# SUMMARY
# ============================================================

total = len(intent_train_df)

labeled_v2 = intent_train_df["weak_intent_v2"].notna().sum()
unlabeled_v2 = total - labeled_v2

ambiguous_v2 = (
    intent_train_df["rule_match_count_v2"] > 1
).sum()

print("=" * 80)
print("WEAK LABELER V2 SUMMARY")
print("=" * 80)

print(f"Total cases       : {total:,}")
print(f"Labeled cases     : {labeled_v2:,}")
print(f"Unlabeled cases   : {unlabeled_v2:,}")
print(f"Coverage          : {labeled_v2 / total * 100:.2f}%")
print(f"Ambiguous cases   : {ambiguous_v2:,}")
print(f"Ambiguity rate    : {ambiguous_v2 / total * 100:.2f}%")

print("\n" + "=" * 80)
print("V2 LABEL DISTRIBUTION")
print("=" * 80)

display(
    intent_train_df["weak_intent_v2"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="count")
)

WEAK LABELER V2 SUMMARY
Total cases       : 83,218
Labeled cases     : 23,536
Unlabeled cases   : 59,682
Coverage          : 28.28%
Ambiguous cases   : 1,883
Ambiguity rate    : 2.26%

V2 LABEL DISTRIBUTION


,intent,count
0,ORDER_DELIVERY,17269
1,RETURNS_REFUNDS,2063
2,DIGITAL_CONTENT,1460
3,PAYMENT_BILLING,1440
4,PREORDER_DELIVERY,396
5,ORDER_CANCELLATION,176
6,ACCOUNT_ACCESS,146
7,DEVICE_TECHNICAL,139
8,ORDER_TRACKING,109
9,ACCOUNT_SECURITY,98


In [ ]:
# ============================================================
# WEAK LABEL QUALITY AUDIT
# ============================================================

audit_frames = []

for intent in sorted(
    intent_train_df["weak_intent_v2"].dropna().unique()
):
    subset = intent_train_df[
        intent_train_df["weak_intent_v2"] == intent
    ]

    n = min(20, len(subset))

    sample_intent = subset.sample(
        n=n,
        random_state=42
    ).copy()

    sample_intent["audit_intent"] = intent

    audit_frames.append(
        sample_intent[
            [
                "case_id",
                "customer_query",
                "support_response",
                "weak_intent_v2",
                "matched_intents_v2"
            ]
        ]
    )

weak_label_audit = pd.concat(
    audit_frames,
    ignore_index=True
)

print("=" * 80)
print("WEAK LABEL QUALITY AUDIT")
print("=" * 80)

print(f"Total examples for audit: {len(weak_label_audit)}")

display(weak_label_audit)

WEAK LABEL QUALITY AUDIT
Total examples for audit: 270


,case_id,customer_query,support_response,weak_intent_v2,matched_intents_v2
0,AMZ-019886,I’m locked out of my @115821 account and apparently can’t get help unless I sign in. Seriously? What am I supposed to do???,@546854 I'm sorry for the account troubles. Please contact us here: https://t.co/jzvkhdlrK5 ^RR,ACCOUNT_ACCESS,[ACCOUNT_ACCESS]
1,AMZ-052618,"@AmazonHelp Done. Look forward to having this resolved at the earliest. Although please note that my concern is with Amazon US, not India. Also, I used my number to sign in through the app so I do not have any email associated with my amazon account.",@686478 Thank you for sharing your details. We'll work on it and reach out to you soon. ^AP,ACCOUNT_ACCESS,[ACCOUNT_ACCESS]
2,AMZ-010982,"@115850 I'm not getting any verification code on my email,as I have changed my phone.. I'm unable to sign in.what can I do??",@299817 Sorry for the hassle. Please report this to our support team here: https://t.co/vlvfJr4nN9 and we'll check this. ^HN,ACCOUNT_ACCESS,[ACCOUNT_ACCESS]
3,AMZ-009101,"@AmazonHelp Sure, I have attached a screenshot below, every time it shows same, even I tried to reset password, but once I reset it, the error is same. https://t.co/Vppez8oFqs",@215661 I'd like to check this for you. Please write to us here: https://t.co/GIJyeYqKE0 and I'll reach out to you. ^SA,ACCOUNT_ACCESS,[ACCOUNT_ACCESS]
4,AMZ-018414,@AmazonHelp I’m unable to login to my account could I get some help unlocking it?,@382623 I'm sorry to see this! Please reach us for live assistance when you can via this link here: https://t.co/jzvkhdlrK5 ^DW,ACCOUNT_ACCESS,[ACCOUNT_ACCESS]
...,...,...,...,...,...
265,AMZ-024330,@115850 I received spurious/counterfeit ink from one of your sellers today. I'll give you proof. I want action taken to ban this seller from the marketplace.,@126724 That's strange. I've forwarded your feedback internally to the appropriate team as it helps us improve customer experience. 1/2 ^RS,SELLER_AUTHENTICITY,[SELLER_AUTHENTICITY]
266,AMZ-008793,"@118919 Second instance I was sent a fake product, same Seller UBSPD - Wheels Behind the Veil - Pms, Cms and Beyond\nSold by UBSPD",@215062 Have you reported it to the support team here: https://t.co/TxK11znixD (2/2)^HR,SELLER_AUTHENTICITY,[SELLER_AUTHENTICITY]
267,AMZ-005947,Really great that @115821 doesn't care that companies are selling fake products to customers through them. They used to have great service.,@195352 That's not what we want to hear! Have you had a chance to report the fake product to us by phone or chat? ^GR,SELLER_AUTHENTICITY,[SELLER_AUTHENTICITY]
268,AMZ-059375,"@1404 @AmazonHelp @115850 AMAZON SELLS FAKE PRODUCTS ON ITS PLATFORM. AFTER YOU BUY PRODUCT EVEN IF U GIVE LETTER/MAIL FROM MANUFACTURER THAT THE PRODUCT ID FAKE, AMAZON WILL NOT ENTERTAIN AS, AMAZON AT ORGANISATIONAL IS INVOLVED IN SELLING FAKE PRODUCTS","@270757 That's quite a remark! Apologies for the unpleasant experience you've had. We'd like to look into this, kindly drop in your details here: https://t.co/beaaDm0muc &amp; we'll have this checked right away. ^EM",SELLER_AUTHENTICITY,[SELLER_AUTHENTICITY]


In [ ]:
# ============================================================
# BASELINE 2 — TF-IDF + LOGISTIC REGRESSION
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. Keep only high-confidence weak labels
# ------------------------------------------------------------

train_labeled = intent_train_df[
    intent_train_df["weak_intent_v2"].notna()
].copy()

X = train_labeled["clean_query"]
y = train_labeled["weak_intent_v2"]

print("=" * 80)
print("BASELINE 2 — TF-IDF + LOGISTIC REGRESSION")
print("=" * 80)

print(f"Labeled training cases: {len(train_labeled):,}")
print(f"Number of intents: {y.nunique()}")

# ------------------------------------------------------------
# 2. Stratified train/validation split
# ------------------------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"\nTrain cases: {len(X_train):,}")
print(f"Validation cases: {len(X_val):,}")

# ------------------------------------------------------------
# 3. TF-IDF + Logistic Regression
# ------------------------------------------------------------

baseline_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            sublinear_tf=True,
            max_features=50000
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

# ------------------------------------------------------------
# 4. Train
# ------------------------------------------------------------

print("\nTraining model...")

baseline_model.fit(X_train, y_train)

print("Training complete.")

# ------------------------------------------------------------
# 5. Validation evaluation
# ------------------------------------------------------------

val_pred = baseline_model.predict(X_val)

val_accuracy = accuracy_score(y_val, val_pred)
val_macro_f1 = f1_score(
    y_val,
    val_pred,
    average="macro"
)
val_weighted_f1 = f1_score(
    y_val,
    val_pred,
    average="weighted"
)

print("\n" + "=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)

print(f"Accuracy    : {val_accuracy:.4f} ({val_accuracy * 100:.2f}%)")
print(f"Macro F1    : {val_macro_f1:.4f} ({val_macro_f1 * 100:.2f}%)")
print(f"Weighted F1 : {val_weighted_f1:.4f} ({val_weighted_f1 * 100:.2f}%)")

print("\nPer-class performance:")
print(
    classification_report(
        y_val,
        val_pred,
        zero_division=0
    )
)

BASELINE 2 — TF-IDF + LOGISTIC REGRESSION
Labeled training cases: 23,536
Number of intents: 14

Train cases: 18,828
Validation cases: 4,708

Training model...
Training complete.

VALIDATION RESULTS
Accuracy    : 0.9760 (97.60%)
Macro F1    : 0.9096 (90.96%)
Weighted F1 : 0.9763 (97.63%)

Per-class performance:
                     precision    recall  f1-score   support

     ACCOUNT_ACCESS       0.93      0.90      0.91        29
   ACCOUNT_SECURITY       1.00      0.90      0.95        20
   DEVICE_TECHNICAL       0.81      0.79      0.80        28
    DIGITAL_CONTENT       0.94      0.92      0.93       292
MARKETPLACE_SELLING       0.75      1.00      0.86        15
 ORDER_CANCELLATION       0.71      0.97      0.82        35
     ORDER_DELIVERY       0.99      0.99      0.99      3454
     ORDER_PURCHASE       0.79      0.95      0.86        20
     ORDER_TRACKING       0.87      0.91      0.89        22
    PAYMENT_BILLING       0.98      0.97      0.97       288
  PREORDER_DELIV

In [ ]:
# ============================================================
# BASELINE 2 — FROZEN GOLDEN SET EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Prepare frozen golden queries
# ------------------------------------------------------------

golden_eval = golden_clean.copy()

golden_eval["clean_query"] = (
    golden_eval["customer_query"]
    .fillna("")
    .astype(str)
    .apply(clean_text)
)

X_golden = golden_eval["clean_query"]
y_golden = golden_eval["intent"]

# ------------------------------------------------------------
# 2. Predict
# ------------------------------------------------------------

golden_pred = baseline_model.predict(X_golden)

# ------------------------------------------------------------
# 3. Metrics
# ------------------------------------------------------------

golden_accuracy = accuracy_score(
    y_golden,
    golden_pred
)

golden_macro_f1 = f1_score(
    y_golden,
    golden_pred,
    average="macro",
    zero_division=0
)

golden_weighted_f1 = f1_score(
    y_golden,
    golden_pred,
    average="weighted",
    zero_division=0
)

print("=" * 80)
print("BASELINE 2 — FROZEN GOLDEN SET RESULTS")
print("=" * 80)

print(f"Evaluation cases : {len(y_golden)}")
print(f"Accuracy         : {golden_accuracy:.4f} ({golden_accuracy * 100:.2f}%)")
print(f"Macro F1         : {golden_macro_f1:.4f} ({golden_macro_f1 * 100:.2f}%)")
print(f"Weighted F1      : {golden_weighted_f1:.4f} ({golden_weighted_f1 * 100:.2f}%)")

print("\n" + "=" * 80)
print("PER-CLASS PERFORMANCE")
print("=" * 80)

print(
    classification_report(
        y_golden,
        golden_pred,
        zero_division=0
    )
)

BASELINE 2 — FROZEN GOLDEN SET RESULTS
Evaluation cases : 200
Accuracy         : 0.4350 (43.50%)
Macro F1         : 0.2855 (28.55%)
Weighted F1      : 0.3469 (34.69%)

PER-CLASS PERFORMANCE
                     precision    recall  f1-score   support

     ACCOUNT_ACCESS       1.00      0.11      0.20         9
   ACCOUNT_SECURITY       1.00      0.50      0.67         4
   DEVICE_TECHNICAL       0.00      0.00      0.00         5
    DIGITAL_CONTENT       0.19      0.67      0.30         9
     GENERAL_SOCIAL       0.00      0.00      0.00        17
    GENERAL_SUPPORT       0.00      0.00      0.00        35
MARKETPLACE_SELLING       0.67      0.67      0.67         3
 ORDER_CANCELLATION       0.40      0.50      0.44         4
     ORDER_DELIVERY       0.46      0.86      0.60        69
     ORDER_PURCHASE       0.00      0.00      0.00         1
     ORDER_TRACKING       1.00      0.33      0.50         9
    PAYMENT_BILLING       0.40      0.55      0.46        11
  PREORDER_DELIV

In [ ]:
# ============================================================
# BASELINE 3 — HISTORICAL TF-IDF RETRIEVAL CORPUS
# ============================================================

import pandas as pd
import re

retrieval_df = resolution_df[
    ~resolution_df["case_id"].isin(golden_case_ids)
].copy()

def redact_pii(text):
    text = str(text)

    # URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " [URL] ",
        text,
        flags=re.IGNORECASE
    )

    # Email addresses
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " [EMAIL] ",
        text
    )

    # Phone-like numbers
    text = re.sub(
        r"\b(?:\+?\d[\d\s().-]{7,}\d)\b",
        " [PHONE_NUMBER] ",
        text
    )

    # Long numeric identifiers
    text = re.sub(
        r"\b\d{7,}\b",
        " [ID] ",
        text
    )

    # Twitter handles
    text = re.sub(
        r"@\w+",
        " [USER] ",
        text
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


retrieval_df["retrieval_query"] = (
    retrieval_df["customer_query"]
    .fillna("")
    .apply(redact_pii)
    .apply(clean_text)
)

retrieval_df["retrieval_response"] = (
    retrieval_df["support_response"]
    .fillna("")
    .apply(redact_pii)
)

print("=" * 80)
print("HISTORICAL RETRIEVAL CORPUS")
print("=" * 80)

print(f"Cases: {len(retrieval_df):,}")
print(f"Golden cases excluded: {len(golden_case_ids)}")

display(
    retrieval_df[
        [
            "case_id",
            "retrieval_query",
            "retrieval_response"
        ]
    ].head(10)
)

HISTORICAL RETRIEVAL CORPUS
Cases: 83,218
Golden cases excluded: 200


,case_id,retrieval_query,retrieval_response
0,AMZ-000001,[user] what in the world is a balance withheld? #customerservice doesnt seem to know!,"[USER] Sorry, I'm not quite sure what you're having trouble with. Was there an authorization you were wondering about? ^JY"
1,AMZ-000002,[user] that's exactly what she did. bought it from amazon / posted the review. but not thru your email to her. she posted via website.,"[USER] Sorry, we would be unable to change this as it is an automatic feature done through email.^CD"
2,AMZ-000003,[user] [user] i think u should cancel your order arpit and book another on flipkart. it will deliver ur order in 2-3 working day,"[USER] We would like to help you, Prakhar. Please share your details with us here: [URL] We'll get in touch shortly. Please don't provide your order details as we consider it personal information. Our twitter page is visible to public. -Bikram"
3,AMZ-000004,1/ [user] are you guys leaking email ids of user who applied for amazon bts offer?my bro got this spam email: [url],[USER] Our customers' security is our utmost important concern &amp; we take all necessary steps to safe guard their info. I assure you. ^AN
4,AMZ-000005,the sound quality of the deer hunter on amazon prime on the xbox is the worst thing i've ever heard. [user] [user],[USER] Let us know if you have any other questions or concerns after you've spoke with our Support team! ^ME
5,AMZ-000006,[user] i got a call today 10/19/17 from that same number + [phone_number] is there any information i can provide to you to help you out,"[USER] As this is not one of our numbers, please do not provide anyone from this any personal information. ^JZ"
6,AMZ-000007,[user] [user] you really want to keep my money? i'm waiting for 3 months now!!!!!!! please resolve the case [phone_number],[USER] I'm sorry for any trouble. Are you a seller on Amazon.in or [URL] ^RW
7,AMZ-000008,"[user] and btw i encounter the same issue with my smartphone. so i guess this is not related to the platform or the app, but to the video file and/or subtitle encryption.",[USER] We'd like to get the details so we can investigate further. Will you e-mail us an example of the title (season and episode number) a time frame of when the error occurs and include your device name again: [URL] ^MG
8,AMZ-000009,[user] i already a2z claims 2 times on seller in 23-3-16 but not get any result /refund even today i call but not any reason 😡,[USER] Hi there! kindly refer to the email sent by our buyer guarantee team. I'm sure they must've sent a correspondence. ^HK
9,AMZ-000010,"compré un #ajedrez por [user] .moderno,vino con 2 reinas y un rey. como no me lo solucionan, tendremos que jugar a otra cosa [user]",[USER] Gracias por notificarnos. Nos gustaría reportarlo. ¿Nos puedes dar el link del producto? ^ST


In [ ]:
# ============================================================
# BASELINE 3 — TF-IDF HISTORICAL RETRIEVAL
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Build TF-IDF index over historical cases
# ------------------------------------------------------------

retrieval_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    max_features=100000
)

retrieval_matrix = retrieval_vectorizer.fit_transform(
    retrieval_df["retrieval_query"]
)

print("=" * 80)
print("BASELINE 3 — TF-IDF RETRIEVAL")
print("=" * 80)

print(f"Historical cases : {len(retrieval_df):,}")
print(f"TF-IDF matrix    : {retrieval_matrix.shape}")

BASELINE 3 — TF-IDF RETRIEVAL
Historical cases : 83,218
TF-IDF matrix    : (83218, 80118)


In [ ]:
# ============================================================
# BASELINE 3 — RETRIEVE TOP-K HISTORICAL CASES
# ============================================================

TOP_K = 5

golden_queries = golden_eval["clean_query"].tolist()

golden_query_matrix = retrieval_vectorizer.transform(
    golden_queries
)

similarities = cosine_similarity(
    golden_query_matrix,
    retrieval_matrix
)

retrieval_results = []

for i in range(len(golden_eval)):

    scores = similarities[i]

    top_indices = np.argsort(scores)[::-1][:TOP_K]

    for rank, idx in enumerate(top_indices, start=1):

        row = retrieval_df.iloc[idx]

        retrieval_results.append({
            "golden_id": golden_eval.iloc[i]["golden_id"],
            "case_id": row["case_id"],
            "rank": rank,
            "similarity": float(scores[idx]),
            "golden_intent": golden_eval.iloc[i]["intent"],
            "historical_query": row["retrieval_query"],
            "historical_response": row["retrieval_response"]
        })

retrieval_results_df = pd.DataFrame(retrieval_results)

print("=" * 80)
print("RETRIEVAL COMPLETE")
print("=" * 80)

print(f"Golden queries : {len(golden_eval)}")
print(f"Top-K          : {TOP_K}")
print(f"Retrieved rows : {len(retrieval_results_df):,}")

display(retrieval_results_df.head(15))

RETRIEVAL COMPLETE
Golden queries : 200
Top-K          : 5
Retrieved rows : 1,000


,golden_id,case_id,rank,similarity,golden_intent,historical_query,historical_response
0,GOLD-0001,AMZ-018291,1,0.256098,PAYMENT_BILLING,[user] over and over and over and over and over...and customer service has gotten worse and basically doesn’t care as long as it arrives eventually,[USER] Sorry to hear that Andrea. Are you currently having an issue with a delivery? ^RS
1,GOLD-0001,AMZ-040492,2,0.228657,PAYMENT_BILLING,[user] meanwhile [user] keeps asking me to fill a fucking form over and over and over again!,[USER] I get you're upset! Our team will get back to you at the earliest with an update. ^PR
2,GOLD-0001,AMZ-036572,3,0.204580,PAYMENT_BILLING,[user] they provide me the following information over and over and over and not reply to my concerns about how can be solved.. thx [url],"[USER] We'd like to look into this further, please send us more details here: [URL]"
3,GOLD-0001,AMZ-057467,4,0.201636,PAYMENT_BILLING,"[user] terrible customer service, charges me over and over again for cancelled subscription. awful!","[USER] I'm sorry for the poor experience. Without including any account details, can you tell us more about what's happened? ^ZW"
4,GOLD-0001,AMZ-021962,5,0.200964,PAYMENT_BILLING,[user] havent recieved order contacted customer service still nothing has happened plz advice what to do,"[USER] So sorry to hear this! When you contacted us, what information and options were provided for the order? Keep us updated! ^TH"
5,GOLD-0002,AMZ-043328,1,0.274645,ORDER_DELIVERY,[user] you've missed the delivery date (today) and haven't even dispatched the item yet.,"[USER] I'm so sorry for the trouble. If you have a moment, please contact us: [URL] so we can look into this. ^LH"
6,GOLD-0002,AMZ-077989,2,0.245248,ORDER_DELIVERY,[user] yes it has missed the delivery date!,[USER] Were you given an updated delivery date or a notice of delay? We'd like to help! ^ZW
7,GOLD-0002,AMZ-073890,3,0.244491,ORDER_DELIVERY,[user] yup missed the delivery date (today),"[USER] Thank you for confirming. We'd like an opportunity to look into this more closely, please contact us here: [URL] ^MH"
8,GOLD-0002,AMZ-038883,4,0.238403,ORDER_DELIVERY,"[user] yes, you’ve missed the “customer experience” part here. [user]","[USER] Sorry to know that, please let us know more about what went wrong? We'll definitely look into it. ^PS"
9,GOLD-0002,AMZ-055508,5,0.231983,ORDER_DELIVERY,"[user] yes, you've missed the delivery date on numerous orders lately. i'm not going anywhere, just sucks to pay for prime for nothing",[USER] That's certainly not the experience we want for our customers and we want to help! I'd like to have a member of our team look into this with you. Please reach out to us here: [URL] ^NS


In [ ]:
# ============================================================
# RETRIEVAL PROXY — INTENT AGREEMENT
# ============================================================

weak_intent_map = (
    intent_train_df[
        intent_train_df["weak_intent_v2"].notna()
    ][
        ["case_id", "weak_intent_v2"]
    ]
    .drop_duplicates("case_id")
    .set_index("case_id")["weak_intent_v2"]
)

retrieval_results_df["historical_weak_intent"] = (
    retrieval_results_df["case_id"].map(weak_intent_map)
)

retrieval_results_df["intent_match"] = (
    retrieval_results_df["golden_intent"]
    == retrieval_results_df["historical_weak_intent"]
)

print("=" * 80)
print("TF-IDF RETRIEVAL — INTENT AGREEMENT PROXY")
print("=" * 80)

for k in [1, 3, 5]:

    top_k = (
        retrieval_results_df[
            retrieval_results_df["rank"] <= k
        ]
        .groupby("golden_id")["intent_match"]
        .max()
    )

    print(
        f"Recall@{k}: "
        f"{top_k.mean() * 100:.2f}%"
    )

TF-IDF RETRIEVAL — INTENT AGREEMENT PROXY
Recall@1: 16.00%
Recall@3: 25.00%
Recall@5: 31.00%


In [ ]:
# ============================================================
# BASELINE 4 — BM25
# ============================================================

!pip -q install rank-bm25

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np
import pandas as pd
import re

# ============================================================
# BUILD BM25 INDEX
# ============================================================

bm25_corpus = (
    retrieval_df["retrieval_query"]
    .fillna("")
    .astype(str)
    .apply(lambda x: x.split())
    .tolist()
)

bm25 = BM25Okapi(bm25_corpus)

print("=" * 80)
print("BASELINE 4 — BM25 INDEX")
print("=" * 80)

print(f"Historical cases : {len(bm25_corpus):,}")
print("BM25 index built successfully.")

BASELINE 4 — BM25 INDEX
Historical cases : 83,218
BM25 index built successfully.


In [ ]:
# ============================================================
# BM25 RETRIEVAL
# ============================================================

TOP_K = 5

bm25_results = []

for i, query in enumerate(
    golden_eval["clean_query"]
):

    query_tokens = query.split()

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:TOP_K]

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        row = retrieval_df.iloc[idx]

        bm25_results.append({
            "golden_id":
                golden_eval.iloc[i]["golden_id"],

            "case_id":
                row["case_id"],

            "rank":
                rank,

            "bm25_score":
                float(scores[idx]),

            "golden_intent":
                golden_eval.iloc[i]["intent"],

            "historical_query":
                row["retrieval_query"],

            "historical_response":
                row["retrieval_response"]
        })

bm25_results_df = pd.DataFrame(
    bm25_results
)

print("=" * 80)
print("BM25 RETRIEVAL COMPLETE")
print("=" * 80)

print(f"Golden queries : {len(golden_eval)}")
print(f"Top-K          : {TOP_K}")
print(f"Retrieved rows : {len(bm25_results_df):,}")

display(
    bm25_results_df.head(15)
)

BM25 RETRIEVAL COMPLETE
Golden queries : 200
Top-K          : 5
Retrieved rows : 1,000


,golden_id,case_id,rank,bm25_score,golden_intent,historical_query,historical_response
0,GOLD-0001,AMZ-072041,1,20.801930,PAYMENT_BILLING,"[user] i know that she talked to customer service, who told her it was an issue with the bank, who told her it was an issue with amazon.","[USER] Thank you for confirming! We'd like to take another look into this. When your wife has the chance, please have her contact us through the link previously given. ^HC"
1,GOLD-0001,AMZ-066674,2,20.652361,PAYMENT_BILLING,[user] what's the best email address to contact you on regarding an order please,[USER] II understand you want to get in touch with us. You may contact us from here: [URL] and we'll assist you accordingly. ^VH
2,GOLD-0001,AMZ-068872,3,20.117014,PAYMENT_BILLING,"[user] order never showed, am now speaking to an incompetent amazon employee over chat. send help","[USER] I'm sorry for the trouble you've had with your order! Though we're unable to view order specifics, can you please confirm what the last tracking status shows, as well as which carrier is associated with your shipment: [URL] ^ST"
3,GOLD-0001,AMZ-008317,4,20.095090,PAYMENT_BILLING,"dear [user] customer service, over 30 min on phone trying to get an address added to my account, so purchased gift from elsewhere online.",[USER] I'm very sorry for the poor experience! I hope we can provide a better experience in the future. ^WT
4,GOLD-0001,AMZ-014410,5,20.048534,PAYMENT_BILLING,"[user] thank you, i've contacted customer service who are going to contact the courier.",[USER] Thanks for keeping us posted! ^AT
5,GOLD-0002,AMZ-067343,1,22.859114,ORDER_DELIVERY,[user] you guys missed todays guaranteed delivery date and now its expected 25th-28th. the item wasn't needed today but it's the principle of sticking to something if it's guaranteed,"[USER] Thank you for clarifying. I can understand how frustrating this must be, and I'm sorry we've let you down. If it hasn't arrived by the new delivery date given to you, or for direct assistance, please reach us here: [URL] ^DW"
6,GOLD-0002,AMZ-029047,2,21.627253,ORDER_DELIVERY,"[user] yes, about twice i’ve missed a delivery date which changed at order confirmation however 90+% of the time it’s missed targeted deliveries.",[USER] This is not the experience we want for our customers! Please contact us to provide carrier feedback [URL] ^NN
7,GOLD-0002,AMZ-073890,3,20.371683,ORDER_DELIVERY,[user] yup missed the delivery date (today),"[USER] Thank you for confirming. We'd like an opportunity to look into this more closely, please contact us here: [URL] ^MH"
8,GOLD-0002,AMZ-081847,4,20.138757,ORDER_DELIVERY,[user] yes. today has been missed and several other orders over the last few months have had issues. i pay for prime for the convenience of next day delivery but more frequently the level of service has been inconsistent.,[USER] We'd like to look into the delivery delays for you. Please contact us here: [URL] ^LI
9,GOLD-0002,AMZ-012333,5,19.944996,ORDER_DELIVERY,[user] how do i pre order something months ago.. it releases and mine hasn’t shipped for 3 days after release?,[USER] I'm sorry that your order is delayed. Contact us via the link so we can go over options with you. [URL] ^EM


In [ ]:
# ============================================================
# BM25 — INTENT AGREEMENT PROXY
# ============================================================

bm25_results_df["historical_weak_intent"] = (
    bm25_results_df["case_id"]
    .map(weak_intent_map)
)

bm25_results_df["intent_match"] = (
    bm25_results_df["golden_intent"]
    ==
    bm25_results_df["historical_weak_intent"]
)

print("=" * 80)
print("BM25 RETRIEVAL — INTENT AGREEMENT PROXY")
print("=" * 80)

for k in [1, 3, 5]:

    top_k = (
        bm25_results_df[
            bm25_results_df["rank"] <= k
        ]
        .groupby("golden_id")["intent_match"]
        .max()
    )

    print(
        f"Recall@{k}: "
        f"{top_k.mean() * 100:.2f}%"
    )

BM25 RETRIEVAL — INTENT AGREEMENT PROXY
Recall@1: 17.00%
Recall@3: 29.00%
Recall@5: 32.50%


In [ ]:
# ============================================================
# BASELINE 5 — VECTOR RETRIEVAL
# ============================================================

!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

print("=" * 80)
print("VECTOR RETRIEVAL ENVIRONMENT")
print("=" * 80)

print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

VECTOR RETRIEVAL ENVIRONMENT
PyTorch version: 2.11.0+cu128
CUDA available : True
GPU: Tesla T4


In [ ]:
# ============================================================
# LOAD BGE LARGE V1.5
# ============================================================

embedding_model = SentenceTransformer(
    "BAAI/bge-large-en-v1.5"
)

print("=" * 80)
print("BGE MODEL LOADED")
print("=" * 80)

print("Model:", "BAAI/bge-large-en-v1.5")
print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

BGE MODEL LOADED
Model: BAAI/bge-large-en-v1.5
Embedding dimension: 1024


/tmp/ipykernel_2236/981421942.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [ ]:
# ============================================================
# EMBEDDING SANITY CHECK
# ============================================================

test_texts = retrieval_df[
    "retrieval_query"
].head(10).tolist()

test_embeddings = embedding_model.encode(
    test_texts,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("=" * 80)
print("EMBEDDING SANITY CHECK")
print("=" * 80)

print("Input texts :", len(test_texts))
print("Embedding shape:", test_embeddings.shape)
print("Data type:", test_embeddings.dtype)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

EMBEDDING SANITY CHECK
Input texts : 10
Embedding shape: (10, 1024)
Data type: float32


In [ ]:
# ============================================================
# BGE-LARGE — EMBED HISTORICAL RETRIEVAL CORPUS
# ============================================================

import numpy as np
import os

EMBEDDING_PATH = "data/embeddings/historical_bge_large.npy"

os.makedirs(
    "data/embeddings",
    exist_ok=True
)

historical_texts = (
    retrieval_df["retrieval_query"]
    .fillna("")
    .astype(str)
    .tolist()
)

print("=" * 80)
print("ENCODING HISTORICAL CASES")
print("=" * 80)

print(f"Cases to encode: {len(historical_texts):,}")

historical_embeddings = embedding_model.encode(
    historical_texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("\nEncoding complete.")
print("Embedding shape:", historical_embeddings.shape)
print("Data type:", historical_embeddings.dtype)

np.save(
    EMBEDDING_PATH,
    historical_embeddings
)

print(f"Saved to: {EMBEDDING_PATH}")

ENCODING HISTORICAL CASES
Cases to encode: 83,218


Batches:   0%|          | 0/2601 [00:00<?, ?it/s]


Encoding complete.
Embedding shape: (83218, 1024)
Data type: float32
Saved to: data/embeddings/historical_bge_large.npy


In [ ]:
# ============================================================
# EMBED GOLDEN EVALUATION QUERIES
# ============================================================

golden_embedding = embedding_model.encode(
    golden_eval["clean_query"].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("=" * 80)
print("GOLDEN QUERY EMBEDDINGS")
print("=" * 80)

print("Shape:", golden_embedding.shape)
print("Data type:", golden_embedding.dtype)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

GOLDEN QUERY EMBEDDINGS
Shape: (200, 1024)
Data type: float32


In [ ]:
# ============================================================
# BASELINE 5 — VECTOR RETRIEVAL
# ============================================================

TOP_K = 5

vector_scores = (
    golden_embedding
    @ historical_embeddings.T
)

vector_results = []

for i in range(len(golden_eval)):

    scores = vector_scores[i]

    top_indices = np.argsort(
        scores
    )[::-1][:TOP_K]

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        row = retrieval_df.iloc[idx]

        vector_results.append({
            "golden_id":
                golden_eval.iloc[i]["golden_id"],

            "case_id":
                row["case_id"],

            "rank":
                rank,

            "vector_score":
                float(scores[idx]),

            "golden_intent":
                golden_eval.iloc[i]["intent"],

            "historical_query":
                row["retrieval_query"],

            "historical_response":
                row["retrieval_response"]
        })

vector_results_df = pd.DataFrame(
    vector_results
)

print("=" * 80)
print("VECTOR RETRIEVAL COMPLETE")
print("=" * 80)

print(f"Golden queries : {len(golden_eval)}")
print(f"Top-K          : {TOP_K}")
print(f"Retrieved rows : {len(vector_results_df):,}")

display(
    vector_results_df.head(15)
)

VECTOR RETRIEVAL COMPLETE
Golden queries : 200
Top-K          : 5
Retrieved rows : 1,000


,golden_id,case_id,rank,vector_score,golden_intent,historical_query,historical_response
0,GOLD-0001,AMZ-000377,1,0.826847,PAYMENT_BILLING,[user] can someone at [user] [user] please help me get my order? your customer service is awful. what has happened? are you outsourcing it?,"[USER] I'm very sorry to hear about the trouble! When you contacted us, what information or options were we able to provide? ^HC"
1,GOLD-0001,AMZ-058158,2,0.826654,PAYMENT_BILLING,[user] help! my order is messed up and your customer service person is clueless. and she disconnected me,"[USER] We'd like to help! Without providing personal or account information, can you give us a little more details on the current issue? ^AG"
2,GOLD-0001,AMZ-058371,3,0.822998,PAYMENT_BILLING,[user] it is current order. [phone_number] &amp; [phone_number] very bad experience with your company,"[USER] Apologies for the delay in the delivery. Kindly report this to our support team here: [URL] &amp; we'll have this checked. Also, please don't provide your order details, as we consider it to be personal information. Our Twitter page is public.​ ^EM"
3,GOLD-0001,AMZ-047115,4,0.822032,PAYMENT_BILLING,absolutely livid with [user] !!! never had a problem with you before until today! taking two payments out my account for the same item and then me having to deal with 2 thick ass bitches on the phone what are meant to be trained in customer service! shocking,[USER] Truly sorry! Have you confirmed with your bank you're not seeing an authorisation? Learn more: [URL] ^JE
4,GOLD-0001,AMZ-032653,5,0.819391,PAYMENT_BILLING,probably having the worst customer experience of my little life with [user],"[USER] I'm sorry for the poor experience! Without giving any personal/account info, can you tell us more about the problem? ^SJ"
5,GOLD-0002,AMZ-082579,1,0.866867,ORDER_DELIVERY,"[user] yet another order just missed the ""guaranteed"" delivery date due to usps. maybe show me when i check out that you'll be using usps so i can buy it elsewhere and actually get my package on time.","[USER] I'm so sorry about this, Jake! Have you had a chance to reach out to us via phone or chat regarding this matter? If so, what options or insight were provided to you? ^JY"
6,GOLD-0002,AMZ-023939,2,0.866252,ORDER_DELIVERY,"absolutely, you have missed the delivery date given at checkout. [url]",[USER] Thanks for the info. What does the current tracking state here: [URL] ^GG
7,GOLD-0002,AMZ-083539,3,0.856913,ORDER_DELIVERY,[user] you have missed the delivery date for my last 3 packages. i have one now saying delivering today but it hasn’t even shipped.,"[USER] Apologies both for missing the mark and letting you down. We always want the best experience for our customers. Please keep us posted on delivery! If anything changes with your order, you can review communications here: [URL] ^FD"
8,GOLD-0002,AMZ-072005,4,0.844339,ORDER_DELIVERY,[user] yes the delivery missed the delivery date in the email and now says this. [url],"[USER] I'm sorry to see this! Late deliveries typically arrive the next day. For more options, please reach out to us here: [URL] ^BH"
9,GOLD-0002,AMZ-082467,5,0.843584,ORDER_DELIVERY,"hey, great job, [user] . you have my amazon order somewhere, it’s already 2 days past the delivery date, and now you change it from “it’s out on the truck” to “eh, maybe wednesday?” so, 6 days late (at best) and you don’t know where the item is. you’re garbage at what you do.","[USER] Uh no, I am so sorry your package as been delayed! We want to make sure you receive your package! Please let us know if you don't receive your package by Wednesday at 8pm. ^LS"


In [ ]:
# ============================================================
# VECTOR RETRIEVAL — INTENT AGREEMENT PROXY
# ============================================================

vector_results_df["historical_weak_intent"] = (
    vector_results_df["case_id"]
    .map(weak_intent_map)
)

vector_results_df["intent_match"] = (
    vector_results_df["golden_intent"]
    ==
    vector_results_df["historical_weak_intent"]
)

print("=" * 80)
print("VECTOR RETRIEVAL — INTENT AGREEMENT PROXY")
print("=" * 80)

for k in [1, 3, 5]:

    top_k = (
        vector_results_df[
            vector_results_df["rank"] <= k
        ]
        .groupby("golden_id")["intent_match"]
        .max()
    )

    print(
        f"Recall@{k}: "
        f"{top_k.mean() * 100:.2f}%"
    )

VECTOR RETRIEVAL — INTENT AGREEMENT PROXY
Recall@1: 21.50%
Recall@3: 32.00%
Recall@5: 36.50%


In [ ]:
# ============================================================
# BASELINE 6 — GENERATE BM25 + BGE CANDIDATES
# ============================================================

RRF_CANDIDATES = 50

rrf_results = []

for i, golden_row in golden_eval.iterrows():

    golden_id = golden_row["golden_id"]

    # -----------------------------
    # BM25 candidates
    # -----------------------------

    query = golden_row["clean_query"]
    query_tokens = query.split()

    bm25_scores = bm25.get_scores(query_tokens)

    bm25_top = np.argsort(
        bm25_scores
    )[::-1][:RRF_CANDIDATES]

    # -----------------------------
    # BGE candidates
    # -----------------------------

    bge_scores = vector_scores[i]

    bge_top = np.argsort(
        bge_scores
    )[::-1][:RRF_CANDIDATES]

    # Store candidate ranks
    bm25_rank = {
        int(idx): rank
        for rank, idx in enumerate(
            bm25_top,
            start=1
        )
    }

    bge_rank = {
        int(idx): rank
        for rank, idx in enumerate(
            bge_top,
            start=1
        )
    }

    # Union of candidates
    candidate_indices = set(
        bm25_rank.keys()
    ) | set(
        bge_rank.keys()
    )

    for idx in candidate_indices:

        row = retrieval_df.iloc[idx]

        rrf_results.append({
            "golden_id": golden_id,
            "historical_index": idx,
            "case_id": row["case_id"],
            "bm25_rank": bm25_rank.get(idx),
            "bge_rank": bge_rank.get(idx),
            "bm25_score": float(bm25_scores[idx]),
            "bge_score": float(bge_scores[idx])
        })

rrf_candidates_df = pd.DataFrame(
    rrf_results
)

print("=" * 80)
print("RRF CANDIDATE GENERATION")
print("=" * 80)

print(
    f"Golden queries: {len(golden_eval)}"
)

print(
    f"Candidate rows: "
    f"{len(rrf_candidates_df):,}"
)

RRF CANDIDATE GENERATION
Golden queries: 200
Candidate rows: 19,047


In [ ]:
# ============================================================
# CALCULATE RRF SCORES
# ============================================================

RRF_K = 60

def rrf_score(row):

    score = 0.0

    if pd.notna(row["bm25_rank"]):
        score += 1.0 / (
            RRF_K + row["bm25_rank"]
        )

    if pd.notna(row["bge_rank"]):
        score += 1.0 / (
            RRF_K + row["bge_rank"]
        )

    return score


rrf_candidates_df["rrf_score"] = (
    rrf_candidates_df
    .apply(rrf_score, axis=1)
)

# Rank candidates within each golden query
rrf_candidates_df["rrf_rank"] = (
    rrf_candidates_df
    .groupby("golden_id")["rrf_score"]
    .rank(
        method="first",
        ascending=False
    )
)

print("=" * 80)
print("RRF SCORING COMPLETE")
print("=" * 80)

display(
    rrf_candidates_df
    .sort_values(
        ["golden_id", "rrf_rank"]
    )
    .head(20)
)

RRF SCORING COMPLETE


,golden_id,historical_index,case_id,bm25_rank,bge_rank,bm25_score,bge_score,rrf_score,rrf_rank
44,GOLD-0001,45429,AMZ-045652,6.0,31.0,19.204027,0.803154,0.026141,1.0
27,GOLD-0001,71745,AMZ-072041,1.0,NaN,20.801930,0.738853,0.016393,2.0
48,GOLD-0001,375,AMZ-000377,NaN,1.0,2.752309,0.826847,0.016393,3.0
24,GOLD-0001,57906,AMZ-058158,NaN,2.0,5.161147,0.826654,0.016129,4.0
39,GOLD-0001,66403,AMZ-066674,2.0,NaN,20.652361,0.670483,0.016129,5.0
3,GOLD-0001,58118,AMZ-058371,NaN,3.0,5.602591,0.822998,0.015873,6.0
91,GOLD-0001,68593,AMZ-068872,3.0,NaN,20.117014,0.745448,0.015873,7.0
21,GOLD-0001,46888,AMZ-047115,NaN,4.0,4.602294,0.822032,0.015625,8.0
33,GOLD-0001,8275,AMZ-008317,4.0,NaN,20.095090,0.753073,0.015625,9.0
77,GOLD-0001,32462,AMZ-032653,NaN,5.0,3.706796,0.819391,0.015385,10.0


In [ ]:
# ============================================================
# RRF — INTENT AGREEMENT PROXY
# ============================================================

rrf_candidates_df["historical_weak_intent"] = (
    rrf_candidates_df["case_id"]
    .map(weak_intent_map)
)

rrf_candidates_df["intent_match"] = (
    rrf_candidates_df["golden_id"]
    .map(
        golden_eval.set_index("golden_id")["intent"]
    )
    ==
    rrf_candidates_df["historical_weak_intent"]
)

print("=" * 80)
print("RRF RETRIEVAL — INTENT AGREEMENT PROXY")
print("=" * 80)

for k in [1, 3, 5, 10]:

    top_k = (
        rrf_candidates_df[
            rrf_candidates_df["rrf_rank"] <= k
        ]
        .groupby("golden_id")["intent_match"]
        .max()
    )

    print(
        f"Recall@{k}: "
        f"{top_k.mean() * 100:.2f}%"
    )

RRF RETRIEVAL — INTENT AGREEMENT PROXY
Recall@1: 22.00%
Recall@3: 33.00%
Recall@5: 38.00%
Recall@10: 42.50%


In [ ]:
# ============================================================
# CROSS-ENCODER RERANKER
# ============================================================

!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

print("=" * 80)
print("CROSS-ENCODER LOADED")
print("=" * 80)

print("Model: BAAI/bge-reranker-base")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

CROSS-ENCODER LOADED
Model: BAAI/bge-reranker-base


In [ ]:
# ============================================================
# CROSS-ENCODER RERANKING
# ============================================================

RERANK_CANDIDATES = 50

reranked_results = []

for golden_id in golden_eval["golden_id"]:

    candidates = (
        rrf_candidates_df[
            rrf_candidates_df["golden_id"] == golden_id
        ]
        .sort_values("rrf_rank")
        .head(RERANK_CANDIDATES)
        .copy()
    )

    if len(candidates) == 0:
        continue

    # --------------------------------------------------------
    # Build query-document pairs
    # --------------------------------------------------------

    golden_query = golden_eval.loc[
        golden_eval["golden_id"] == golden_id,
        "clean_query"
    ].iloc[0]

    pairs = [
        [
            golden_query,
            retrieval_df.loc[
                retrieval_df["case_id"] == case_id,
                "retrieval_query"
            ].iloc[0]
        ]
        for case_id in candidates["case_id"]
    ]

    # --------------------------------------------------------
    # Cross-encoder scores
    # --------------------------------------------------------

    ce_scores = reranker.predict(
        pairs,
        show_progress_bar=False
    )

    candidates["ce_score"] = ce_scores

    # --------------------------------------------------------
    # Rerank
    # --------------------------------------------------------

    candidates = candidates.sort_values(
        "ce_score",
        ascending=False
    ).reset_index(drop=True)

    candidates["ce_rank"] = (
        np.arange(len(candidates)) + 1
    )

    reranked_results.append(
        candidates[
            [
                "golden_id",
                "case_id",
                "ce_rank",
                "ce_score",
                "rrf_rank",
                "rrf_score",
                "bm25_rank",
                "bge_rank"
            ]
        ]
    )

reranked_df = pd.concat(
    reranked_results,
    ignore_index=True
)

print("=" * 80)
print("CROSS-ENCODER RERANKING COMPLETE")
print("=" * 80)

print(
    f"Golden queries: "
    f"{reranked_df['golden_id'].nunique()}"
)

print(
    f"Reranked candidates: "
    f"{len(reranked_df):,}"
)

display(
    reranked_df.head(20)
)

CROSS-ENCODER RERANKING COMPLETE
Golden queries: 200
Reranked candidates: 10,000


,golden_id,case_id,ce_rank,ce_score,rrf_rank,rrf_score,bm25_rank,bge_rank
0,GOLD-0001,AMZ-073280,1,0.615286,23.0,0.013889,12.0,NaN
1,GOLD-0001,AMZ-057467,2,0.249812,13.0,0.014925,7.0,NaN
2,GOLD-0001,AMZ-058158,3,0.167556,4.0,0.016129,NaN,2.0
3,GOLD-0001,AMZ-000377,4,0.125092,3.0,0.016393,NaN,1.0
4,GOLD-0001,AMZ-044915,5,0.115635,30.0,0.013333,15.0,NaN
5,GOLD-0001,AMZ-045652,6,0.055511,1.0,0.026141,6.0,31.0
6,GOLD-0001,AMZ-006003,7,0.048986,50.0,0.011765,NaN,25.0
7,GOLD-0001,AMZ-063640,8,0.043576,44.0,0.012195,22.0,NaN
8,GOLD-0001,AMZ-038728,9,0.041852,41.0,0.012346,21.0,NaN
9,GOLD-0001,AMZ-073638,10,0.040314,34.0,0.012987,NaN,17.0


In [ ]:
# ============================================================
# CROSS-ENCODER — INTENT AGREEMENT PROXY
# ============================================================

reranked_df["historical_weak_intent"] = (
    reranked_df["case_id"]
    .map(weak_intent_map)
)

reranked_df["golden_intent"] = (
    reranked_df["golden_id"]
    .map(
        golden_eval.set_index("golden_id")["intent"]
    )
)

reranked_df["intent_match"] = (
    reranked_df["golden_intent"]
    ==
    reranked_df["historical_weak_intent"]
)

print("=" * 80)
print("CROSS-ENCODER RETRIEVAL — INTENT AGREEMENT")
print("=" * 80)

for k in [1, 3, 5, 10]:

    top_k = (
        reranked_df[
            reranked_df["ce_rank"] <= k
        ]
        .groupby("golden_id")["intent_match"]
        .max()
    )

    print(
        f"Recall@{k}: "
        f"{top_k.mean() * 100:.2f}%"
    )

CROSS-ENCODER RETRIEVAL — INTENT AGREEMENT
Recall@1: 25.00%
Recall@3: 35.00%
Recall@5: 39.00%
Recall@10: 44.00%


In [ ]:
# ============================================================
# STRUCTURED HISTORICAL RESOLUTION CORPUS
# ============================================================

rag_df = retrieval_df.copy()

rag_df["customer_problem"] = (
    rag_df["retrieval_query"]
)

rag_df["support_action"] = (
    rag_df["retrieval_response"]
)

# ------------------------------------------------------------
# Initial resolution type
# ------------------------------------------------------------

def infer_resolution_type(response):
    text = str(response).lower()

    if any(x in text for x in [
        "refund",
        "money back",
        "reimburse"
    ]):
        return "REFUND_OR_COMPENSATION"

    if any(x in text for x in [
        "contact us",
        "reach us",
        "send us",
        "provide",
        "details",
        "information"
    ]):
        return "INVESTIGATION_OR_FOLLOWUP"

    if any(x in text for x in [
        "check",
        "look into",
        "investigate",
        "forwarded"
    ]):
        return "INVESTIGATION_OR_FOLLOWUP"

    if any(x in text for x in [
        "cancel"
    ]):
        return "CANCELLATION"

    if any(x in text for x in [
        "replace",
        "replacement"
    ]):
        return "REPLACEMENT"

    return "GENERAL_SUPPORT"


rag_df["resolution_type"] = (
    rag_df["support_action"]
    .apply(infer_resolution_type)
)

# ------------------------------------------------------------
# Evidence quality
# ------------------------------------------------------------

def evidence_quality(row):
    query = str(row["customer_problem"]).strip()
    response = str(row["support_action"]).strip()

    if len(query) < 15 or len(response) < 15:
        return "LOW"

    if (
        "[URL]" in response
        or "contact" in response.lower()
        or "provide" in response.lower()
        or "check" in response.lower()
        or "look into" in response.lower()
    ):
        return "MEDIUM"

    return "HIGH"


rag_df["evidence_quality"] = (
    rag_df.apply(
        evidence_quality,
        axis=1
    )
)

# ------------------------------------------------------------
# Source metadata
# ------------------------------------------------------------

rag_df["source"] = "AmazonHelp historical support case"

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

rag_corpus = rag_df[
    [
        "case_id",
        "customer_problem",
        "support_action",
        "resolution_type",
        "evidence_quality",
        "source"
    ]
].copy()

print("=" * 80)
print("STRUCTURED RAG CORPUS")
print("=" * 80)

print(f"Documents: {len(rag_corpus):,}")

display(
    rag_corpus.head(10)
)

STRUCTURED RAG CORPUS
Documents: 83,218


,case_id,customer_problem,support_action,resolution_type,evidence_quality,source
0,AMZ-000001,[user] what in the world is a balance withheld? #customerservice doesnt seem to know!,"[USER] Sorry, I'm not quite sure what you're having trouble with. Was there an authorization you were wondering about? ^JY",GENERAL_SUPPORT,HIGH,AmazonHelp historical support case
1,AMZ-000002,[user] that's exactly what she did. bought it from amazon / posted the review. but not thru your email to her. she posted via website.,"[USER] Sorry, we would be unable to change this as it is an automatic feature done through email.^CD",GENERAL_SUPPORT,HIGH,AmazonHelp historical support case
2,AMZ-000003,[user] [user] i think u should cancel your order arpit and book another on flipkart. it will deliver ur order in 2-3 working day,"[USER] We would like to help you, Prakhar. Please share your details with us here: [URL] We'll get in touch shortly. Please don't provide your order details as we consider it personal information. Our twitter page is visible to public. -Bikram",INVESTIGATION_OR_FOLLOWUP,MEDIUM,AmazonHelp historical support case
3,AMZ-000004,1/ [user] are you guys leaking email ids of user who applied for amazon bts offer?my bro got this spam email: [url],[USER] Our customers' security is our utmost important concern &amp; we take all necessary steps to safe guard their info. I assure you. ^AN,GENERAL_SUPPORT,HIGH,AmazonHelp historical support case
4,AMZ-000005,the sound quality of the deer hunter on amazon prime on the xbox is the worst thing i've ever heard. [user] [user],[USER] Let us know if you have any other questions or concerns after you've spoke with our Support team! ^ME,GENERAL_SUPPORT,HIGH,AmazonHelp historical support case
5,AMZ-000006,[user] i got a call today 10/19/17 from that same number + [phone_number] is there any information i can provide to you to help you out,"[USER] As this is not one of our numbers, please do not provide anyone from this any personal information. ^JZ",INVESTIGATION_OR_FOLLOWUP,MEDIUM,AmazonHelp historical support case
6,AMZ-000007,[user] [user] you really want to keep my money? i'm waiting for 3 months now!!!!!!! please resolve the case [phone_number],[USER] I'm sorry for any trouble. Are you a seller on Amazon.in or [URL] ^RW,GENERAL_SUPPORT,MEDIUM,AmazonHelp historical support case
7,AMZ-000008,"[user] and btw i encounter the same issue with my smartphone. so i guess this is not related to the platform or the app, but to the video file and/or subtitle encryption.",[USER] We'd like to get the details so we can investigate further. Will you e-mail us an example of the title (season and episode number) a time frame of when the error occurs and include your device name again: [URL] ^MG,INVESTIGATION_OR_FOLLOWUP,MEDIUM,AmazonHelp historical support case
8,AMZ-000009,[user] i already a2z claims 2 times on seller in 23-3-16 but not get any result /refund even today i call but not any reason 😡,[USER] Hi there! kindly refer to the email sent by our buyer guarantee team. I'm sure they must've sent a correspondence. ^HK,GENERAL_SUPPORT,HIGH,AmazonHelp historical support case
9,AMZ-000010,"compré un #ajedrez por [user] .moderno,vino con 2 reinas y un rey. como no me lo solucionan, tendremos que jugar a otra cosa [user]",[USER] Gracias por notificarnos. Nos gustaría reportarlo. ¿Nos puedes dar el link del producto? ^ST,GENERAL_SUPPORT,HIGH,AmazonHelp historical support case


In [ ]:
# ============================================================
# TOP RERANKED EVIDENCE
# ============================================================

TOP_EVIDENCE = 5

top_evidence = (
    reranked_df[
        reranked_df["ce_rank"] <= TOP_EVIDENCE
    ]
    .copy()
)

# Join structured RAG metadata
evidence_df = top_evidence.merge(
    rag_corpus,
    on="case_id",
    how="left",
    suffixes=("", "_rag")
)

# Add golden query + authoritative labels
golden_meta = golden_eval[
    [
        "golden_id",
        "customer_query",
        "intent",
        "resolution_status",
        "risk_level",
        "automation_decision"
    ]
].copy()

evidence_df = evidence_df.merge(
    golden_meta,
    on="golden_id",
    how="left"
)

evidence_df = evidence_df.sort_values(
    ["golden_id", "ce_rank"]
)

print("=" * 80)
print("TOP HISTORICAL EVIDENCE")
print("=" * 80)

print(
    f"Golden cases: "
    f"{evidence_df['golden_id'].nunique()}"
)

print(
    f"Evidence rows: "
    f"{len(evidence_df):,}"
)

display(
    evidence_df[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_rank",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type",
            "evidence_quality"
        ]
    ].head(15)
)

TOP HISTORICAL EVIDENCE
Golden cases: 200
Evidence rows: 1,000


,golden_id,customer_query,intent,case_id,ce_rank,ce_score,customer_problem,support_action,resolution_type,evidence_quality
0,GOLD-0001,"Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!",PAYMENT_BILLING,AMZ-073280,1,0.615286,[user] will someone please contact mw urgently regarding an order i made?,"[USER] Oh no! Sorry to hear this! Without providing any account specific information, will you tell us what's going on? We are here to help! ^AD",INVESTIGATION_OR_FOLLOWUP,HIGH
1,GOLD-0001,"Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!",PAYMENT_BILLING,AMZ-057467,2,0.249812,"[user] terrible customer service, charges me over and over again for cancelled subscription. awful!","[USER] I'm sorry for the poor experience. Without including any account details, can you tell us more about what's happened? ^ZW",INVESTIGATION_OR_FOLLOWUP,HIGH
2,GOLD-0001,"Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!",PAYMENT_BILLING,AMZ-058158,3,0.167556,[user] help! my order is messed up and your customer service person is clueless. and she disconnected me,"[USER] We'd like to help! Without providing personal or account information, can you give us a little more details on the current issue? ^AG",INVESTIGATION_OR_FOLLOWUP,HIGH
3,GOLD-0001,"Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!",PAYMENT_BILLING,AMZ-000377,4,0.125092,[user] can someone at [user] [user] please help me get my order? your customer service is awful. what has happened? are you outsourcing it?,"[USER] I'm very sorry to hear about the trouble! When you contacted us, what information or options were we able to provide? ^HC",INVESTIGATION_OR_FOLLOWUP,MEDIUM
4,GOLD-0001,"Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!",PAYMENT_BILLING,AMZ-044915,5,0.115635,[user] i issues with placing an order or wrongly placed an order. pls contact me imdtly.,[USER] We'd like to help. Kindly get in touch with us here: [URL] ^AK,GENERAL_SUPPORT,MEDIUM
5,GOLD-0002,"@AmazonHelp LOL...you've missed the delivery date of every order of mine shipped via usps, for the last 5 months including todays order.",ORDER_DELIVERY,AMZ-032817,1,0.998420,"[user] yes, u missed the provided delivery date",[USER] Thanks for confirming! Who's the carrier listed for your order? What's the most current tracking information state? ^HC,INVESTIGATION_OR_FOLLOWUP,HIGH
6,GOLD-0002,"@AmazonHelp LOL...you've missed the delivery date of every order of mine shipped via usps, for the last 5 months including todays order.",ORDER_DELIVERY,AMZ-023939,2,0.997259,"absolutely, you have missed the delivery date given at checkout. [url]",[USER] Thanks for the info. What does the current tracking state here: [URL] ^GG,GENERAL_SUPPORT,MEDIUM
7,GOLD-0002,"@AmazonHelp LOL...you've missed the delivery date of every order of mine shipped via usps, for the last 5 months including todays order.",ORDER_DELIVERY,AMZ-077989,3,0.995812,[user] yes it has missed the delivery date!,[USER] Were you given an updated delivery date or a notice of delay? We'd like to help! ^ZW,GENERAL_SUPPORT,HIGH
8,GOLD-0002,"@AmazonHelp LOL...you've missed the delivery date of every order of mine shipped via usps, for the last 5 months including todays order.",ORDER_DELIVERY,AMZ-081847,4,0.994597,[user] yes. today has been missed and several other orders over the last few months have had issues. i pay for prime for the convenience of next day delivery but more frequently the level of service has been inconsistent.,[USER] We'd like to look into the delivery delays for you. Please contact us here: [URL] ^LI,INVESTIGATION_OR_FOLLO

In [ ]:
# ============================================================
# EVIDENCE QUALITY DISTRIBUTION
# ============================================================

print("=" * 80)
print("RETRIEVED EVIDENCE QUALITY")
print("=" * 80)

display(
    evidence_df[
        "evidence_quality"
    ]
    .value_counts()
    .rename_axis("evidence_quality")
    .reset_index(name="count")
)

RETRIEVED EVIDENCE QUALITY


,evidence_quality,count
0,MEDIUM,575
1,HIGH,403
2,LOW,22


In [ ]:
# ============================================================
# TOP-1 EVIDENCE MANUAL REVIEW SET
# ============================================================

evidence_review = (
    evidence_df[
        evidence_df["ce_rank"] == 1
    ]
    [
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type"
        ]
    ]
    .copy()
)

# Add empty human-review fields
evidence_review["evidence_relevance"] = ""
evidence_review["review_notes"] = ""

print("=" * 80)
print("TOP-1 EVIDENCE REVIEW SET")
print("=" * 80)

print(f"Cases to review: {len(evidence_review)}")

display(evidence_review.head(20))

TOP-1 EVIDENCE REVIEW SET
Cases to review: 200


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type,evidence_relevance,review_notes
0,GOLD-0001,"Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!",PAYMENT_BILLING,AMZ-073280,0.615286,[user] will someone please contact mw urgently regarding an order i made?,"[USER] Oh no! Sorry to hear this! Without providing any account specific information, will you tell us what's going on? We are here to help! ^AD",INVESTIGATION_OR_FOLLOWUP,,
5,GOLD-0002,"@AmazonHelp LOL...you've missed the delivery date of every order of mine shipped via usps, for the last 5 months including todays order.",ORDER_DELIVERY,AMZ-032817,0.998420,"[user] yes, u missed the provided delivery date",[USER] Thanks for confirming! Who's the carrier listed for your order? What's the most current tracking information state? ^HC,INVESTIGATION_OR_FOLLOWUP,,
10,GOLD-0003,So bummed that my package was delayed even with Amazon Prime shipping. Now I don't have a costume for tomorrow. ☹️,ORDER_DELIVERY,AMZ-077675,0.991000,hey [user] i don’t pay for prime 2 day shipping to have my packages delayed ☹️,"[USER] Hi Madison, without going into personal account info can you tell us more about this please? ^HS",GENERAL_SUPPORT,,
15,GOLD-0004,I love @115821 for my Christmas shopping gift needs.,GENERAL_SOCIAL,AMZ-047124,0.957855,. [user] has to be the best thing ever for christmas shopping...,[USER] Can't beat skipping out on waiting in long shopping lines! We're glad you chose us this holiday season! 😊 ^KJ,GENERAL_SUPPORT,,
20,GOLD-0005,@AmazonHelp Bleg on the phone 😕 what version should it be showing?,DEVICE_TECHNICAL,AMZ-001463,0.859899,[user] will it work on my phone?,[USER] We appreciate you reaching out! Here are the current devices compatible: [URL] ^GG,GENERAL_SUPPORT,,
25,GOLD-0006,@115850 amazon stick from https://t.co/cvJQ06p7ve is authorized seller?,SELLER_AUTHENTICITY,AMZ-082165,0.770269,[user] it was sold by amazon.,[USER] Thanks for confirming! Please reach us by phone here: [URL] so we can look at available options with you. ^JM,INVESTIGATION_OR_FOLLOWUP,,
30,GOLD-0007,"@117795 Unethical delivery setup. Worst cust care. Order no 404-1061350-8629132 against canceled order at high cost, late delivery.",ORDER_DELIVERY,AMZ-006544,0.951277,[user] dishonest delivery setup. worst cust care. order no [phone_number] against canceled order at high price &amp; late delivery.,[USER] Please don't provide your order details as we consider it personal info. Our twitter page is visible to public.(2/2) ^CB,INVESTIGATION_OR_FOLLOWUP,,
35,GOLD-0008,@AmazonHelp expected delivery yesterday and it's not even there. This was a gift. So much for that.,ORDER_DELIVERY,AMZ-079961,0.999397,[user] it's not even out for delivery yet.,"[USER] Unless we've provided notification otherwise, we still expect it to arrive on time. It's not uncommon for orders to ship the same day they are due to arrive. We deliver until 8 PM. Please keep us posted on the arrival!",INVESTIGATION_OR_FOLLOWUP,,
40,GOLD-0009,@667472 @115833 My 5 year old son ordered a Nintendo switch on Alexa.,PURCHASE_CONTROL,AMZ-035797,0.450266,"dear [user] , alexa needs to enforce good manners, my kids are learning that they can get whatever they want without saying please",[USER] We are always looking for ways to improve. Please leave feedback by following these steps: [URL] ^AF,GENERAL_SUPPORT,,
45,GOLD-0010,#KaroMilkeLateDelivery is the mantra of @115850 can say it with experience. They don't delivery at commmitted time &amp; give excuses https://t.co/6I4kO0jtRL,ORDER_DELIVERY,AMZ-048759,0.363737,"[user] [user] this is d 3rd time with me the product is not delivered on time every time same excuse fed up with u guys,very2 bad experience again nd again repeatedly.. shameful u can never improve wid ur delivery service.. height of harrasment..",[USER] Apologies for the unpleasant experience. Please contact 

In [ ]:
evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("Saved:")
print("data/evaluation/top1_evidence_review.csv")

Saved:
data/evaluation/top1_evidence_review.csv


In [ ]:
# ============================================================
# TOP-1 EVIDENCE INSPECTION
# ============================================================

top1_evidence = (
    evidence_df[
        evidence_df["ce_rank"] == 1
    ]
    .copy()
)

print("=" * 80)
print("TOP-1 HISTORICAL EVIDENCE")
print("=" * 80)

display(
    top1_evidence[
        [
            "golden_id",
            "customer_query",
            "intent",
            "case_id",
            "ce_score",
            "customer_problem",
            "support_action",
            "resolution_type",
            "evidence_quality"
        ]
    ].head(20)
)

TOP-1 HISTORICAL EVIDENCE


,golden_id,customer_query,intent,case_id,ce_score,customer_problem,support_action,resolution_type,evidence_quality
0,GOLD-0001,"Contact @115821 regarding payment on an order. Contacted customer service, got outsourced to an employee who repeats herself over and over!!",PAYMENT_BILLING,AMZ-073280,0.615286,[user] will someone please contact mw urgently regarding an order i made?,"[USER] Oh no! Sorry to hear this! Without providing any account specific information, will you tell us what's going on? We are here to help! ^AD",INVESTIGATION_OR_FOLLOWUP,HIGH
5,GOLD-0002,"@AmazonHelp LOL...you've missed the delivery date of every order of mine shipped via usps, for the last 5 months including todays order.",ORDER_DELIVERY,AMZ-032817,0.998420,"[user] yes, u missed the provided delivery date",[USER] Thanks for confirming! Who's the carrier listed for your order? What's the most current tracking information state? ^HC,INVESTIGATION_OR_FOLLOWUP,HIGH
10,GOLD-0003,So bummed that my package was delayed even with Amazon Prime shipping. Now I don't have a costume for tomorrow. ☹️,ORDER_DELIVERY,AMZ-077675,0.991000,hey [user] i don’t pay for prime 2 day shipping to have my packages delayed ☹️,"[USER] Hi Madison, without going into personal account info can you tell us more about this please? ^HS",GENERAL_SUPPORT,HIGH
15,GOLD-0004,I love @115821 for my Christmas shopping gift needs.,GENERAL_SOCIAL,AMZ-047124,0.957855,. [user] has to be the best thing ever for christmas shopping...,[USER] Can't beat skipping out on waiting in long shopping lines! We're glad you chose us this holiday season! 😊 ^KJ,GENERAL_SUPPORT,HIGH
20,GOLD-0005,@AmazonHelp Bleg on the phone 😕 what version should it be showing?,DEVICE_TECHNICAL,AMZ-001463,0.859899,[user] will it work on my phone?,[USER] We appreciate you reaching out! Here are the current devices compatible: [URL] ^GG,GENERAL_SUPPORT,MEDIUM
25,GOLD-0006,@115850 amazon stick from https://t.co/cvJQ06p7ve is authorized seller?,SELLER_AUTHENTICITY,AMZ-082165,0.770269,[user] it was sold by amazon.,[USER] Thanks for confirming! Please reach us by phone here: [URL] so we can look at available options with you. ^JM,INVESTIGATION_OR_FOLLOWUP,MEDIUM
30,GOLD-0007,"@117795 Unethical delivery setup. Worst cust care. Order no 404-1061350-8629132 against canceled order at high cost, late delivery.",ORDER_DELIVERY,AMZ-006544,0.951277,[user] dishonest delivery setup. worst cust care. order no [phone_number] against canceled order at high price &amp; late delivery.,[USER] Please don't provide your order details as we consider it personal info. Our twitter page is visible to public.(2/2) ^CB,INVESTIGATION_OR_FOLLOWUP,MEDIUM
35,GOLD-0008,@AmazonHelp expected delivery yesterday and it's not even there. This was a gift. So much for that.,ORDER_DELIVERY,AMZ-079961,0.999397,[user] it's not even out for delivery yet.,"[USER] Unless we've provided notification otherwise, we still expect it to arrive on time. It's not uncommon for orders to ship the same day they are due to arrive. We deliver until 8 PM. Please keep us posted on the arrival!",INVESTIGATION_OR_FOLLOWUP,MEDIUM
40,GOLD-0009,@667472 @115833 My 5 year old son ordered a Nintendo switch on Alexa.,PURCHASE_CONTROL,AMZ-035797,0.450266,"dear [user] , alexa needs to enforce good manners, my kids are learning that they can get whatever they want without saying please",[USER] We are always looking for ways to improve. Please leave feedback by following these steps: [URL] ^AF,GENERAL_SUPPORT,MEDIUM
45,GOLD-0010,#KaroMilkeLateDelivery is the mantra of @115850 can say it with experience. They don't delivery at commmitted time &amp; give excuses https://t.co/6I4kO0jtRL,ORDER_DELIVERY,AMZ-048759,0.363737,"[user] [user] this is d 3rd time with me the product is not delivered on time every time same excuse fed up with u guys,very2 bad experience again nd again repeatedly.. shameful u can never improve wid ur delivery service.. height of harrasment..",[USER] Apologies for the unpleasant exper

In [ ]:
evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

print("Saved:")
print("data/evaluation/top1_evidence_review.csv")

Saved:
data/evaluation/top1_evidence_review.csv


In [ ]:
# ============================================================
# RETRIEVAL STAGE — FINAL CHECKPOINT
# ============================================================

import os
import numpy as np
import pandas as pd

os.makedirs("data/checkpoints", exist_ok=True)
os.makedirs("data/evaluation", exist_ok=True)
os.makedirs("data/embeddings", exist_ok=True)

# ------------------------------------------------------------
# 1. Frozen golden set
# ------------------------------------------------------------

golden_clean.to_csv(
    "data/evaluation/golden_evaluation_set_frozen.csv",
    index=False
)

# ------------------------------------------------------------
# 2. Structured RAG corpus
# ------------------------------------------------------------

rag_corpus.to_csv(
    "data/checkpoints/structured_rag_corpus.csv",
    index=False
)

# ------------------------------------------------------------
# 3. Top-5 historical evidence
# ------------------------------------------------------------

evidence_df.to_csv(
    "data/evaluation/top5_historical_evidence.csv",
    index=False
)

# ------------------------------------------------------------
# 4. Top-1 manual review sheet
# ------------------------------------------------------------

evidence_review.to_csv(
    "data/evaluation/top1_evidence_review.csv",
    index=False
)

# ------------------------------------------------------------
# 5. Cross-Encoder results
# ------------------------------------------------------------

reranked_df.to_csv(
    "data/checkpoints/cross_encoder_reranked_results.csv",
    index=False
)

# ------------------------------------------------------------
# 6. RRF candidates
# ------------------------------------------------------------

rrf_candidates_df.to_csv(
    "data/checkpoints/rrf_candidates.csv",
    index=False
)

# ------------------------------------------------------------
# 7. BGE embeddings
# ------------------------------------------------------------

if "historical_embeddings" in globals():

    np.save(
        "data/embeddings/historical_bge_large.npy",
        historical_embeddings
    )

# ------------------------------------------------------------
# 8. Retrieval benchmark
# ------------------------------------------------------------

benchmark_summary = pd.DataFrame([
    {
        "method": "TF-IDF",
        "recall_at_1": 0.160,
        "recall_at_3": 0.250,
        "recall_at_5": 0.310,
        "recall_at_10": np.nan
    },
    {
        "method": "BM25",
        "recall_at_1": 0.170,
        "recall_at_3": 0.290,
        "recall_at_5": 0.325,
        "recall_at_10": np.nan
    },
    {
        "method": "BGE-large",
        "recall_at_1": 0.215,
        "recall_at_3": 0.320,
        "recall_at_5": 0.365,
        "recall_at_10": np.nan
    },
    {
        "method": "BM25+BGE RRF",
        "recall_at_1": 0.220,
        "recall_at_3": 0.330,
        "recall_at_5": 0.380,
        "recall_at_10": 0.425
    },
    {
        "method": "RRF+CrossEncoder",
        "recall_at_1": 0.250,
        "recall_at_3": 0.350,
        "recall_at_5": 0.390,
        "recall_at_10": 0.440
    }
])

benchmark_summary.to_csv(
    "data/evaluation/retrieval_benchmark.csv",
    index=False
)

# ------------------------------------------------------------
# 9. Checkpoint manifest
# ------------------------------------------------------------

manifest = {
    "stage": "retrieval_complete",
    "historical_cases": int(len(retrieval_df)),
    "golden_cases": int(len(golden_clean)),
    "embedding_model": "BAAI/bge-large-en-v1.5",
    "embedding_dimension": 1024,
    "rrf_candidates": 50,
    "cross_encoder": "BAAI/bge-reranker-base",
    "cross_encoder_candidates": 50
}

pd.DataFrame([manifest]).to_json(
    "data/checkpoints/retrieval_checkpoint.json",
    orient="records",
    indent=2
)

print("=" * 80)
print("CHECKPOINT CREATED")
print("=" * 80)

for root, dirs, files in os.walk("data"):
    for file in files:
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{path:<65} {size_mb:>8.2f} MB")

CHECKPOINT CREATED
data/checkpoints/retrieval_checkpoint.json                            0.00 MB
data/checkpoints/cross_encoder_reranked_results.csv                   0.89 MB
data/checkpoints/rrf_candidates.csv                                   1.91 MB
data/checkpoints/structured_rag_corpus.csv                           24.78 MB
data/embeddings/historical_bge_large.npy                            325.07 MB
data/evaluation/retrieval_benchmark.csv                               0.00 MB
data/evaluation/top5_historical_evidence.csv                          0.52 MB
data/evaluation/top1_evidence_review.csv                              0.08 MB
data/evaluation/golden_evaluation_set.csv                             0.09 MB
data/evaluation/golden_evaluation_set_frozen.csv                      0.09 MB
data/evaluation/unlabeled_intent_sample_200.csv                       0.05 MB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p "/content/drive/MyDrive/hiver_support_agent_checkpoint"
!cp -r data "/content/drive/MyDrive/hiver_support_agent_checkpoint/"

In [ ]:
!find "/content/drive/MyDrive/hiver_support_agent_checkpoint/data" -type f | sort

/content/drive/MyDrive/hiver_support_agent_checkpoint/data/checkpoints/cross_encoder_reranked_results.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/checkpoints/retrieval_checkpoint.json
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/checkpoints/rrf_candidates.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/checkpoints/structured_rag_corpus.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/embeddings/historical_bge_large.npy
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/golden_evaluation_set.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/golden_evaluation_set_frozen.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/retrieval_benchmark.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/top1_evidence_review.csv
/content/drive/MyDrive/hiver_support_agent_checkpoint/data/evaluation/top5_historical_evidence.csv
/content/drive/MyDrive/hiver